
# Notebook 05 — Simulation Rules Feasibility & Selection

## Overview

> **Organized outputs (path update):** Code paths in Cells 6–15 now follow the
> dimension-based report structure. The main integrated Parquet stays in the
> report root; individual dictionaries are grouped under `data_dictionaries/`.
> Previously displayed cell outputs were retained as a record of earlier runs;
> this notebook was not re-executed on your local data when its paths were edited.
> Save the notebook with the supplied corrected filename if re-running Cell 14.


This notebook focuses on **evaluating, comparing, and selecting the structural rules and dimensions** used to design realistic multi-channel data exfiltration scenarios in MUBSIR.

Unlike V1, which relied on predefined scenario configurations, V2 follows a **data-driven approach**, where alternative rules are evaluated against users' observed behavior before final selection.

## Main Objective

Determine which scenario dimensions, values, and constraints are feasible and appropriate for the observed user population, while preserving realistic attack behavior, sufficient scenario diversity, and broad user coverage.

## Scenario Dimensions Under Evaluation

1. **Attack Duration:** Evaluate 5, 10, 20, 30, 40, and 50 eligible workdays.
2. **Off-Days / Attack Cadence:** Evaluate the number and distribution of days without injection within each attack duration.
3. **Channel Switching:** Evaluate switching levels, achievable switching counts, and the five-day exception.
4. **Multichannel Activity:** Evaluate optional same-day multichannel activity, including scenarios with zero multichannel days.

Additional structural considerations include channel combinations, primary-channel selection, and rotation patterns based on observed user activity.

## Evaluation Principles

- Use the 995 validated multi-channel candidates.
- Prioritize observed user behavior over predefined scenario patterns.
- Preserve the historical Baseline (2010) without modification.
- Restrict attack opportunities to the Future period (2011-01-01 to 2011-06-01).
- Consider only eligible workdays and naturally active channels for injection.
- Ensure zero injection on off-days.
- Allow channel rotation across different days without requiring simultaneous multichannel activity.
- Evaluate each rule based on feasibility, user coverage, and scenario diversity.
- Avoid unnecessary constraints that exclude otherwise eligible users.
- Focus exclusively on event frequency/counts, not bytes or data volume.
- Maintain a provisional target of 400 malicious users while preserving benign multi-channel users.

## Evaluation Approach

For each scenario dimension:

1. Define the candidate values or alternative rules.
2. Evaluate their feasibility using existing validated data.
3. Measure user coverage and available execution opportunities.
4. Compare alternatives and identify unnecessary restrictions.
5. Select the most appropriate rules based on numerical evidence.
6. Document the selected rules and the justification for each decision.

## Expected Outcomes

By the end of this notebook, we aim to:

- Identify feasible values and configurations for each structural scenario dimension.
- Determine the impact of different rules on user eligibility and scenario diversity.
- Establish a justified and validated set of structural simulation rules.
- Assess whether the selected rules can support the provisional target of 400 malicious users.
- Document selected rules, rejected alternatives, and remaining limitations.
- Produce a rule specification for the subsequent scenario-design stage.

## Scope & Boundaries

This notebook is dedicated to **structural rule evaluation and selection only**.

It does not perform final scenario generation, malicious-user assignment, or attack injection.

**Stealth levels, injection quantity formulas, and injection multipliers are outside the scope of this notebook.** These will be evaluated separately in Notebook 06 — Injection Frequency & Stealth Evaluation.

Existing results from Notebooks 00–03 will be reused where applicable to avoid redundant calculations.

Notebook 04 is an independent historical documentation notebook and will not be used as an input.

## Documentation

All final outputs will be consolidated under:

`reports/05_Simulation_Rules_Feasibility/`

The final `notebook_summary.md` will follow the standardized six-section structure:

1. Objective
2. Data Scope
3. Key Findings
4. Validation
5. Final Deliverables
6. Limitations & Next Step



## 5.1 Dimension 1 — Attack Duration Feasibility

### Objective
Evaluate the preliminary feasibility of six attack durations (5, 10, 20, 30, 40, and 50 eligible workdays) using validated temporal and rotation audit results from Notebook 02.

### Evaluation Criteria
- The user belongs to the 995 multi-channel candidates.
- At least one window of the required duration exists.
- At least one window supports two naturally active channels on distinct eligible workdays.
- At least one chronological channel transition is feasible.
- Same-day multichannel activity is not mandatory.
- Three-channel rotation is optional, not required.

### Evaluation Metrics
1. Number of users with available windows for each duration.
2. Number of users with preliminary rotation-feasible windows.
3. Number and proportion of windows supporting channel rotation.
4. Availability of three-channel rotation as an optional characteristic.
5. Whether each duration has sufficient preliminary user coverage for the provisional target of 400 attackers.

### Decision Principle
Evaluate all six durations based on observed user coverage and natural channel-rotation opportunities, without imposing predefined duration quotas or excluding longer durations in advance.

### Scope & Limitations
This section evaluates preliminary duration feasibility only.

It does not establish whether a complete attack schedule can satisfy the selected Off-Days, Switching Levels, or Multichannel Activity rules.

Existing Notebook 02 audit results will be reused to avoid redundant window calculations. The original User-Day data will be used in subsequent evaluations to verify actual dates, channel availability, and complete schedule feasibility.

Injection quantities and Stealth will be evaluated separately in Notebook 06.


In [1]:

# ============================================================
# 5.1 — ATTACK DURATION FEASIBILITY
# Population: 995 multi-channel candidates
# Source: Validated Notebook 02 & 03 outputs
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORTS_ROOT = PROJECT_ROOT / "reports"

DURATIONS_51 = [5, 10, 20, 30, 40, 50]
TARGET_51 = 400

CANDIDATE_FILE_51 = (
    REPORTS_ROOT / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

WINDOW_FILE_51 = (
    REPORTS_ROOT / "02_Temporal_Window_Audit"
    / "user_window_length_audit.parquet"
)

assert CANDIDATE_FILE_51.is_file()
assert WINDOW_FILE_51.is_file()

# ------------------------------------------------------------
# 2. LOAD REQUIRED DATA
# ------------------------------------------------------------

candidates_51 = pd.read_parquet(
    CANDIDATE_FILE_51,
    columns=["user"]
)

windows_51 = pd.read_parquet(
    WINDOW_FILE_51,
    columns=[
        "user",
        "window_workdays",
        "eligible_windows",
        "windows_supporting_2steps",
        "windows_3channel_3steps"
    ]
)

for df in (candidates_51, windows_51):
    df["user"] = df["user"].astype("string").str.strip()

assert candidates_51["user"].notna().all()
assert candidates_51["user"].is_unique
assert len(candidates_51) == 995

candidate_ids_51 = sorted(candidates_51["user"].tolist())

# ------------------------------------------------------------
# 3. FILTER CANDIDATES AND DURATIONS
# ------------------------------------------------------------

selected_51 = windows_51.loc[
    windows_51["user"].isin(candidate_ids_51)
    & windows_51["window_workdays"].isin(DURATIONS_51)
].copy()

assert not selected_51.duplicated(
    ["user", "window_workdays"]
).any()

# Include every candidate for every requested duration.
# Missing user-duration combinations mean zero available windows.

all_pairs_51 = pd.MultiIndex.from_product(
    [candidate_ids_51, DURATIONS_51],
    names=["user", "window_workdays"]
)

count_cols_51 = [
    "eligible_windows",
    "windows_supporting_2steps",
    "windows_3channel_3steps"
]

user_duration_51 = (
    selected_51
    .set_index(["user", "window_workdays"])[count_cols_51]
    .reindex(all_pairs_51, fill_value=0)
    .reset_index()
)

assert user_duration_51[count_cols_51].notna().all().all()

# ------------------------------------------------------------
# 4. APPLY PRELIMINARY FEASIBILITY CRITERIA
# ------------------------------------------------------------

# At least one eligible window of the required length.
user_duration_51["has_window"] = (
    user_duration_51["eligible_windows"] > 0
)

# At least one window supports an alternating sequence
# of two naturally active channel-days.
user_duration_51["rotation_feasible"] = (
    user_duration_51["windows_supporting_2steps"] > 0
)

# Optional: at least one window supports an alternating
# sequence using Email, USB, and HTTP across distinct days.
user_duration_51["three_channel_possible"] = (
    user_duration_51["windows_3channel_3steps"] > 0
)

# ------------------------------------------------------------
# 5. AGGREGATE RESULTS BY DURATION
# ------------------------------------------------------------

duration_results_51 = (
    user_duration_51
    .groupby("window_workdays", as_index=False)
    .agg(
        Users_with_windows=("has_window", "sum"),
        Feasible_users=("rotation_feasible", "sum"),
        Three_channel_users=("three_channel_possible", "sum"),
        Total_windows=("eligible_windows", "sum"),
        Rotation_windows=("windows_supporting_2steps", "sum"),
        Three_channel_windows=("windows_3channel_3steps", "sum")
    )
    .rename(columns={"window_workdays": "Duration"})
)

duration_results_51["User_coverage_pct"] = (
    duration_results_51["Feasible_users"] / 995 * 100
).round(2)

duration_results_51["Rotation_window_pct"] = (
    duration_results_51["Rotation_windows"]
    .div(
        duration_results_51["Total_windows"].replace(0, float("nan"))
    )
    .mul(100)
    .round(2)
    .fillna(0)
)

duration_results_51["Supports_400"] = (
    duration_results_51["Feasible_users"] >= TARGET_51
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert len(user_duration_51) == 995 * 6
assert len(duration_results_51) == 6

assert (
    user_duration_51["windows_supporting_2steps"]
    <= user_duration_51["eligible_windows"]
).all()

assert (
    user_duration_51["windows_3channel_3steps"]
    <= user_duration_51["windows_supporting_2steps"]
).all()

assert (
    duration_results_51["Feasible_users"]
    <= duration_results_51["Users_with_windows"]
).all()

assert (
    duration_results_51["Three_channel_users"]
    <= duration_results_51["Feasible_users"]
).all()

# ------------------------------------------------------------
# 7. OUTPUT
# ------------------------------------------------------------

print("=" * 76)
print("DIMENSION 1 — ATTACK DURATION FEASIBILITY")
print("=" * 76)

print("\nINPUT SCOPE")
print("-" * 76)
print(f"Candidate users      : {len(candidate_ids_51):,}")
print(f"Durations tested     : {DURATIONS_51}")
print(f"User-duration pairs  : {len(user_duration_51):,}")
print(f"Provisional target   : {TARGET_51}")

print("\nA. USER FEASIBILITY")
print("-" * 76)
print(
    duration_results_51[
        [
            "Duration",
            "Users_with_windows",
            "Feasible_users",
            "Three_channel_users",
            "User_coverage_pct",
            "Supports_400"
        ]
    ].to_string(index=False)
)

print("\nB. WINDOW FEASIBILITY")
print("-" * 76)
print(
    duration_results_51[
        [
            "Duration",
            "Total_windows",
            "Rotation_windows",
            "Three_channel_windows",
            "Rotation_window_pct"
        ]
    ].to_string(index=False)
)

print("\nVALIDATION")
print("-" * 76)
print("995 candidate users       : PASS")
print("Six durations evaluated   : PASS")
print("Window count consistency  : PASS")
print("Rotation subset checks    : PASS")
print("Off-days constraints      : NOT YET")
print("Switching-rate constraints: NOT YET")
print("Same-day multichannel     : NOT YET")
print("Source data modified      : NO")
print("Files written             : NONE")


DIMENSION 1 — ATTACK DURATION FEASIBILITY

INPUT SCOPE
----------------------------------------------------------------------------
Candidate users      : 995
Durations tested     : [5, 10, 20, 30, 40, 50]
User-duration pairs  : 5,970
Provisional target   : 400

A. USER FEASIBILITY
----------------------------------------------------------------------------
 Duration  Users_with_windows  Feasible_users  Three_channel_users  User_coverage_pct  Supports_400
        5                 995             995                  288             100.00          True
       10                 993             993                  291              99.80          True
       20                 992             992                  290              99.70          True
       30                 988             988                  289              99.30          True
       40                 983             983                  287              98.79          True
       50                 982           


## 5.2 Dimension 2 — Channel Switching Feasibility & Selection

### Objective
Evaluate natural channel-switching behavior and feasible rotation capacity among the 995 multi-channel candidates across the six approved attack durations.

The goal is to establish data-driven switching rules that are compatible with users' observed behavior and available channel activity.

### Data Sources
- **Notebook 01:** Natural channel transitions across eligible Future days.
- **Notebook 02:** Maximum feasible rotation steps, switching capacity, and supporting windows.
- **Notebook 03:** Validated population of 995 multi-channel candidates.

### Evaluation Criteria
- Include only the 995 validated multi-channel candidates.
- Evaluate all six attack durations: 5, 10, 20, 30, 40, and 50 eligible workdays.
- Distinguish observed natural channel transitions from feasible selected-channel rotations.
- Measure achievable switching capacity without imposing predefined channel sequences.
- Allow both two-channel and three-channel rotation.
- Do not impose predefined Off-Day counts or injection schedules.

### Evaluation Metrics
1. Natural channel-switching characteristics of the candidate population.
2. Maximum feasible switching counts by duration.
3. Distribution of switching capacity across users.
4. Number of users supporting different switching-count thresholds.
5. Number of windows supporting those thresholds.

### Decision Principle
Switching rules should reflect both natural behavioral patterns and feasible channel rotations.

Maximum switching capacity represents an upper bound, not a required attack pattern.

### Scope & Limitations
This section initially evaluates switching capacity using validated existing audit results.

Notebook 01 natural-switching summaries describe the full Future period, while Notebook 02 feasibility metrics describe user-duration windows. These measurements must not be treated as equivalent.

Final Low, Medium, and High switching-rate compatibility will be evaluated after accounting for the number of selected injection days and their actual channel sequences.

No user assignment or injection is performed.


In [2]:

# ============================================================
# 5.2 — CHANNEL SWITCHING FEASIBILITY (REVISED)
# Natural behavior vs. window rotation capacity
# Population: 995 candidates | Six approved durations
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CONFIGURATION & PATHS
# ------------------------------------------------------------

ROOT_52 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORTS_52 = ROOT_52 / "reports"
DURATIONS_52 = [5, 10, 20, 30, 40, 50]

POP_FILE_52 = (
    REPORTS_52 / "01_User_Population_Audit"
    / "user_population_profile.parquet"
)

WINDOW_FILE_52 = (
    REPORTS_52 / "02_Temporal_Window_Audit"
    / "user_window_length_audit.parquet"
)

CANDIDATE_FILE_52 = (
    REPORTS_52 / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

for path in [POP_FILE_52, WINDOW_FILE_52, CANDIDATE_FILE_52]:
    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------------------------------
# 2. LOAD VALIDATED AUDIT RESULTS
# ------------------------------------------------------------

candidates_52 = pd.read_parquet(
    CANDIDATE_FILE_52,
    columns=["user"]
)

population_52 = pd.read_parquet(
    POP_FILE_52,
    columns=[
        "user",
        "email_days",
        "usb_days",
        "http_days",
        "adjacent_direct_switches",
        "active_day_direct_switches",
        "adjacent_set_changes",
        "active_day_overlap_changes"
    ]
)

windows_52 = pd.read_parquet(
    WINDOW_FILE_52,
    columns=[
        "user",
        "window_workdays",
        "eligible_windows",
        "max_feasible_rotation_steps",
        "max_feasible_switches",
        "mean_max_steps_across_windows",
        "windows_supporting_2steps",
        "windows_supporting_3steps",
        "windows_supporting_5steps",
        "windows_supporting_10steps"
    ]
)

for df in [candidates_52, population_52, windows_52]:
    df["user"] = df["user"].astype("string").str.strip()

assert len(candidates_52) == 995
assert candidates_52["user"].is_unique
assert population_52["user"].is_unique

# Restrict to validated candidate users.
population_52 = population_52.merge(
    candidates_52,
    on="user",
    how="inner",
    validate="one_to_one"
)

assert len(population_52) == 995

# Actual number of naturally active channels per user.
population_52["observed_channels"] = (
    population_52[
        ["email_days", "usb_days", "http_days"]
    ].gt(0).sum(axis=1)
)

assert population_52["observed_channels"].isin([2, 3]).all()

# ------------------------------------------------------------
# 3. NATURAL SWITCHING — FULL FUTURE CONTEXT
# ------------------------------------------------------------

natural_columns_52 = {
    "Adjacent direct switches": "adjacent_direct_switches",
    "Active-day direct switches": "active_day_direct_switches",
    "Adjacent channel-set changes": "adjacent_set_changes",
    "Active-day overlap changes": "active_day_overlap_changes"
}

natural_summary_52 = pd.DataFrame([
    {
        "Metric": label,
        "Users_with_changes": int(population_52[col].gt(0).sum()),
        "Median": float(population_52[col].median()),
        "P75": float(population_52[col].quantile(0.75))
    }
    for label, col in natural_columns_52.items()
])

# Important:
# These counts cover the FULL Future period.
# They are not per-window attack-switch counts.

# ------------------------------------------------------------
# 4. BUILD USER-DURATION CAPACITY TABLE
# ------------------------------------------------------------

candidate_ids_52 = sorted(population_52["user"].tolist())

selected_52 = windows_52.loc[
    windows_52["user"].isin(candidate_ids_52)
    & windows_52["window_workdays"].isin(DURATIONS_52)
].copy()

assert not selected_52.duplicated(
    ["user", "window_workdays"]
).any()

metrics_52 = [
    "eligible_windows",
    "max_feasible_rotation_steps",
    "max_feasible_switches",
    "mean_max_steps_across_windows",
    "windows_supporting_2steps",
    "windows_supporting_3steps",
    "windows_supporting_5steps",
    "windows_supporting_10steps"
]

assert selected_52[metrics_52].notna().all().all()

all_pairs_52 = pd.MultiIndex.from_product(
    [candidate_ids_52, DURATIONS_52],
    names=["user", "window_workdays"]
)

capacity_52 = (
    selected_52
    .set_index(["user", "window_workdays"])[metrics_52]
    .reindex(all_pairs_52, fill_value=0)
    .reset_index()
)

capacity_52 = capacity_52.merge(
    population_52[["user", "observed_channels"]],
    on="user",
    validate="many_to_one"
)

capacity_52["has_window"] = (
    capacity_52["eligible_windows"].gt(0)
)

# Thresholds are structural possibilities, NOT approved rules.
for steps in [2, 3, 5, 10]:
    switches = steps - 1

    capacity_52[f"supports_{switches}switches"] = (
        capacity_52["max_feasible_switches"].ge(switches)
    )

    capacity_52[f"window_pct_{switches}switches"] = np.where(
        capacity_52["eligible_windows"].gt(0),
        100 * capacity_52[f"windows_supporting_{steps}steps"]
        / capacity_52["eligible_windows"].replace(0, np.nan),
        0.0
    )

# ------------------------------------------------------------
# 5. CAPACITY BY DURATION
# ------------------------------------------------------------

valid_52 = capacity_52.loc[
    capacity_52["has_window"]
].copy()

capacity_stats_52 = (
    valid_52.groupby("window_workdays")
    .agg(
        Median_best_switches=("max_feasible_switches", "median"),
        P25_best_switches=(
            "max_feasible_switches",
            lambda x: x.quantile(0.25)
        ),
        P75_best_switches=(
            "max_feasible_switches",
            lambda x: x.quantile(0.75)
        ),
        Median_mean_window_steps=(
            "mean_max_steps_across_windows", "median"
        ),
        Median_user_4switch_window_pct=(
            "window_pct_4switches", "median"
        )
    )
    .reset_index()
)

user_counts_52 = (
    capacity_52.groupby("window_workdays")
    .agg(
        Users_with_windows=("has_window", "sum"),
        Users_1plus=("supports_1switches", "sum"),
        Users_2plus=("supports_2switches", "sum"),
        Users_4plus=("supports_4switches", "sum"),
        Users_9plus=("supports_9switches", "sum")
    )
    .reset_index()
)

switch_summary_52 = user_counts_52.merge(
    capacity_stats_52,
    on="window_workdays",
    validate="one_to_one"
).rename(columns={"window_workdays": "Duration"})

# ------------------------------------------------------------
# 6. TWO-CHANNEL VS THREE-CHANNEL USERS
# ------------------------------------------------------------

channel_group_52 = (
    capacity_52
    .groupby(["window_workdays", "observed_channels"])
    .agg(
        Users_with_windows=("has_window", "sum"),
        Users_1plus=("supports_1switches", "sum"),
        Users_4plus=("supports_4switches", "sum"),
        Users_9plus=("supports_9switches", "sum")
    )
    .reset_index()
    .rename(columns={
        "window_workdays": "Duration",
        "observed_channels": "Natural_channels"
    })
)

# ------------------------------------------------------------
# 7. WINDOW-LEVEL SUPPORT
# ------------------------------------------------------------

window_summary_52 = (
    capacity_52
    .groupby("window_workdays")
    .agg(
        Total_windows=("eligible_windows", "sum"),
        Windows_1plus=("windows_supporting_2steps", "sum"),
        Windows_2plus=("windows_supporting_3steps", "sum"),
        Windows_4plus=("windows_supporting_5steps", "sum"),
        Windows_9plus=("windows_supporting_10steps", "sum")
    )
    .reset_index()
    .rename(columns={"window_workdays": "Duration"})
)

for threshold in [1, 2, 4, 9]:
    window_summary_52[f"Pct_{threshold}plus"] = (
        100
        * window_summary_52[f"Windows_{threshold}plus"]
        / window_summary_52["Total_windows"]
    ).round(2)

# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

assert len(capacity_52) == 995 * 6
assert len(switch_summary_52) == 6
assert len(channel_group_52) == 12
assert len(window_summary_52) == 6

assert (
    capacity_52["max_feasible_switches"]
    == (capacity_52["max_feasible_rotation_steps"] - 1)
       .clip(lower=0)
).all()

assert (
    capacity_52["max_feasible_switches"]
    <= capacity_52["window_workdays"] - 1
).all()

assert (
    capacity_52["mean_max_steps_across_windows"]
    <= capacity_52["window_workdays"]
).all()

for smaller, larger in [(10, 5), (5, 3), (3, 2)]:
    assert (
        capacity_52[f"windows_supporting_{smaller}steps"]
        <= capacity_52[f"windows_supporting_{larger}steps"]
    ).all()

assert (
    capacity_52["windows_supporting_2steps"]
    <= capacity_52["eligible_windows"]
).all()

assert (
    population_52["observed_channels"].value_counts().to_dict()
    == {2: 704, 3: 291}
)

# ------------------------------------------------------------
# 9. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 82)
print("DIMENSION 2 — CHANNEL SWITCHING FEASIBILITY (REVISED)")
print("=" * 82)

print("\nINPUT SCOPE")
print("-" * 82)
print("Candidates          :", len(population_52))
print("Two-channel users   :", int(population_52["observed_channels"].eq(2).sum()))
print("Three-channel users :", int(population_52["observed_channels"].eq(3).sum()))
print("User-duration pairs :", len(capacity_52))
print("Durations           :", DURATIONS_52)

print("\nA. NATURAL SWITCHING — FULL FUTURE (CONTEXT ONLY)")
print("-" * 82)
print(natural_summary_52.to_string(index=False))

print("\nB. SWITCHING CAPACITY BY DURATION")
print("-" * 82)
print(switch_summary_52.round(2).to_string(index=False))

print("\nC. CAPACITY BY NATURAL CHANNEL COUNT")
print("-" * 82)
print(channel_group_52.to_string(index=False))

print("\nD. WINDOW SUPPORT PERCENTAGES")
print("-" * 82)
print(
    window_summary_52[
        [
            "Duration", "Total_windows",
            "Pct_1plus", "Pct_2plus",
            "Pct_4plus", "Pct_9plus"
        ]
    ].to_string(index=False)
)

print("\nVALIDATION")
print("-" * 82)
print("Candidate population       : PASS")
print("2-channel / 3-channel split: PASS")
print("Six durations evaluated    : PASS")
print("Rotation capacity checks   : PASS")
print("Nested window thresholds   : PASS")
print("Natural vs capacity        : SEPARATE")
print("Exact per-window natural switching : NOT YET MEASURED")
print("Low / Medium / High rules  : NOT YET SELECTED")
print("Files written              : NONE")
print("Source data modified       : NO")


DIMENSION 2 — CHANNEL SWITCHING FEASIBILITY (REVISED)

INPUT SCOPE
----------------------------------------------------------------------------------
Candidates          : 995
Two-channel users   : 704
Three-channel users : 291
User-duration pairs : 5970
Durations           : [5, 10, 20, 30, 40, 50]

A. NATURAL SWITCHING — FULL FUTURE (CONTEXT ONLY)
----------------------------------------------------------------------------------
                      Metric  Users_with_changes  Median  P75
    Adjacent direct switches                 919     7.0 10.0
  Active-day direct switches                 928    10.0 14.0
Adjacent channel-set changes                 994    31.0 44.0
  Active-day overlap changes                 985    29.0 41.0

B. SWITCHING CAPACITY BY DURATION
----------------------------------------------------------------------------------
 Duration  Users_with_windows  Users_1plus  Users_2plus  Users_4plus  Users_9plus  Median_best_switches  P25_best_switches  P75_best_swit


### 5.2.1 Natural Switching Within Actual Windows

#### Objective
Measure observed natural channel transitions within each actual Future window for the 995 multi-channel candidates across all six approved durations.

The purpose is to understand users' natural switching behavior at the same temporal scale as the proposed attacks, rather than relying on whole-Future summaries or maximum rotation capacity alone.

#### Data Sources
- Notebook 01: Validated natural-transition definitions and user-level reference results.
- Notebook 02: Validated window availability and minimum rotation feasibility.
- Notebook 03: The 995 validated multi-channel candidates.
- Data Engineering: Eligible Future daily activity from `behavior_features_r62_v1.parquet`.

#### Evaluation
For each actual user-window:
1. Identify its start and end dates.
2. Measure naturally active Email, USB, and HTTP days.
3. Count direct channel switches and channel-set changes within the window.
4. Include transitions involving overlapping channel activity.
5. Measure natural switching frequency relative to active-day transitions.
6. Verify window counts and minimum rotation feasibility against Notebook 02.

#### Important Distinction
Natural transitions are measured from observed channel activity, not from an injected or artificially selected channel sequence.

Channel-set changes are not equivalent to changes in the ML model's normalized dominant-channel feature.

#### Expected Outcome
Produce a window-level natural-switching audit to support the subsequent evaluation of Low, Medium, and High switching rules.

No switching level, malicious user, or injection schedule is selected in this step.


In [3]:

# ============================================================
# CELL 6 — NATURAL SWITCHING WITHIN ACTUAL WINDOWS
# 995 candidates | 6 durations | Read-only source data
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS & CONFIGURATION
# ------------------------------------------------------------

ROOT_53 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORTS_53 = ROOT_53 / "reports"
DURATIONS_53 = [5, 10, 20, 30, 40, 50]

DAILY_PATH_53 = (
    ROOT_53.parent / "Data_Engineering"
    / "Data_engineering" / "data" / "features"
    / "behavior_features_r62_v1.parquet"
)

CANDIDATE_PATH_53 = (
    REPORTS_53 / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

PROFILE_PATH_53 = (
    REPORTS_53 / "01_User_Population_Audit"
    / "user_population_profile.parquet"
)

WINDOW_PATH_53 = (
    REPORTS_53 / "02_Temporal_Window_Audit"
    / "user_window_length_audit.parquet"
)

OUTPUT_DIR_53 = (
    REPORTS_53 / "05_Simulation_Rules_Feasibility"
    / "02_Switching"
)

for path in [
    DAILY_PATH_53, CANDIDATE_PATH_53,
    PROFILE_PATH_53, WINDOW_PATH_53
]:
    assert path.is_file(), f"Missing: {path}"

# ------------------------------------------------------------
# 2. LOAD EXISTING VALIDATED DATA
# ------------------------------------------------------------

candidates_53 = pd.read_parquet(
    CANDIDATE_PATH_53, columns=["user"]
)

switch_cols_53 = [
    "adjacent_direct_switches",
    "adjacent_set_changes",
    "active_day_direct_switches",
    "active_day_overlap_changes"
]

profile_53 = pd.read_parquet(
    PROFILE_PATH_53, columns=["user"] + switch_cols_53
)

window_ref_53 = pd.read_parquet(
    WINDOW_PATH_53,
    columns=[
        "user", "window_workdays",
        "eligible_windows",
        "windows_supporting_2steps"
    ]
)

daily_53 = pd.read_parquet(
    DAILY_PATH_53,
    columns=[
        "user", "day",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

for df in [
    candidates_53, profile_53, window_ref_53, daily_53
]:
    df["user"] = df["user"].astype("string").str.strip()

assert len(candidates_53) == 995
assert candidates_53["user"].is_unique

candidate_ids_53 = set(candidates_53["user"])

daily_53 = daily_53.loc[
    daily_53["user"].isin(candidate_ids_53)
].copy()

daily_53["day"] = pd.to_datetime(daily_53["day"])

assert not daily_53.duplicated(["user", "day"]).any()
assert daily_53["day"].between(
    "2011-01-01", "2011-06-01"
).all()

daily_53 = daily_53.sort_values(["user", "day"])

# ------------------------------------------------------------
# 3. HELPER: PREFIX COUNTS
# ------------------------------------------------------------

def prefix_53(values):
    return np.r_[0, np.cumsum(
        np.asarray(values, dtype=np.int64)
    )]

def window_count_53(prefix, starts, ends):
    return prefix[ends] - prefix[starts]

# ------------------------------------------------------------
# 4. ANALYZE EACH USER'S ACTUAL WINDOWS
# ------------------------------------------------------------

window_parts_53 = []
qa_rows_53 = []

for user, g in daily_53.groupby("user", sort=False):

    dates = g["day"].to_numpy()
    n = len(g)

    E = g["external_email_count"].to_numpy() > 0
    U = g["usb_copy_count"].to_numpy() > 0
    H = g["www_upload_count"].to_numpy() > 0

    # Channel-state bitmask:
    # 0=NONE, 1=E, 2=U, 4=H,
    # 3=EU, 5=EH, 6=UH, 7=EUH

    state = (
        E.astype(np.int8)
        + 2 * U.astype(np.int8)
        + 4 * H.astype(np.int8)
    )

    # Adjacent eligible-day transitions.
    previous = state[:-1]
    current = state[1:]

    single_previous = np.isin(previous, [1, 2, 4])
    single_current = np.isin(current, [1, 2, 4])

    adjacent_direct = np.r_[
        False,
        single_previous & single_current
        & (previous != current)
    ]

    adjacent_set = np.r_[
        False,
        (previous > 0) & (current > 0)
        & (previous != current)
    ]

    # Successive active-day transitions.
    # Inactive days are skipped, but only within
    # the boundaries of each evaluated window.

    active_positions = np.flatnonzero(state > 0)
    active_states = state[active_positions]

    ap = active_states[:-1]
    ac = active_states[1:]

    active_direct = np.r_[
        False,
        np.isin(ap, [1, 2, 4])
        & np.isin(ac, [1, 2, 4])
        & (ap != ac)
    ]

    active_set = np.r_[False, ap != ac]

    # Reconcile whole-Future transitions with NB01.
    qa_rows_53.append({
        "user": user,
        "adjacent_direct_switches": int(adjacent_direct.sum()),
        "adjacent_set_changes": int(adjacent_set.sum()),
        "active_day_direct_switches": int(active_direct.sum()),
        "active_day_overlap_changes": int(
            active_set.sum() - active_direct.sum()
        )
    })

    # Prefix arrays for all window lengths.
    pE = prefix_53(E)
    pU = prefix_53(U)
    pH = prefix_53(H)

    pActive = prefix_53(state > 0)
    pMulti = prefix_53(np.isin(state, [3, 5, 6, 7]))

    pAdjDirect = prefix_53(adjacent_direct)
    pAdjSet = prefix_53(adjacent_set)

    pActDirect = prefix_53(active_direct)
    pActSet = prefix_53(active_set)

    for L in DURATIONS_53:

        if n < L:
            continue

        starts = np.arange(n - L + 1)
        ends = starts + L

        email_days = window_count_53(pE, starts, ends)
        usb_days = window_count_53(pU, starts, ends)
        http_days = window_count_53(pH, starts, ends)

        active_days = window_count_53(
            pActive, starts, ends
        )

        multi_days = window_count_53(
            pMulti, starts, ends
        )

        observed_channels = (
            (email_days > 0).astype(int)
            + (usb_days > 0).astype(int)
            + (http_days > 0).astype(int)
        )

        # Adjacent pairs inside the window only.
        adj_direct = (
            pAdjDirect[ends] - pAdjDirect[starts + 1]
        )
        adj_set = (
            pAdjSet[ends] - pAdjSet[starts + 1]
        )

        # Locate naturally active days inside
        # each window; exclude outside transitions.
        lo = np.searchsorted(active_positions, starts)
        hi = np.searchsorted(active_positions, ends)

        first_pair = np.minimum(lo + 1, hi)

        act_direct = (
            pActDirect[hi] - pActDirect[first_pair]
        )
        act_set = (
            pActSet[hi] - pActSet[first_pair]
        )

        act_overlap = act_set - act_direct

        active_pairs = np.maximum(active_days - 1, 0)

        natural_rate = (
            100 * act_set / np.maximum(active_pairs, 1)
        )

        # Minimum alternating two-step feasibility.
        supports_2step = (
            (active_days >= 2)
            & (observed_channels >= 2)
        )

        window_parts_53.append(pd.DataFrame({
            "user": user,
            "window_workdays": L,
            "window_start": dates[starts],
            "window_end": dates[ends - 1],
            "start_eligible_position": starts,
            "end_eligible_position": ends - 1,
            "natural_active_days": active_days,
            "email_active_days": email_days,
            "usb_active_days": usb_days,
            "http_active_days": http_days,
            "observed_channels": observed_channels,
            "natural_multichannel_days": multi_days,
            "natural_adjacent_direct_switches": adj_direct,
            "natural_adjacent_set_changes": adj_set,
            "natural_active_direct_switches": act_direct,
            "natural_active_overlap_changes": act_overlap,
            "natural_active_set_changes": act_set,
            "natural_active_transition_pairs": active_pairs,
            "natural_set_change_rate_pct": natural_rate.round(2),
            "supports_2step": supports_2step
        }))

natural_windows_53 = pd.concat(
    window_parts_53, ignore_index=True
)

# ------------------------------------------------------------
# 5. VALIDATE AGAINST NOTEBOOK 01
# ------------------------------------------------------------

qa_actual_53 = (
    pd.DataFrame(qa_rows_53)
    .set_index("user")
    .sort_index()
)

qa_expected_53 = (
    profile_53.loc[
        profile_53["user"].isin(candidate_ids_53)
    ]
    .set_index("user")
    .sort_index()
)

assert qa_actual_53.index.equals(qa_expected_53.index)

for col in switch_cols_53:
    assert (
        qa_actual_53[col].to_numpy()
        == qa_expected_53[col].to_numpy()
    ).all(), f"Notebook 01 mismatch: {col}"

# ------------------------------------------------------------
# 6. VALIDATE AGAINST NOTEBOOK 02
# ------------------------------------------------------------

expected_53 = (
    window_ref_53.loc[
        window_ref_53["user"].isin(candidate_ids_53)
        & window_ref_53["window_workdays"].isin(DURATIONS_53)
    ]
    .set_index(["user", "window_workdays"])
    .sort_index()
)

actual_53 = (
    natural_windows_53
    .groupby(["user", "window_workdays"])
    .agg(
        eligible_windows=("user", "size"),
        windows_supporting_2steps=("supports_2step", "sum")
    )
    .sort_index()
)

assert actual_53.index.equals(expected_53.index)

for col in [
    "eligible_windows", "windows_supporting_2steps"
]:
    assert np.array_equal(
        actual_53[col].to_numpy(),
        expected_53[col].to_numpy()
    ), f"Notebook 02 mismatch: {col}"

assert len(natural_windows_53) == 472667

# ------------------------------------------------------------
# 7. SUMMARIZE NATURAL SWITCHING BY DURATION
# ------------------------------------------------------------

natural_duration_53 = (
    natural_windows_53
    .groupby("window_workdays")
    .agg(
        Windows=("user", "size"),
        Users=("user", "nunique"),
        Median_direct=(
            "natural_active_direct_switches", "median"
        ),
        Median_set_changes=(
            "natural_active_set_changes", "median"
        ),
        P75_set_changes=(
            "natural_active_set_changes",
            lambda x: x.quantile(0.75)
        ),
        Median_overlap_changes=(
            "natural_active_overlap_changes", "median"
        ),
        Median_change_rate_pct=(
            "natural_set_change_rate_pct", "median"
        )
    )
    .reset_index()
    .rename(columns={"window_workdays": "Duration"})
)

# Give each user equal weight in the second summary.
user_medians_53 = (
    natural_windows_53
    .groupby(["user", "window_workdays"])
    ["natural_active_set_changes"]
    .median()
    .reset_index(name="User_median_set_changes")
)

natural_user_summary_53 = (
    user_medians_53
    .groupby("window_workdays")
    .agg(
        Users=("user", "nunique"),
        Median_user_median=(
            "User_median_set_changes", "median"
        ),
        P75_user_median=(
            "User_median_set_changes",
            lambda x: x.quantile(0.75)
        )
    )
    .reset_index()
    .rename(columns={"window_workdays": "Duration"})
)

# ------------------------------------------------------------
# 8. SAVE REUSABLE WINDOW-LEVEL RESULT
# ------------------------------------------------------------

OUTPUT_DIR_53.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH_53 = (
    OUTPUT_DIR_53
    / "natural_window_switching_audit.parquet"
)

natural_windows_53.to_parquet(
    OUTPUT_PATH_53, index=False
)

# ------------------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------------------

print("=" * 78)
print("CELL 6 — NATURAL SWITCHING WITHIN ACTUAL WINDOWS")
print("=" * 78)

print("\nINPUT SCOPE")
print("-" * 78)
print("Candidate users   :", len(candidate_ids_53))
print("Future user-days  :", f"{len(daily_53):,}")
print("Durations         :", DURATIONS_53)
print("Actual windows    :", f"{len(natural_windows_53):,}")

print("\nA. WINDOW-LEVEL NATURAL SWITCHING")
print("-" * 78)
print(natural_duration_53.round(2).to_string(index=False))

print("\nB. EQUAL-WEIGHT USER SUMMARY")
print("-" * 78)
print(natural_user_summary_53.round(2).to_string(index=False))

print("\nVALIDATION")
print("-" * 78)
print("Notebook 01 natural transitions : PASS")
print("Notebook 02 window counts       : PASS")
print("Notebook 02 two-step support    : PASS")
print("Inside-window boundaries        : PASS")
print("Six durations                   : PASS")
print("Low / Medium / High assigned    : NO")
print("Injection performed             : NO")
print("Original data modified          : NO")

print("\nSAVED")
print("File:", OUTPUT_PATH_53)
print("Shape:", natural_windows_53.shape)
print("Key: user + window_workdays + window_start")


CELL 6 — NATURAL SWITCHING WITHIN ACTUAL WINDOWS

INPUT SCOPE
------------------------------------------------------------------------------
Candidate users   : 995
Future user-days  : 103,395
Durations         : [5, 10, 20, 30, 40, 50]
Actual windows    : 472,667

A. WINDOW-LEVEL NATURAL SWITCHING
------------------------------------------------------------------------------
 Duration  Windows  Users  Median_direct  Median_set_changes  P75_set_changes  Median_overlap_changes  Median_change_rate_pct
        5    99415    995            0.0                 1.0              2.0                     1.0                   50.00
       10    94444    993            0.0                 3.0              5.0                     2.0                   55.56
       20    84518    992            2.0                 7.0             10.0                     5.0                   53.85
       30    74602    988            2.0                11.0             15.0                     8.0                


### 5.2.2 Natural Dominant-Channel Switching

#### Objective
Identify the naturally dominant channel on each eligible Future day and measure observed dominant-channel switches within every actual window of the 995 multi-channel candidates.

#### Methodology
1. Normalize daily Email, USB, and HTTP event counts using their respective historical P95 values.
2. Identify the uniquely dominant positive channel for each day.
3. Exclude inactive days and tied dominant-channel observations from the switching sequence.
4. Calculate dominant-channel switches within each actual 5, 10, 20, 30, 40, and 50-workday window.
5. Preserve the existing window-level results from Cell 6 and append the new dominant-channel measurements.
6. Validate results against the available channel-rotation capacity.

#### Expected Output
A window-level dataset containing:
- User and actual window dates.
- Naturally active channels and their activity days.
- Natural channel-set changes from Cell 6.
- Dominant-channel observations for Email, USB, and HTTP.
- Natural dominant-channel switching count and rate.

#### Purpose
Provide a consistent behavioral reference for evaluating Low, Medium, and High switching levels before attack scenario assignment.

#### Scope
Only natural activity is analyzed. No attackers are selected and no events are injected.


In [4]:

# ============================================================
# CELL 7 — NATURAL DOMINANT-CHANNEL SWITCHING
# 995 candidates | 472,667 actual windows | Counts only
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT_54 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORTS_54 = ROOT_54 / "reports"
DURATIONS_54 = [5, 10, 20, 30, 40, 50]

DAILY_PATH_54 = (
    ROOT_54.parent / "Data_Engineering"
    / "Data_engineering" / "data" / "features"
    / "behavior_features_r62_v1.parquet"
)

CANDIDATE_PATH_54 = (
    REPORTS_54 / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

WINDOW_PATH_54 = (
    REPORTS_54 / "05_Simulation_Rules_Feasibility"
    / "02_Switching"
    / "natural_window_switching_audit.parquet"
)

CAPACITY_PATH_54 = (
    REPORTS_54 / "02_Temporal_Window_Audit"
    / "user_window_length_audit.parquet"
)

OUTPUT_PATH_54 = (
    REPORTS_54 / "05_Simulation_Rules_Feasibility"
    / "02_Switching"
    / "natural_window_switching_enriched.parquet"
)

for path in [
    DAILY_PATH_54, CANDIDATE_PATH_54,
    WINDOW_PATH_54, CAPACITY_PATH_54
]:
    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------------------------------
# 2. LOAD VALIDATED SOURCES
# ------------------------------------------------------------

candidates_54 = pd.read_parquet(
    CANDIDATE_PATH_54,
    columns=["user"]
)

daily_54 = pd.read_parquet(
    DAILY_PATH_54,
    columns=[
        "user", "day",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count",
        "email_count_p95",
        "usb_copy_p95",
        "http_upload_p95"
    ]
)

windows_54 = pd.read_parquet(WINDOW_PATH_54)

capacity_ref_54 = pd.read_parquet(
    CAPACITY_PATH_54,
    columns=[
        "user",
        "window_workdays",
        "max_feasible_switches"
    ]
)

for frame in [
    candidates_54, daily_54,
    windows_54, capacity_ref_54
]:
    frame["user"] = (
        frame["user"].astype("string").str.strip()
    )

assert len(candidates_54) == 995
assert candidates_54["user"].is_unique

candidate_ids_54 = set(candidates_54["user"])

daily_54 = daily_54.loc[
    daily_54["user"].isin(candidate_ids_54)
].copy()

daily_54["day"] = pd.to_datetime(
    daily_54["day"]
).dt.normalize()

for col in ["window_start", "window_end"]:
    windows_54[col] = pd.to_datetime(
        windows_54[col]
    ).dt.normalize()

KEYS_54 = ["user", "window_workdays", "window_start"]

assert len(daily_54) == 103395
assert len(windows_54) == 472667
assert not daily_54.duplicated(["user", "day"]).any()
assert not windows_54.duplicated(KEYS_54).any()
assert set(daily_54["user"]) == candidate_ids_54

daily_54 = daily_54.sort_values(["user", "day"])

# ------------------------------------------------------------
# 3. DOMINANT-CHANNEL LOGIC
# ------------------------------------------------------------

CHANNEL_NAMES_54 = ["Email", "USB", "HTTP"]

COUNT_COLS_54 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

P95_COLS_54 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

def prefix_54(values):
    return np.r_[
        0,
        np.cumsum(np.asarray(values, dtype=np.int64))
    ]

def count_inside_54(prefix, starts, ends):
    return prefix[ends] - prefix[starts]

def dominant_switch_counts_54(labels, starts, ends):
    """
    Count changes between consecutive valid dominant labels
    within each window. Invalid/tied days are skipped.
    """
    valid_positions = np.flatnonzero(labels >= 0)
    valid_labels = labels[valid_positions]

    changes = np.r_[
        0,
        (valid_labels[1:] != valid_labels[:-1]).astype(int)
    ]

    cumulative = prefix_54(changes)

    left = np.searchsorted(valid_positions, starts)
    right = np.searchsorted(valid_positions, ends)

    first_transition = np.minimum(left + 1, right)

    switches = (
        cumulative[right]
        - cumulative[first_transition]
    )

    valid_days = right - left

    return switches, valid_days

# Validate boundary handling with a known sequence.
# E, E, U, tied, E -> two dominant switches.

test_labels_54 = np.array([0, 0, 1, -1, 0])

test_result_54, _ = dominant_switch_counts_54(
    test_labels_54,
    np.array([0, 1]),
    np.array([5, 3])
)

assert test_result_54.tolist() == [2, 1]

# ------------------------------------------------------------
# 4. MEASURE DOMINANCE FOR EACH USER AND WINDOW
# ------------------------------------------------------------

parts_54 = []

daily_diagnostics_54 = {
    "unique_dominant_days": 0,
    "tied_dominant_days": 0,
    "no_positive_activity_days": 0
}

for user, group in daily_54.groupby("user", sort=False):

    counts = group[COUNT_COLS_54].to_numpy(dtype=float)
    p95 = group[P95_COLS_54].to_numpy(dtype=float)

    dates = group["day"].to_numpy()
    n = len(group)

    assert np.isfinite(counts).all()
    assert (counts >= 0).all()

    valid_p95 = np.isfinite(p95) & (p95 > 0)

    # No invented historical P95 values.
    missing_active_baseline = (
        (counts > 0) & ~valid_p95
    )

    assert not missing_active_baseline.any(), (
        f"{user}: Positive channel activity without "
        "a valid historical P95. Review baseline coverage."
    )

    normalized = np.full(
        counts.shape, -np.inf, dtype=float
    )

    np.divide(
        counts,
        p95,
        out=normalized,
        where=valid_p95
    )

    # Unique positive maximum identifies dominance.
    maximum = normalized.max(axis=1)
    positive = maximum > 0

    tied_at_max = np.isclose(
        normalized,
        maximum[:, None],
        rtol=1e-9,
        atol=1e-12
    )

    winning_count = tied_at_max.sum(axis=1)

    unique = positive & (winning_count == 1)
    tied = positive & (winning_count > 1)
    no_activity = ~positive

    # -1 = no unique positive dominant channel
    #  0 = Email, 1 = USB, 2 = HTTP

    dominant = np.full(n, -1, dtype=np.int8)

    dominant[unique] = np.argmax(
        normalized, axis=1
    )[unique]

    daily_diagnostics_54["unique_dominant_days"] += int(
        unique.sum()
    )
    daily_diagnostics_54["tied_dominant_days"] += int(
        tied.sum()
    )
    daily_diagnostics_54["no_positive_activity_days"] += int(
        no_activity.sum()
    )

    # Prefix counts: days dominated by each channel.
    p_email = prefix_54(dominant == 0)
    p_usb = prefix_54(dominant == 1)
    p_http = prefix_54(dominant == 2)
    p_tied = prefix_54(tied)
    p_inactive = prefix_54(no_activity)

    for duration in DURATIONS_54:

        if n < duration:
            continue

        starts = np.arange(n - duration + 1)
        ends = starts + duration

        switches, valid_days = (
            dominant_switch_counts_54(
                dominant, starts, ends
            )
        )

        email_days = count_inside_54(
            p_email, starts, ends
        )
        usb_days = count_inside_54(
            p_usb, starts, ends
        )
        http_days = count_inside_54(
            p_http, starts, ends
        )

        tied_days = count_inside_54(
            p_tied, starts, ends
        )
        inactive_days = count_inside_54(
            p_inactive, starts, ends
        )

        dominant_channels = (
            (email_days > 0).astype(int)
            + (usb_days > 0).astype(int)
            + (http_days > 0).astype(int)
        )

        transition_pairs = np.maximum(valid_days - 1, 0)

        rate = np.full(
            len(starts), np.nan, dtype=float
        )

        np.divide(
            100 * switches,
            transition_pairs,
            out=rate,
            where=transition_pairs > 0
        )

        parts_54.append(pd.DataFrame({
            "user": user,
            "window_workdays": duration,
            "window_start": dates[starts],
            "window_end": dates[ends - 1],
            "dominant_email_days": email_days,
            "dominant_usb_days": usb_days,
            "dominant_http_days": http_days,
            "dominant_valid_days": valid_days,
            "dominant_tied_days": tied_days,
            "dominant_no_activity_days": inactive_days,
            "dominant_channels_observed": dominant_channels,
            "natural_dominant_switches": switches,
            "dominant_transition_pairs": transition_pairs,
            "natural_dominant_switch_rate_pct": np.round(
                rate, 2
            )
        }))

dominant_windows_54 = pd.concat(
    parts_54, ignore_index=True
)

# ------------------------------------------------------------
# 5. MERGE WITH CELL 6 — SAME ACTUAL WINDOWS
# ------------------------------------------------------------

assert len(dominant_windows_54) == 472667
assert not dominant_windows_54.duplicated(KEYS_54).any()

enriched_54 = windows_54.merge(
    dominant_windows_54,
    on=KEYS_54,
    how="inner",
    validate="one_to_one",
    suffixes=("", "_calculated")
)

assert len(enriched_54) == len(windows_54)

assert (
    enriched_54["window_end"]
    == enriched_54["window_end_calculated"]
).all()

enriched_54 = enriched_54.drop(
    columns=["window_end_calculated"]
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert (
    enriched_54["dominant_valid_days"]
    + enriched_54["dominant_tied_days"]
    + enriched_54["dominant_no_activity_days"]
    == enriched_54["window_workdays"]
).all()

assert (
    enriched_54["dominant_valid_days"]
    <= enriched_54["natural_active_days"]
).all()

assert (
    enriched_54["dominant_channels_observed"]
    <= enriched_54["observed_channels"]
).all()

assert (
    enriched_54["natural_dominant_switches"]
    <= enriched_54["dominant_transition_pairs"]
).all()

assert (
    enriched_54["natural_dominant_switches"]
    <= enriched_54["window_workdays"] - 1
).all()

# Check against Notebook 02's maximum feasible switches.
capacity_check_54 = capacity_ref_54.loc[
    capacity_ref_54["user"].isin(candidate_ids_54)
    & capacity_ref_54["window_workdays"].isin(DURATIONS_54)
]

assert not capacity_check_54.duplicated(
    ["user", "window_workdays"]
).any()

check_54 = enriched_54.merge(
    capacity_check_54,
    on=["user", "window_workdays"],
    how="left",
    validate="many_to_one"
)

assert check_54["max_feasible_switches"].notna().all()

assert (
    check_54["natural_dominant_switches"]
    <= check_54["max_feasible_switches"]
).all(), (
    "Dominant switching exceeds Notebook 02 "
    "rotation capacity."
)

assert len(enriched_54) == 472667
assert enriched_54["user"].nunique() == 995
assert enriched_54["window_workdays"].nunique() == 6

# ------------------------------------------------------------
# 7. SUMMARIZE BY DURATION
# ------------------------------------------------------------

enriched_54["has_dominant_switch"] = (
    enriched_54["natural_dominant_switches"] > 0
)

enriched_54["has_tied_day"] = (
    enriched_54["dominant_tied_days"] > 0
)

summary_54 = (
    enriched_54
    .groupby("window_workdays")
    .agg(
        Windows=("user", "size"),
        Users=("user", "nunique"),
        Windows_with_switches=("has_dominant_switch", "sum"),
        Median_dominant_switches=(
            "natural_dominant_switches", "median"
        ),
        Median_set_changes=(
            "natural_active_set_changes", "median"
        ),
        Windows_with_ties=("has_tied_day", "sum")
    )
    .reset_index()
    .rename(columns={"window_workdays": "Duration"})
)

summary_54["Switching_windows_pct"] = (
    100 * summary_54["Windows_with_switches"]
    / summary_54["Windows"]
).round(2)

# ------------------------------------------------------------
# 8. SAVE ENRICHED WINDOW-LEVEL AUDIT
# ------------------------------------------------------------

# Do not save temporary display/helper columns.
enriched_54 = enriched_54.drop(
    columns=["has_dominant_switch", "has_tied_day"]
)

enriched_54.to_parquet(
    OUTPUT_PATH_54,
    index=False
)

# ------------------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("CELL 7 — NATURAL DOMINANT-CHANNEL SWITCHING")
print("=" * 80)

print("\nA. DAILY DOMINANCE DIAGNOSTICS")
print("-" * 80)

print("Candidate user-days :", f"{len(daily_54):,}")

for name, value in daily_diagnostics_54.items():
    print(f"{name:30s}: {value:,}")

print("\nB. DOMINANT SWITCHING BY DURATION")
print("-" * 80)

print(summary_54.round(2).to_string(index=False))

print("\nC. EXAMPLE OF FINAL WINDOW RECORDS")
print("-" * 80)

sample_cols_54 = [
    "user",
    "window_workdays",
    "window_start",
    "observed_channels",
    "natural_active_set_changes",
    "dominant_channels_observed",
    "natural_dominant_switches",
    "natural_dominant_switch_rate_pct"
]

print(
    enriched_54[sample_cols_54]
    .head(6)
    .to_string(index=False)
)

print("\nVALIDATION")
print("-" * 80)
print("995 candidate users             : PASS")
print("472,667 original windows        : PASS")
print("Dominance normalization         : PASS")
print("Ties and inactive days handled  : PASS")
print("Inside-window switch boundaries : PASS")
print("Notebook 02 capacity comparison : PASS")
print("Existing Cell 6 data preserved  : PASS")
print("Injection performed             : NO")
print("Original source files modified  : NO")

print("\nSAVED")
print("File :", OUTPUT_PATH_54)
print("Shape:", enriched_54.shape)
print("Key  : user + window_workdays + window_start")


CELL 7 — NATURAL DOMINANT-CHANNEL SWITCHING

A. DAILY DOMINANCE DIAGNOSTICS
--------------------------------------------------------------------------------
Candidate user-days : 103,395
unique_dominant_days          : 75,069
tied_dominant_days            : 1,522
no_positive_activity_days     : 26,804

B. DOMINANT SWITCHING BY DURATION
--------------------------------------------------------------------------------
 Duration  Windows  Users  Windows_with_switches  Median_dominant_switches  Median_set_changes  Windows_with_ties  Switching_windows_pct
        5    99415    995                  61547                       1.0                 1.0               6386                  61.91
       10    94444    993                  79737                       2.0                 3.0              10542                  84.43
       20    84518    992                  81357                       5.0                 7.0              15094                  96.26
       30    74602    988        


### 5.2.3 Switching Rule Feasibility & Range Comparison

#### Objective
Evaluate the mathematical feasibility and preliminary natural-channel support of four candidate Primary-Channel Switching ranges across the six approved attack durations.

This evaluation focuses on selecting suitable simulation rules, without assigning users, selecting injection dates, or generating attack schedules.

#### Switching Definition
A Primary-Channel Switch occurs when the selected primary injection channel changes between consecutive attack-active days.

Switching Rate = Number of Primary-Channel Switches / (Attack-Active Days - 1)

#### Candidate Ranges

| Range | Low | Medium | High |
|---|---|---|---|
| A | 10–20% | 25–40% | 45–60% |
| B | 15–25% | 30–45% | 50–65% |
| C | 10–25% | 30–45% | 50–65% |
| D | 15–30% | 35–50% | 55–70% |

#### Evaluation Approach

1. Test all six durations: 5, 10, 20, 30, 40, and 50 eligible workdays.
2. Evaluate possible attack-active-day counts from 2 to the window duration, without imposing Off-Day quotas.
3. Convert each switching-rate interval into valid integer switching counts.
4. Compare range feasibility and the diversity of integer switching options.
5. Use Notebook 02's existing rotation-capacity results to assess preliminary support for required switching counts.
6. Evaluate the five-day duration separately and reconsider the V1 exception.

#### Important Distinction
A mathematically valid switching count is not automatically executable.

Notebook 02's rotation metrics indicate whether naturally active channels can support alternating steps. They do not guarantee a complete attack schedule satisfying a particular rate and attack-active-day count.

#### Expected Outcomes
- A comparative assessment of Ranges A, B, C, and D.
- Identification of infeasible switching-rate combinations.
- Preliminary evidence of natural rotation support.
- A justified recommendation concerning the five-day exception.
- Evidence for selecting structural switching rules before scenario design.

#### Scope
This is a read-only rule-evaluation stage.

No attack-active days, primary channels, users, or injection schedules are selected.

Dominant-channel switching results from Cell 7 remain descriptive behavioral references, but are not used to establish the arithmetic feasibility of Primary-Channel Switching.


In [5]:

# ============================================================
# CELL 8 — CORRECTED PRIMARY SWITCHING RULE EVALUATION
# Integer feasibility + natural rotation-capacity screening
# No schedule generation, assignment, or injection
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT_55 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORTS_55 = ROOT_55 / "reports"

OUTPUT_DIR_55 = (
    REPORTS_55 / "05_Simulation_Rules_Feasibility"
    / "02_Switching"
)

CANDIDATE_PATH_55 = (
    REPORTS_55 / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

WINDOW_PATH_55 = (
    REPORTS_55 / "02_Temporal_Window_Audit"
    / "user_window_length_audit.parquet"
)

DURATIONS_55 = [5, 10, 20, 30, 40, 50]
LEVELS_55 = ["Low", "Medium", "High"]

RANGES_55 = {
    "A": {
        "Low": (0.10, 0.20),
        "Medium": (0.25, 0.40),
        "High": (0.45, 0.60)
    },
    "B": {
        "Low": (0.15, 0.25),
        "Medium": (0.30, 0.45),
        "High": (0.50, 0.65)
    },
    "C": {
        "Low": (0.10, 0.25),
        "Medium": (0.30, 0.45),
        "High": (0.50, 0.65)
    },
    "D": {
        "Low": (0.15, 0.30),
        "Medium": (0.35, 0.50),
        "High": (0.55, 0.70)
    }
}

STEP_THRESHOLDS_55 = [
    2, 3, 4, 5, 6, 8, 10, 15, 20, 30
]

for path in [CANDIDATE_PATH_55, WINDOW_PATH_55]:
    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------------------------------
# 2. LOAD EXISTING AUDIT RESULTS
# ------------------------------------------------------------

candidates_55 = pd.read_parquet(
    CANDIDATE_PATH_55,
    columns=["user"]
)

window_cols_55 = [
    "user",
    "window_workdays",
    "eligible_windows",
    "max_feasible_rotation_steps",
    "max_feasible_switches"
] + [
    f"windows_supporting_{k}steps"
    for k in STEP_THRESHOLDS_55
]

windows_55 = pd.read_parquet(
    WINDOW_PATH_55,
    columns=window_cols_55
)

for df in [candidates_55, windows_55]:
    df["user"] = (
        df["user"].astype("string").str.strip()
    )

assert len(candidates_55) == 995
assert candidates_55["user"].is_unique

candidate_users_55 = set(candidates_55["user"])

selected_55 = windows_55.loc[
    windows_55["user"].isin(candidate_users_55)
    & windows_55["window_workdays"].isin(DURATIONS_55)
].copy()

assert not selected_55.duplicated(
    ["user", "window_workdays"]
).any()

# Include all 995 users for all six durations.
all_pairs_55 = pd.MultiIndex.from_product(
    [
        sorted(candidate_users_55),
        DURATIONS_55
    ],
    names=["user", "window_workdays"]
)

user_duration_55 = (
    selected_55
    .set_index(["user", "window_workdays"])[
        window_cols_55[2:]
    ]
    .reindex(all_pairs_55, fill_value=0)
    .reset_index()
)

assert len(user_duration_55) == 995 * 6
assert user_duration_55[window_cols_55[2:]].notna().all().all()

# ------------------------------------------------------------
# 3. INTEGER SWITCHING FEASIBILITY
# ------------------------------------------------------------

def valid_switch_counts_55(active_days, minimum, maximum):
    """
    Return all positive integer switch counts satisfying:
    minimum <= switches / (active_days - 1) <= maximum
    """
    transitions = active_days - 1

    if transitions < 1:
        return ()

    return tuple(
        s for s in range(1, transitions + 1)
        if minimum - 1e-12 <= s / transitions
        <= maximum + 1e-12
    )

cases_55 = []

for range_id, bands in RANGES_55.items():

    for duration in DURATIONS_55:

        # No predetermined Off-Day quota.
        # These are mathematical possibilities only.
        for active_days in range(2, duration + 1):

            for level in LEVELS_55:

                low, high = bands[level]

                switches = valid_switch_counts_55(
                    active_days, low, high
                )

                cases_55.append({
                    "range_id": range_id,
                    "duration": duration,
                    "attack_active_days": active_days,
                    "level": level,
                    "min_rate": low,
                    "max_rate": high,
                    "valid_switch_counts": ",".join(
                        str(s) for s in switches
                    ),
                    "option_count": len(switches),
                    "integer_feasible": bool(switches),
                    "min_switches": (
                        min(switches) if switches else np.nan
                    ),
                    "max_switches": (
                        max(switches) if switches else np.nan
                    )
                })

integer_cases_55 = pd.DataFrame(cases_55)

CASE_KEYS_55 = [
    "range_id", "duration",
    "attack_active_days", "level"
]

assert not integer_cases_55.duplicated(
    CASE_KEYS_55
).any()

assert len(integer_cases_55) == (
    4 * 3 * sum(d - 1 for d in DURATIONS_55)
)

# ------------------------------------------------------------
# 4. COMPARE RANGE DIVERSITY & FEASIBILITY
# ------------------------------------------------------------

range_summary_55 = (
    integer_cases_55
    .groupby("range_id")
    .agg(
        Level_cases=("level", "size"),
        Feasible_level_cases=("integer_feasible", "sum"),
        Total_integer_options=("option_count", "sum")
    )
    .reset_index()
    .rename(columns={"range_id": "Range"})
)

range_summary_55["Infeasible_level_cases"] = (
    range_summary_55["Level_cases"]
    - range_summary_55["Feasible_level_cases"]
)

# Count active-day configurations that support
# all three levels under the SAME active-day count.
all_levels_55 = (
    integer_cases_55
    .groupby([
        "range_id", "duration", "attack_active_days"
    ])["integer_feasible"]
    .all()
    .groupby("range_id")
    .sum()
)

range_summary_55["Active_day_configs_all_3_levels"] = (
    range_summary_55["Range"]
    .map(all_levels_55)
    .astype(int)
)

# ------------------------------------------------------------
# 5. FIVE-DAY DIAGNOSTIC
# ------------------------------------------------------------

five_day_55 = integer_cases_55.loc[
    integer_cases_55["duration"].eq(5)
].copy()

five_day_table_55 = (
    five_day_55.loc[
        five_day_55["range_id"].isin(["C", "D"])
    ]
    .pivot(
        index=["range_id", "attack_active_days"],
        columns="level",
        values="valid_switch_counts"
    )
    .reset_index()
    .rename(columns={
        "range_id": "Range",
        "attack_active_days": "Active_days"
    })
)

five_day_table_55 = five_day_table_55[
    ["Range", "Active_days", "Low", "Medium", "High"]
]

# Check the V1 exception at exactly five active days.
V1_EXCEPTION_55 = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

v1_exception_rows_55 = []

for level, switches in V1_EXCEPTION_55.items():

    actual_rate = switches / 4
    low, high = RANGES_55["C"][level]

    v1_exception_rows_55.append({
        "Level": level,
        "V1_switches": switches,
        "Actual_rate_pct": round(actual_rate * 100, 2),
        "Within_original_C_range": (
            low - 1e-12 <= actual_rate
            <= high + 1e-12
        )
    })

v1_exception_55 = pd.DataFrame(v1_exception_rows_55)

# ------------------------------------------------------------
# 6. NOTEBOOK 02 — NATURAL ROTATION SUPPORT
# ------------------------------------------------------------

# Diagnostic only:
# Use all eligible workdays as potential attack-active
# days (A = Duration) to compare the rules at one
# common mathematical reference point.
#
# This is NOT a required injection density.
#
# If a rate needs S switches, it requires at least
# S+1 alternating channel steps somewhere in the
# candidate window.
#
# This is NECESSARY, but NOT SUFFICIENT, to execute
# a complete schedule with A active days.

full_day_cases_55 = integer_cases_55.loc[
    integer_cases_55["attack_active_days"]
    == integer_cases_55["duration"]
].copy()

support_rows_55 = []

for case in full_day_cases_55.itertuples(index=False):

    duration = int(case.duration)

    group = user_duration_55.loc[
        user_duration_55["window_workdays"].eq(duration)
    ]

    total_windows = int(group["eligible_windows"].sum())

    row = {
        "Range": case.range_id,
        "Duration": duration,
        "Level": case.level,
        "Active_days_tested": duration,
        "Integer_feasible": bool(case.integer_feasible),
        "Min_required_switches": None,
        "Max_allowed_switches": None,
        "Users_supporting_min_steps": 0,
        "Rotation_window_lower_bound": 0,
        "Rotation_window_upper_bound": 0
    }

    if case.integer_feasible:

        minimum_switches = int(case.min_switches)
        maximum_switches = int(case.max_switches)

        # S switches need at least S+1 alternating steps.
        required_steps = minimum_switches + 1

        row["Min_required_switches"] = minimum_switches
        row["Max_allowed_switches"] = maximum_switches

        row["Users_supporting_min_steps"] = int(
            group["max_feasible_rotation_steps"]
            .ge(required_steps)
            .sum()
        )

        # Notebook 02 stores exact window counts for
        # selected step thresholds, not every k.
        #
        # If k is unavailable, use valid bounds:
        # windows supporting next higher threshold
        # <= windows supporting k
        # <= windows supporting previous lower threshold.

        lower_threshold = next(
            (
                k for k in STEP_THRESHOLDS_55
                if k >= required_steps
            ),
            None
        )

        upper_threshold = next(
            (
                k for k in reversed(STEP_THRESHOLDS_55)
                if k <= required_steps
            ),
            None
        )

        if lower_threshold is not None:
            lower_windows = int(
                group[
                    f"windows_supporting_{lower_threshold}steps"
                ].sum()
            )
        else:
            lower_windows = 0

        if upper_threshold is not None:
            upper_windows = int(
                group[
                    f"windows_supporting_{upper_threshold}steps"
                ].sum()
            )
        else:
            upper_windows = total_windows

        assert 0 <= lower_windows <= upper_windows
        assert upper_windows <= total_windows

        row["Rotation_window_lower_bound"] = lower_windows
        row["Rotation_window_upper_bound"] = upper_windows

    support_rows_55.append(row)

full_day_support_55 = pd.DataFrame(support_rows_55)

# ------------------------------------------------------------
# 7. VALIDATION
# ------------------------------------------------------------

assert len(range_summary_55) == 4
assert len(full_day_support_55) == 4 * 6 * 3

assert (
    integer_cases_55["option_count"].gt(0)
    == integer_cases_55["integer_feasible"]
).all()

assert (
    integer_cases_55["attack_active_days"]
    <= integer_cases_55["duration"]
).all()

assert (
    user_duration_55["max_feasible_switches"]
    == (
        user_duration_55["max_feasible_rotation_steps"] - 1
    ).clip(lower=0)
).all()

assert (
    full_day_support_55["Users_supporting_min_steps"]
    <= 995
).all()

assert (
    full_day_support_55["Rotation_window_lower_bound"]
    <= full_day_support_55["Rotation_window_upper_bound"]
).all()

# Verify stored window-support thresholds are nested.
for small, large in zip(
    STEP_THRESHOLDS_55[:-1],
    STEP_THRESHOLDS_55[1:]
):
    assert (
        user_duration_55[
            f"windows_supporting_{large}steps"
        ]
        <= user_duration_55[
            f"windows_supporting_{small}steps"
        ]
    ).all()

# Switching-level intervals do not overlap.
for bands in RANGES_55.values():
    assert bands["Low"][1] < bands["Medium"][0]
    assert bands["Medium"][1] < bands["High"][0]

# ------------------------------------------------------------
# 8. SAVE THE RULE-EVALUATION RESULTS
# ------------------------------------------------------------

OUTPUT_DIR_55.mkdir(parents=True, exist_ok=True)

INTEGER_OUTPUT_55 = (
    OUTPUT_DIR_55
    / "switching_rule_integer_feasibility.parquet"
)

CAPACITY_OUTPUT_55 = (
    OUTPUT_DIR_55
    / "switching_full_density_capacity_screen.parquet"
)

COMPARISON_OUTPUT_55 = (
    OUTPUT_DIR_55
    / "switching_range_comparison.csv"
)

integer_cases_55.to_parquet(
    INTEGER_OUTPUT_55, index=False
)

full_day_support_55.to_parquet(
    CAPACITY_OUTPUT_55, index=False
)

range_summary_55.to_csv(
    COMPARISON_OUTPUT_55, index=False
)

# ------------------------------------------------------------
# 9. CONCISE OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("CELL 8 — CORRECTED PRIMARY-CHANNEL SWITCHING RULES")
print("=" * 80)

print("\nINPUT SCOPE")
print("-" * 80)
print("Candidate users       :", len(candidate_users_55))
print("Durations             :", DURATIONS_55)
print("User-duration pairs   :", len(user_duration_55))
print("Range alternatives    :", list(RANGES_55))
print("No predefined Off-Days: YES")

print("\nA. RANGE COMPARISON — INTEGER FEASIBILITY")
print("-" * 80)
print(range_summary_55.to_string(index=False))

print("\nB. FIVE-DAY DURATION — RANGES C AND D")
print("-" * 80)
print(five_day_table_55.to_string(index=False))

print("\nC. V1 FIVE-DAY EXCEPTION")
print("-" * 80)
print(v1_exception_55.to_string(index=False))

print("\nD. RANGE C — PRELIMINARY ROTATION CAPACITY")
print("Reference only: Active days = Duration")
print("-" * 80)

print(
    full_day_support_55.loc[
        full_day_support_55["Range"].eq("C"),
        [
            "Duration",
            "Level",
            "Integer_feasible",
            "Min_required_switches",
            "Users_supporting_min_steps",
            "Rotation_window_lower_bound",
            "Rotation_window_upper_bound"
        ]
    ].to_string(index=False)
)

print("\nVALIDATION")
print("-" * 80)
print("995 candidates                 : PASS")
print("Six durations                  : PASS")
print("Integer switching arithmetic   : PASS")
print("Non-overlapping ranges         : PASS")
print("Notebook 02 capacity checks    : PASS")
print("No injection-day selection     : PASS")
print("No scenario assignment         : PASS")
print("Source data modified           : NO")

print("\nSAVED OUTPUTS")
print("-" * 80)

for path in [
    INTEGER_OUTPUT_55,
    CAPACITY_OUTPUT_55,
    COMPARISON_OUTPUT_55
]:
    print(path.name)


CELL 8 — CORRECTED PRIMARY-CHANNEL SWITCHING RULES

INPUT SCOPE
--------------------------------------------------------------------------------
Candidate users       : 995
Durations             : [5, 10, 20, 30, 40, 50]
User-duration pairs   : 5970
Range alternatives    : ['A', 'B', 'C', 'D']
No predefined Off-Days: YES

A. RANGE COMPARISON — INTEGER FEASIBILITY
--------------------------------------------------------------------------------
Range  Level_cases  Feasible_level_cases  Total_integer_options  Infeasible_level_cases  Active_day_configs_all_3_levels
    A          447                   399                   1134                      48                              125
    B          447                   394                   1134                      53                              120
    C          447                   399                   1273                      48                              125
    D          447                   399                   1255      


### 5.3 Same-Day Multichannel Activity Feasibility

#### Objective
Evaluate whether same-day multichannel activity should be mandatory or optional in attack simulation scenarios, using the observed natural activity of the 995 candidate users.

#### Definitions
- **Single-Channel Day:** Only one of Email, USB, or HTTP is naturally active.
- **Two-Channel Day:** Exactly two target channels are naturally active on the same eligible workday.
- **Three-Channel Day:** All three target channels are naturally active on the same eligible workday.
- **Multichannel Day:** At least two target channels are naturally active on the same eligible workday.

#### Proposed Rule
- Zero multichannel injection days must be allowed.
- One or more multichannel injection days may be allowed when naturally supported.
- Three-channel same-day activity must not be mandatory.
- Injection must never use a channel that is naturally inactive on the selected day.
- Switching between primary channels remains independent of same-day multichannel injection.

#### Evaluation
1. Reuse the 995 candidate users and 472,667 actual windows.
2. Measure naturally available two-channel and three-channel days inside each window.
3. Compare optional multichannel activity against hypothetical minimum requirements of 1, 2, or 3 multichannel days.
4. Measure the resulting window and user coverage across all six durations.
5. Assess whether mandatory same-day multichannel activity introduces unnecessary restrictions.

#### Scope
This is a rule-feasibility analysis, not an attack scheduling stage.

No attack dates, injection channels, users, or scenarios are assigned.

#### Expected Outcome
Determine whether Same-Day Multichannel should remain an optional simulation dimension, supported by measured user and window coverage.


In [6]:

# ============================================================
# CELL 9 — SAME-DAY MULTICHANNEL RULE FEASIBILITY
# Natural availability and impact of mandatory requirements
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT_59 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_59 = (
    ROOT_59 / "reports" / "05_Simulation_Rules_Feasibility"
    / "03_Same_Day_Multichannel"
)

PROFILE_PATH_59 = (
    ROOT_59 / "reports" / "01_User_Population_Audit"
    / "user_population_profile.parquet"
)

WINDOW_PATH_59 = (
    REPORT_59.parent / "02_Switching" / "natural_window_switching_enriched.parquet"
)

DAILY_PATH_59 = (
    ROOT_59.parent / "Data_Engineering"
    / "Data_engineering" / "data" / "features"
    / "behavior_features_r62_v1.parquet"
)

for path in [PROFILE_PATH_59, WINDOW_PATH_59, DAILY_PATH_59]:
    assert path.is_file(), f"Missing input: {path}"

DURATIONS_59 = [5, 10, 20, 30, 40, 50]

# ------------------------------------------------------------
# 2. LOAD VALIDATED SOURCES
# ------------------------------------------------------------

profile_59 = pd.read_parquet(
    PROFILE_PATH_59,
    columns=["user", "active_channel_count",
             "days_2ch", "days_3ch"]
)

candidates_59 = set(
    profile_59.loc[
        profile_59["active_channel_count"] >= 2, "user"
    ]
)

assert len(candidates_59) == 995

windows_59 = pd.read_parquet(
    WINDOW_PATH_59,
    columns=[
        "user", "window_workdays",
        "window_start", "window_end",
        "supports_2step"
    ]
)

assert len(windows_59) == 472667
assert windows_59["user"].isin(candidates_59).all()
assert windows_59["user"].nunique() == 995
assert set(windows_59["window_workdays"]) == set(DURATIONS_59)

windows_59["window_start"] = pd.to_datetime(
    windows_59["window_start"]
)
windows_59["window_end"] = pd.to_datetime(
    windows_59["window_end"]
)

KEYS_59 = ["user", "window_workdays", "window_start"]

assert not windows_59.duplicated(KEYS_59).any()
assert windows_59.notna().all().all()

daily_59 = pd.read_parquet(
    DAILY_PATH_59,
    columns=[
        "user", "day",
        "external_email_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

daily_59 = daily_59.loc[
    daily_59["user"].isin(candidates_59)
].copy()

daily_59["day"] = pd.to_datetime(daily_59["day"])

daily_59 = daily_59.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert len(daily_59) == 103395
assert not daily_59.duplicated(["user", "day"]).any()

# ------------------------------------------------------------
# 3. DAILY NATURAL CHANNEL COMBINATIONS
# ------------------------------------------------------------

count_cols_59 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

assert daily_59[count_cols_59].notna().all().all()
assert daily_59[count_cols_59].ge(0).all().all()

# Bit encoding:
# Email = 1, USB = 2, HTTP = 4
#
# E=1, U=2, EU=3, H=4, EH=5, UH=6, EUH=7

flags_59 = daily_59[count_cols_59].gt(0)

daily_59["channel_state"] = (
    flags_59.to_numpy(dtype=np.int8)
    @ np.array([1, 2, 4], dtype=np.int8)
)

daily_59["exactly_2"] = (
    daily_59["channel_state"].isin([3, 5, 6])
).astype(np.int8)

daily_59["exactly_3"] = (
    daily_59["channel_state"].eq(7)
).astype(np.int8)

# Reconcile with Notebook 01
profile_check_59 = profile_59.set_index("user")

user_counts_59 = (
    daily_59.groupby("user")[["exactly_2", "exactly_3"]]
    .sum()
)

assert (
    user_counts_59["exactly_2"]
    == profile_check_59.loc[
        user_counts_59.index, "days_2ch"
    ]
).all()

assert (
    user_counts_59["exactly_3"]
    == profile_check_59.loc[
        user_counts_59.index, "days_3ch"
    ]
).all()

# ------------------------------------------------------------
# 4. COUNT SAME-DAY OVERLAP INSIDE ACTUAL WINDOWS
# ------------------------------------------------------------

parts_59 = []

metric_names_59 = [
    "days_2plus",
    "days_3ch",
    "days_EU",
    "days_EH",
    "days_UH",
    "days_EUH"
]

daily_groups_59 = {
    user: group
    for user, group in daily_59.groupby("user", sort=False)
}

for user, group in windows_59.groupby("user", sort=False):

    source = daily_groups_59[user]

    dates = source["day"].to_numpy(dtype="datetime64[ns]")
    state = source["channel_state"].to_numpy()

    metrics = np.column_stack([
        np.isin(state, [3, 5, 6, 7]),
        state == 7,
        state == 3,
        state == 5,
        state == 6,
        state == 7
    ]).astype(np.int32)

    prefix = np.vstack([
        np.zeros((1, len(metric_names_59)), dtype=np.int32),
        np.cumsum(metrics, axis=0, dtype=np.int32)
    ])

    starts = group["window_start"].to_numpy(
        dtype="datetime64[ns]"
    )
    ends = group["window_end"].to_numpy(
        dtype="datetime64[ns]"
    )

    start_idx = np.searchsorted(dates, starts)
    end_idx = np.searchsorted(dates, ends, side="right")

    assert (start_idx < len(dates)).all()
    assert (end_idx > 0).all()
    assert np.array_equal(dates[start_idx], starts)
    assert np.array_equal(dates[end_idx - 1], ends)

    # Exact eligible-workday window length
    assert np.array_equal(
        end_idx - start_idx,
        group["window_workdays"].to_numpy(dtype=int)
    )

    counts = prefix[end_idx] - prefix[start_idx]

    part = group.copy()

    for i, name in enumerate(metric_names_59):
        part[name] = counts[:, i]

    part["days_exactly_2ch"] = (
        part["days_EU"]
        + part["days_EH"]
        + part["days_UH"]
    )

    assert (
        part["days_2plus"]
        == part["days_exactly_2ch"] + part["days_3ch"]
    ).all()

    parts_59.append(part)

audit_59 = pd.concat(parts_59, ignore_index=True)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

assert len(audit_59) == 472667
assert not audit_59.duplicated(KEYS_59).any()
assert audit_59[metric_names_59].ge(0).all().all()

assert (
    audit_59["days_2plus"]
    <= audit_59["window_workdays"]
).all()

assert (
    audit_59["days_3ch"]
    <= audit_59["days_2plus"]
).all()

assert (
    audit_59["days_3ch"] == audit_59["days_EUH"]
).all()

# ------------------------------------------------------------
# 6. COMPARE OPTIONAL VS MANDATORY RULES
# ------------------------------------------------------------

summary_rows_59 = []

for duration, group in audit_59.groupby("window_workdays"):

    # Focus on windows already supporting natural
    # two-step rotation (not yet complete attack schedules).
    eligible = group.loc[
        group["supports_2step"].astype(bool)
    ]

    total = len(eligible)
    total_users = eligible["user"].nunique()

    for requirement, mask in [
        ("Optional / zero allowed",
         pd.Series(True, index=eligible.index)),
        ("Require >= 1 multichannel day",
         eligible["days_2plus"] >= 1),
        ("Require >= 2 multichannel days",
         eligible["days_2plus"] >= 2),
        ("Require >= 3 multichannel days",
         eligible["days_2plus"] >= 3),
        ("Require >= 1 three-channel day",
         eligible["days_3ch"] >= 1)
    ]:

        retained = eligible.loc[mask]
        kept_windows = len(retained)
        kept_users = retained["user"].nunique()

        summary_rows_59.append({
            "Duration": duration,
            "Requirement": requirement,
            "Retained_windows": kept_windows,
            "Window_coverage_pct": round(
                100 * kept_windows / total, 2
            ) if total else 0,
            "Users_with_a_window": kept_users,
            "Users_lost_vs_optional": (
                total_users - kept_users
            )
        })

summary_59 = pd.DataFrame(summary_rows_59)

# Compact comparison of key conditions
overview_59 = (
    summary_59.loc[
        summary_59["Requirement"].isin([
            "Optional / zero allowed",
            "Require >= 1 multichannel day",
            "Require >= 1 three-channel day"
        ])
    ]
    .copy()
)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

REPORT_59.mkdir(parents=True, exist_ok=True)

audit_path_59 = (
    REPORT_59 / "same_day_multichannel_window_audit.parquet"
)

summary_path_59 = (
    REPORT_59 / "same_day_multichannel_rule_comparison.csv"
)

audit_59.to_parquet(audit_path_59, index=False)
summary_59.to_csv(summary_path_59, index=False)

# ------------------------------------------------------------
# 8. OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("CELL 9 — SAME-DAY MULTICHANNEL FEASIBILITY")
print("=" * 80)

print("\nINPUT")
print("-" * 80)
print("Candidate users     :", len(candidates_59))
print("Actual windows      :", f"{len(audit_59):,}")
print("Durations           :", DURATIONS_59)

print("\nA. NATURAL MULTICHANNEL AVAILABILITY")
print("-" * 80)
print("Users with >=1 two-channel day:",
      int(user_counts_59["exactly_2"].gt(0).sum()))
print("Users with >=1 three-channel day:",
      int(user_counts_59["exactly_3"].gt(0).sum()))
print("Users with no same-day multichannel activity:",
      int(
          (
              user_counts_59["exactly_2"]
              + user_counts_59["exactly_3"]
          ).eq(0).sum()
      ))

print("\nB. OPTIONAL VS MANDATORY — KEY COMPARISON")
print("-" * 80)
print(overview_59.to_string(index=False))

print("\nC. ALL TESTED REQUIREMENTS")
print("-" * 80)
print(summary_59.to_string(index=False))

print("\nVALIDATION")
print("-" * 80)
print("Notebook 01 user counts matched : PASS")
print("Actual window boundaries        : PASS")
print("Window overlap counts           : PASS")
print("Original data modified          : NO")
print("Injection dates selected        : NO")
print("Attack scenarios assigned       : NO")

print("\nSAVED")
print("-" * 80)
print(audit_path_59.name, audit_59.shape)
print(summary_path_59.name, summary_59.shape)


CELL 9 — SAME-DAY MULTICHANNEL FEASIBILITY

INPUT
--------------------------------------------------------------------------------
Candidate users     : 995
Actual windows      : 472,667
Durations           : [5, 10, 20, 30, 40, 50]

A. NATURAL MULTICHANNEL AVAILABILITY
--------------------------------------------------------------------------------
Users with >=1 two-channel day: 985
Users with >=1 three-channel day: 239
Users with no same-day multichannel activity: 10

B. OPTIONAL VS MANDATORY — KEY COMPARISON
--------------------------------------------------------------------------------
 Duration                    Requirement  Retained_windows  Window_coverage_pct  Users_with_a_window  Users_lost_vs_optional
        5        Optional / zero allowed             74244               100.00                  995                       0
        5  Require >= 1 multichannel day             61112                82.31                  984                      11
        5 Require >= 1 thr

In [7]:

# ============================================================
# CELL 9.1 — INSPECT SAVED MULTICHANNEL RESULTS
# Read-only inspection; no new feasibility analysis
# ============================================================

from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT_591 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_591 = (
    ROOT_591 / "reports" / "05_Simulation_Rules_Feasibility"
    / "03_Same_Day_Multichannel"
)

audit_591 = pd.read_parquet(
    REPORT_591 / "same_day_multichannel_window_audit.parquet"
)

comparison_591 = pd.read_csv(
    REPORT_591 / "same_day_multichannel_rule_comparison.csv"
)

assert audit_591.shape == (472667, 12)
assert comparison_591.shape == (30, 6)

# ------------------------------------------------------------
# A. ACTUAL WINDOW SAMPLES — 20-DAY WINDOWS
# ------------------------------------------------------------

cols_591 = [
    "user", "window_workdays",
    "window_start", "window_end",
    "supports_2step",
    "days_EU", "days_EH", "days_UH",
    "days_EUH", "days_exactly_2ch",
    "days_3ch", "days_2plus"
]

base_591 = (
    audit_591["window_workdays"].eq(20)
    & audit_591["supports_2step"].astype(bool)
)

def sample_windows_591(label, condition, n=3):
    result = (
        audit_591.loc[
            base_591 & condition, cols_591
        ]
        .sort_values(["user", "window_start"])
        .head(n)
        .copy()
    )
    result.insert(0, "Example", label)
    return result

samples_591 = pd.concat([
    sample_windows_591(
        "Exactly 2 channels",
        (audit_591["days_exactly_2ch"] > 0)
        & (audit_591["days_3ch"] == 0)
    ),
    sample_windows_591(
        "3 channels",
        audit_591["days_3ch"] > 0
    ),
    sample_windows_591(
        "No multichannel days",
        audit_591["days_2plus"] == 0
    )
], ignore_index=True)

print("A. ACTUAL WINDOW SAMPLES (20 WORKDAYS)")
display(samples_591)

# ------------------------------------------------------------
# B. USER-LEVEL SUPPORT SUMMARY
# ------------------------------------------------------------

temp_591 = audit_591.assign(
    has_exactly_2ch=(
        audit_591["days_exactly_2ch"] > 0
    ),
    has_3ch=(
        audit_591["days_3ch"] > 0
    ),
    rotation_and_2plus=(
        audit_591["supports_2step"].astype(bool)
        & audit_591["days_2plus"].gt(0)
    ),
    rotation_and_3ch=(
        audit_591["supports_2step"].astype(bool)
        & audit_591["days_3ch"].gt(0)
    )
)

users_591 = (
    temp_591.groupby("user", as_index=False)
    .agg(
        total_windows=("window_workdays", "size"),
        windows_exactly_2ch=("has_exactly_2ch", "sum"),
        windows_3ch=("has_3ch", "sum"),
        windows_rotation_2plus=("rotation_and_2plus", "sum"),
        windows_rotation_3ch=("rotation_and_3ch", "sum")
    )
)

users_591["supports_exactly_2ch"] = (
    users_591["windows_exactly_2ch"] > 0
)

users_591["supports_3ch"] = (
    users_591["windows_3ch"] > 0
)

print("\nB. USER SUPPORT TOTALS")
print("Total candidates:", len(users_591))
print(
    "Users supporting exactly 2 channels:",
    users_591["supports_exactly_2ch"].sum()
)
print(
    "Users supporting 3 channels:",
    users_591["supports_3ch"].sum()
)
print(
    "Users with no multichannel days:",
    (
        ~users_591["supports_exactly_2ch"]
        & ~users_591["supports_3ch"]
    ).sum()
)

print("\nC. SAMPLE USER IDs — THREE CHANNELS")
display(
    users_591.loc[users_591["supports_3ch"]]
    .sort_values("windows_3ch", ascending=False)
    .head(5)
)

print("\nD. SAMPLE USER IDs — EXACTLY TWO, NEVER THREE")
display(
    users_591.loc[
        users_591["supports_exactly_2ch"]
        & ~users_591["supports_3ch"]
    ]
    .sort_values("windows_exactly_2ch", ascending=False)
    .head(5)
)

print("\nE. USERS WITH NO SAME-DAY MULTICHANNEL")
display(
    users_591.loc[
        ~users_591["supports_exactly_2ch"]
        & ~users_591["supports_3ch"]
    ]
)

# ------------------------------------------------------------
# F. EXISTING RULE-COMPARISON SAMPLE
# ------------------------------------------------------------

print("\nF. SAVED RULE COMPARISON — 20 DAYS")
display(
    comparison_591.loc[
        comparison_591["Duration"] == 20
    ]
)

print("\nINSPECTION COMPLETE — NO FILES MODIFIED")


A. ACTUAL WINDOW SAMPLES (20 WORKDAYS)


,Example,user,window_workdays,window_start,window_end,supports_2step,days_EU,days_EH,days_UH,days_EUH,days_exactly_2ch,days_3ch,days_2plus
0,Exactly 2 channels,AAC0610,20,2011-01-03,2011-01-28,True,2,0,0,0,2,0,2
1,Exactly 2 channels,AAC0610,20,2011-01-04,2011-01-31,True,2,0,0,0,2,0,2
2,Exactly 2 channels,AAC0610,20,2011-01-05,2011-02-01,True,2,0,0,0,2,0,2
3,3 channels,ABD3426,20,2011-01-03,2011-01-28,True,15,0,0,5,15,5,20
4,3 channels,ABD3426,20,2011-01-04,2011-01-31,True,15,0,0,5,15,5,20
5,3 channels,ABD3426,20,2011-01-05,2011-02-01,True,14,0,1,5,15,5,20
6,No multichannel days,AAC0610,20,2011-01-19,2011-02-15,True,0,0,0,0,0,0,0
7,No multichannel days,AAS2987,20,2011-03-30,2011-04-27,True,0,0,0,0,0,0,0
8,No multichannel days,AAS2987,20,2011-03-31,2011-04-28,True,0,0,0,0,0,0,0



B. USER SUPPORT TOTALS
Total candidates: 995
Users supporting exactly 2 channels: 985
Users supporting 3 channels: 239
Users with no multichannel days: 10

C. SAMPLE USER IDs — THREE CHANNELS


,user,total_windows,windows_exactly_2ch,windows_3ch,windows_rotation_2plus,windows_rotation_3ch,supports_exactly_2ch,supports_3ch
916,TVS0520,481,478,469,481,469,True,True
540,KDC0444,487,485,468,487,468,True,True
828,SDH2394,481,481,464,481,464,True,True
641,MIC2325,481,481,451,481,451,True,True
5,ABD3426,487,487,448,487,448,True,True



D. SAMPLE USER IDs — EXACTLY TWO, NEVER THREE


,user,total_windows,windows_exactly_2ch,windows_3ch,windows_rotation_2plus,windows_rotation_3ch,supports_exactly_2ch,supports_3ch
216,CRM0139,499,499,0,499,0,True,False
491,JKC2754,493,493,0,493,0,True,False
778,REJ2666,493,493,0,493,0,True,False
195,CJM0273,493,493,0,493,0,True,False
896,TJS3067,493,492,0,492,0,True,False



E. USERS WITH NO SAME-DAY MULTICHANNEL


,user,total_windows,windows_exactly_2ch,windows_3ch,windows_rotation_2plus,windows_rotation_3ch,supports_exactly_2ch,supports_3ch
70,APP3678,481,0,0,0,0,False,False
108,BDP3912,481,0,0,0,0,False,False
142,BRD3314,481,0,0,0,0,False,False
347,FRG1576,481,0,0,0,0,False,False
462,JBB0847,481,0,0,0,0,False,False
816,SAD3259,481,0,0,0,0,False,False
827,SDH1556,487,0,0,0,0,False,False
843,SKB3336,481,0,0,0,0,False,False
854,SOM2865,481,0,0,0,0,False,False
865,SWO3258,481,0,0,0,0,False,False



F. SAVED RULE COMPARISON — 20 DAYS


,Duration,Requirement,Retained_windows,Window_coverage_pct,Users_with_a_window,Users_lost_vs_optional
10,20,Optional / zero allowed,82914,100.00,992,0
11,20,Require >= 1 multichannel day,76137,91.83,982,10
12,20,Require >= 2 multichannel days,65772,79.33,950,42
13,20,Require >= 3 multichannel days,54377,65.58,881,111
14,20,Require >= 1 three-channel day,11677,14.08,239,753



INSPECTION COMPLETE — NO FILES MODIFIED



### 5.4 Off-Days Rule Feasibility — V1 vs. V2

#### Objective
Evaluate the impact of V1's fixed Off-Day configurations compared with the proposed flexible Off-Day rule for V2.

The analysis uses the 995 candidate users and their 472,667 validated actual windows across six durations.

#### Definitions
- Duration (D): Number of eligible workdays in the attack window.
- Off-Days (O): Eligible workdays without injected attack activity.
- Active Days (A): Number of injection days, calculated as A = D - O.
- Available Channel Day: An eligible workday where at least one target channel is naturally active.
- Forced Off-Day: An eligible workday where no target channel is naturally active.
- Optional Off-Day: A naturally active day that may be left without injection.

#### V1 — Original Fixed Rules

| Duration | Allowed Off-Days |
|---|---|
| 5 | 0 |
| 10 | 1–2 |
| 20 | 2–4 |
| 30 | 4–6 |
| 40 | 6–8 |
| 50 | 8–10 |

Total: 15 duration/off-day configurations.

#### V2 — Proposed Flexible Rule
Do not impose a fixed Off-Day quota before evaluating the user's actual activity.

For a candidate window:

MinOffDays = Duration - AvailableChannelDays

MaxOffDays = Duration - 2

The upper bound assumes at least two injection-active days are needed for a primary-channel switch.

These are preliminary availability bounds, not guarantees of an executable attack schedule.

#### Evaluation
1. Reuse the existing actual windows and natural daily channel activity.
2. Identify the minimum unavoidable Off-Days in each window.
3. Compare each of the 15 original V1 configurations with natural channel availability.
4. Measure how many rotation-supporting windows and users remain under each V1 quota.
5. Compare V1's combined permitted quotas against V2's flexible approach.
6. Identify windows excluded by V1's fixed quotas but not by the flexible rule.

#### Important Limitations
- Off-Days are not the same as weekends or non-eligible days.
- A day without injection preserves its original benign activity.
- Passing this test does not prove that switching targets or stealth constraints can be satisfied.
- No actual injection dates or attack schedules are selected.

#### Expected Outputs
- Window-level Off-Day capacity audit.
- V1 configuration comparison (15 rows).
- V1 versus V2 comparison by duration (6 rows).
- Evidence to support the final Off-Day rule.


In [8]:

# ============================================================
# CELL 10 — OFF-DAYS FEASIBILITY: V1 VS V2
# Natural availability screening only
# No assignment, scheduling, or injection
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT_510 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_510 = (
    ROOT_510 / "reports" / "05_Simulation_Rules_Feasibility"
    / "04_Off_Days"
)

WINDOW_PATH_510 = (
    REPORT_510.parent / "03_Same_Day_Multichannel" / "same_day_multichannel_window_audit.parquet"
)

DAILY_PATH_510 = (
    ROOT_510.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
    / "features"
    / "behavior_features_r62_v1.parquet"
)

# Exact V1 configurations verified in the original notebook.
V1_OFFDAY_RULES_510 = {
    5:  [0],
    10: [1, 2],
    20: [2, 3, 4],
    30: [4, 5, 6],
    40: [6, 7, 8],
    50: [8, 9, 10]
}

assert sum(
    len(v) for v in V1_OFFDAY_RULES_510.values()
) == 15

for path in [WINDOW_PATH_510, DAILY_PATH_510]:
    assert path.is_file(), f"Missing input: {path}"

# ------------------------------------------------------------
# 2. LOAD EXISTING ACTUAL WINDOWS
# ------------------------------------------------------------

windows_510 = pd.read_parquet(WINDOW_PATH_510)

assert windows_510.shape == (472667, 12)
assert windows_510["user"].nunique() == 995
assert set(windows_510["window_workdays"]) == set(
    V1_OFFDAY_RULES_510
)

KEYS_510 = [
    "user", "window_workdays", "window_start"
]

assert not windows_510.duplicated(KEYS_510).any()

for col in ["window_start", "window_end"]:
    windows_510[col] = pd.to_datetime(windows_510[col])

assert windows_510[
    ["user", "window_workdays", "window_start",
     "window_end", "supports_2step"]
].notna().all().all()

# ------------------------------------------------------------
# 3. READ NATURAL DAILY CHANNEL ACTIVITY
# ------------------------------------------------------------

COUNT_COLS_510 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

daily_510 = pd.read_parquet(
    DAILY_PATH_510,
    columns=["user", "day"] + COUNT_COLS_510
)

daily_510 = daily_510.loc[
    daily_510["user"].isin(windows_510["user"].unique())
].copy()

daily_510["day"] = pd.to_datetime(daily_510["day"])

daily_510 = daily_510.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert len(daily_510) == 103395
assert daily_510["user"].nunique() == 995
assert not daily_510.duplicated(["user", "day"]).any()
assert daily_510[COUNT_COLS_510].notna().all().all()
assert daily_510[COUNT_COLS_510].ge(0).all().all()

# A naturally available day needs at least
# one positive target-channel event count.

daily_510["channel_available"] = (
    daily_510[COUNT_COLS_510]
    .gt(0)
    .any(axis=1)
    .astype(np.int8)
)

# ------------------------------------------------------------
# 4. CALCULATE FORCED OFF-DAYS PER ACTUAL WINDOW
# ------------------------------------------------------------

daily_groups_510 = {
    user: group
    for user, group in daily_510.groupby("user", sort=False)
}

parts_510 = []

for user, group in windows_510.groupby("user", sort=False):

    source = daily_groups_510[user]

    dates = source["day"].to_numpy(dtype="datetime64[ns]")

    available = source["channel_available"].to_numpy(
        dtype=np.int32
    )

    prefix = np.concatenate([
        np.array([0], dtype=np.int32),
        np.cumsum(available, dtype=np.int32)
    ])

    starts = group["window_start"].to_numpy(
        dtype="datetime64[ns]"
    )

    ends = group["window_end"].to_numpy(
        dtype="datetime64[ns]"
    )

    left = np.searchsorted(dates, starts)
    right = np.searchsorted(dates, ends, side="right")

    assert (left < len(dates)).all()
    assert (right > 0).all()

    assert np.array_equal(dates[left], starts)
    assert np.array_equal(dates[right - 1], ends)

    # Windows must contain exactly D eligible workdays.
    assert np.array_equal(
        right - left,
        group["window_workdays"].to_numpy(dtype=int)
    )

    part = group.copy()

    part["available_channel_days"] = (
        prefix[right] - prefix[left]
    )

    part["forced_offdays"] = (
        part["window_workdays"]
        - part["available_channel_days"]
    )

    parts_510.append(part)

audit_510 = pd.concat(parts_510, ignore_index=True)

# ------------------------------------------------------------
# 5. VALIDATE WINDOW-LEVEL COUNTS
# ------------------------------------------------------------

assert len(audit_510) == 472667
assert not audit_510.duplicated(KEYS_510).any()

assert audit_510["available_channel_days"].ge(0).all()

assert (
    audit_510["available_channel_days"]
    <= audit_510["window_workdays"]
).all()

assert audit_510["forced_offdays"].ge(0).all()

assert (
    audit_510["forced_offdays"]
    + audit_510["available_channel_days"]
    == audit_510["window_workdays"]
).all()

# All multichannel days must also be available days.
assert (
    audit_510["days_2plus"]
    <= audit_510["available_channel_days"]
).all()

# Reuse previously validated rotation support.
rotation_510 = audit_510["supports_2step"].astype(bool)

assert (
    audit_510.loc[
        rotation_510, "available_channel_days"
    ] >= 2
).all()

# ------------------------------------------------------------
# 6. V2 FLEXIBLE OFF-DAY BOUNDS
# ------------------------------------------------------------

# At least two attack-active days are required
# for a primary-channel switch.
#
# These are availability bounds, NOT selected dates.

audit_510["v2_min_offdays"] = audit_510["forced_offdays"]

audit_510["v2_max_offdays"] = (
    audit_510["window_workdays"] - 2
)

audit_510["v2_flexible_not_excluded"] = rotation_510 & (
    audit_510["v2_min_offdays"]
    <= audit_510["v2_max_offdays"]
)

# ------------------------------------------------------------
# 7. TEST ALL 15 ORIGINAL V1 CONFIGURATIONS
# ------------------------------------------------------------

audit_510["v1_possible_quota_count"] = 0
audit_510["v1_any_quota_not_excluded"] = False

quota_rows_510 = []
comparison_rows_510 = []
forced_rows_510 = []

for duration in sorted(V1_OFFDAY_RULES_510):

    group = audit_510.loc[
        audit_510["window_workdays"].eq(duration)
        & audit_510["v2_flexible_not_excluded"]
    ].copy()

    total_windows = len(group)
    total_users = group["user"].nunique()

    assert total_windows > 0

    forced = group["forced_offdays"].to_numpy()

    quota_masks = []

    # Evaluate each actual V1 quota separately.
    for offdays in V1_OFFDAY_RULES_510[duration]:

        active_days = duration - offdays

        # Basic condition:
        # 1. Enough naturally available days.
        # 2. At least two injection-active days.
        possible = (
            (forced <= offdays)
            & (active_days >= 2)
        )

        quota_masks.append(possible)

        retained = group.loc[possible]

        quota_rows_510.append({
            "Duration": duration,
            "V1_OffDays": offdays,
            "V1_ActiveDays": active_days,
            "Windows_not_excluded": len(retained),
            "Window_retention_pct": round(
                100 * len(retained) / total_windows, 2
            ),
            "Users_with_window": retained["user"].nunique(),
            "Users_lost_vs_V2": (
                total_users - retained["user"].nunique()
            )
        })

    quota_matrix = np.column_stack(quota_masks)

    # Number of V1 quotas supported by each window.
    option_counts = quota_matrix.sum(axis=1)

    # Window retained if ANY permitted V1 quota works.
    v1_any = quota_matrix.any(axis=1)

    audit_510.loc[
        group.index, "v1_possible_quota_count"
    ] = option_counts

    audit_510.loc[
        group.index, "v1_any_quota_not_excluded"
    ] = v1_any

    retained_union = group.loc[v1_any]

    # Compare V1 as a whole against V2's flexible rule.
    comparison_rows_510.append({
        "Duration": duration,
        "V2_windows": total_windows,
        "V2_users": total_users,
        "V1_windows": len(retained_union),
        "V1_users": retained_union["user"].nunique(),
        "Windows_lost_by_V1": (
            total_windows - len(retained_union)
        ),
        "Users_lost_by_V1": (
            total_users - retained_union["user"].nunique()
        ),
        "V1_window_retention_pct": round(
            100 * len(retained_union) / total_windows, 2
        )
    })

    zero_forced = group["forced_offdays"].eq(0)
    oneplus_forced = group["forced_offdays"].ge(1)

    forced_rows_510.append({
        "Duration": duration,
        "Rotation_windows": total_windows,
        "Zero_forced_windows": int(zero_forced.sum()),
        "Oneplus_forced_windows": int(
            oneplus_forced.sum()
        ),
        "Oneplus_forced_pct": round(
            100 * oneplus_forced.mean(), 2
        ),
        "Users_with_zero_forced_window": (
            group.loc[zero_forced, "user"].nunique()
        )
    })

quota_510 = pd.DataFrame(quota_rows_510)
comparison_510 = pd.DataFrame(comparison_rows_510)
forced_summary_510 = pd.DataFrame(forced_rows_510)

# ------------------------------------------------------------
# 8. FINAL VALIDATION
# ------------------------------------------------------------

assert quota_510.shape == (15, 7)
assert comparison_510.shape == (6, 8)
assert forced_summary_510.shape == (6, 6)

assert not quota_510.duplicated(
    ["Duration", "V1_OffDays"]
).any()

assert (
    forced_summary_510["Zero_forced_windows"]
    + forced_summary_510["Oneplus_forced_windows"]
    == forced_summary_510["Rotation_windows"]
).all()

assert (
    comparison_510["V1_windows"]
    <= comparison_510["V2_windows"]
).all()

assert (
    comparison_510["V1_users"]
    <= comparison_510["V2_users"]
).all()

assert (
    audit_510["v1_possible_quota_count"].gt(0)
    == audit_510["v1_any_quota_not_excluded"]
).all()

assert (
    ~audit_510["v1_any_quota_not_excluded"]
    | audit_510["v2_flexible_not_excluded"]
).all()

# As V1 permits more off-days within the same
# duration, the basic availability test cannot
# retain fewer windows.
for duration, group in quota_510.groupby("Duration"):

    ordered = group.sort_values("V1_OffDays")

    assert ordered[
        "Windows_not_excluded"
    ].is_monotonic_increasing

# ------------------------------------------------------------
# 9. SAVE REUSABLE RESULTS
# ------------------------------------------------------------

REPORT_510.mkdir(parents=True, exist_ok=True)

AUDIT_OUTPUT_510 = (
    REPORT_510 / "offday_window_capacity_audit.parquet"
)

QUOTA_OUTPUT_510 = (
    REPORT_510 / "offday_v1_15_quota_comparison.csv"
)

COMPARISON_OUTPUT_510 = (
    REPORT_510 / "offday_v1_vs_v2_summary.csv"
)

audit_510.to_parquet(AUDIT_OUTPUT_510, index=False)

quota_510.to_csv(QUOTA_OUTPUT_510, index=False)

comparison_510.to_csv(
    COMPARISON_OUTPUT_510, index=False
)

# ------------------------------------------------------------
# 10. OUTPUT
# ------------------------------------------------------------

print("=" * 80)
print("CELL 10 — OFF-DAYS FEASIBILITY: V1 VS V2")
print("=" * 80)

print("\nINPUT SCOPE")
print("-" * 80)
print("Candidate users          :", audit_510["user"].nunique())
print("Actual windows           :", f"{len(audit_510):,}")
print("Original V1 combinations :", len(quota_510))
print("V2 fixed Off-Day quotas  : NONE")

print("\nA. UNAVOIDABLE OFF-DAYS")
print("-" * 80)
print(forced_summary_510.to_string(index=False))

print("\nB. ALL 15 ORIGINAL V1 CONFIGURATIONS")
print("-" * 80)
print(quota_510.to_string(index=False))

print("\nC. V1 VS V2 — COMBINED COMPARISON")
print("-" * 80)
print(comparison_510.to_string(index=False))

print("\nVALIDATION")
print("-" * 80)
print("Original 15 V1 quotas verified : PASS")
print("995 candidate users            : PASS")
print("472,667 actual windows         : PASS")
print("Natural day boundaries         : PASS")
print("Rotation-support check         : PASS")
print("No actual attack schedule      : PASS")
print("No injection performed         : PASS")
print("Original data modified         : NO")

print("\nSAVED OUTPUTS")
print("-" * 80)

for path in [
    AUDIT_OUTPUT_510,
    QUOTA_OUTPUT_510,
    COMPARISON_OUTPUT_510
]:
    print(path.name)

print("\nIMPORTANT")
print(
    "This is a necessary availability screen, "
    "not proof of complete schedule feasibility."
)


CELL 10 — OFF-DAYS FEASIBILITY: V1 VS V2

INPUT SCOPE
--------------------------------------------------------------------------------
Candidate users          : 995
Actual windows           : 472,667
Original V1 combinations : 15
V2 fixed Off-Day quotas  : NONE

A. UNAVOIDABLE OFF-DAYS
--------------------------------------------------------------------------------
 Duration  Rotation_windows  Zero_forced_windows  Oneplus_forced_windows  Oneplus_forced_pct  Users_with_zero_forced_window
        5             74244                30870                   43374               58.42                            894
       10             85888                17579                   68309               79.53                            688
       20             82914                 9006                   73908               89.14                            302
       30             74133                 6051                   68082               91.84                            170
       40  


### 5.5 Forced-Only Off-Days and Injection Density Feasibility

#### Objective
Evaluate a revised Off-Day rule that preserves a high number of injection-active days without imposing fixed Off-Day quotas.

The new rule will be compared with the original V1 quotas and the unrestricted flexible V2 proposal tested in Cell 10.

#### Proposed Rules

**Rule 1 — Five-Day Duration**
- A five-workday attack window must have zero Off-Days.
- Every eligible workday in the window must have natural activity in at least one target channel.
- Five consecutive eligible workdays do not necessarily mean five consecutive calendar dates.

**Rule 2 — Forced Off-Days Only**
- An Off-Day is permitted only when Email, USB, and HTTP Upload are all naturally inactive on that eligible workday.
- Naturally available target-channel days are not voluntarily skipped.
- All original benign activity must remain unchanged.

**Rule 3 — Minimum Injection-Day Density**
- Off-Days must not exceed half the number of injection-active days.

Definitions:

D = Duration in eligible workdays

O = ForcedOffDays = D - AvailableChannelDays

A = ActiveDays = D - O

Constraint:

O <= A / 2

Equivalent integer limit:

O <= floor(D / 3)

For D = 5, the stricter exception O = 0 applies.

#### Evaluation
1. Reuse the validated window-level Off-Day results from Cell 10.
2. Calculate the proposed number of injection-active days for every window.
3. Apply the five-day exception and minimum injection-density requirement.
4. Measure retained windows and users for all six durations.
5. Compare the proposed rule with V1 fixed quotas and V2 unrestricted flexibility.
6. Identify windows retained or excluded by each approach.

#### Important Limitations
- This is a structural rule-feasibility test.
- ActiveDays indicates the number of days that would require injection under the proposed rule; no actual injection dates are selected.
- Natural channel availability and preliminary rotation support do not guarantee that a complete switching and stealth-compatible schedule exists.
- Final schedule feasibility will be verified in the pre-injection dry run.

#### Expected Outputs
- A window-level audit of the proposed Off-Day and injection-density rule.
- A six-duration comparison against V1 and the previous flexible V2 proposal.
- Evidence for deciding whether to adopt the revised rule.

#### Scope
No existing Cell 10 outputs are overwritten.
No scenarios, dates, primary channels, or injection payloads are assigned.


In [9]:

# ============================================================
# CELL 11 — FORCED-ONLY OFF-DAYS & INJECTION DENSITY
# New proposed rule vs V1 and flexible V2
# Reads Cell 10 outputs; does not overwrite them
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------------------

ROOT_511 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_511 = (
    ROOT_511 / "reports" / "05_Simulation_Rules_Feasibility"
    / "04_Off_Days"
)

INPUT_511 = (
    REPORT_511 / "offday_window_capacity_audit.parquet"
)

WINDOW_OUTPUT_511 = (
    REPORT_511
    / "offday_forced_only_density_window_audit.parquet"
)

SUMMARY_OUTPUT_511 = (
    REPORT_511
    / "offday_forced_only_density_comparison.csv"
)

DURATIONS_511 = [5, 10, 20, 30, 40, 50]

assert INPUT_511.is_file(), f"Missing input: {INPUT_511}"

# ------------------------------------------------------------
# 2. LOAD SAVED CELL 10 RESULTS
# ------------------------------------------------------------

needed_cols_511 = [
    "user",
    "window_workdays",
    "window_start",
    "window_end",
    "supports_2step",
    "available_channel_days",
    "forced_offdays",
    "v1_any_quota_not_excluded",
    "v2_flexible_not_excluded"
]

audit_511 = pd.read_parquet(
    INPUT_511,
    columns=needed_cols_511
)

keys_511 = [
    "user", "window_workdays", "window_start"
]

assert len(audit_511) == 472667
assert audit_511["user"].nunique() == 995
assert not audit_511.duplicated(keys_511).any()
assert audit_511[needed_cols_511].notna().all().all()

assert set(audit_511["window_workdays"]) == set(
    DURATIONS_511
)

# ------------------------------------------------------------
# 3. CALCULATE THE NEW OFF-DAY RULE
# ------------------------------------------------------------

# Forced-only rule:
# Every day with an active target channel is a
# proposed attack-active day.
#
# This does NOT select dates or inject events.

audit_511["proposed_offdays"] = (
    audit_511["forced_offdays"].astype(int)
)

audit_511["proposed_active_days"] = (
    audit_511["window_workdays"]
    - audit_511["proposed_offdays"]
)

# General rule:
# O <= A / 2
#
# Equivalent to:
# O <= floor(D / 3)
#
# Five-day exception:
# O = 0

audit_511["max_allowed_offdays"] = np.where(
    audit_511["window_workdays"].eq(5),
    0,
    audit_511["window_workdays"] // 3
).astype(int)

audit_511["density_rule_pass"] = (
    audit_511["proposed_offdays"]
    <= audit_511["max_allowed_offdays"]
)

audit_511["five_day_rule_pass"] = (
    audit_511["window_workdays"].ne(5)
    | audit_511["proposed_offdays"].eq(0)
)

# Preliminary rotation-support requirement.
rotation_511 = (
    audit_511["supports_2step"].astype(bool)
)

# At least two proposed injection-active days.
min_active_511 = (
    audit_511["proposed_active_days"].ge(2)
)

audit_511["new_rule_not_excluded"] = (
    rotation_511
    & min_active_511
    & audit_511["density_rule_pass"]
    & audit_511["five_day_rule_pass"]
)

# Reuse Cell 10 comparison flags.
audit_511["v1_reference_pass"] = (
    audit_511["v1_any_quota_not_excluded"].astype(bool)
)

audit_511["flexible_v2_reference_pass"] = (
    audit_511["v2_flexible_not_excluded"].astype(bool)
)

# Identify why the new rule excludes a window.
audit_511["new_rule_status"] = np.select(
    [
        ~rotation_511,
        ~min_active_511,
        ~audit_511["density_rule_pass"]
    ],
    [
        "No preliminary rotation support",
        "Fewer than two active channel days",
        "Too many forced Off-Days"
    ],
    default="Not excluded by proposed rule"
)

# ------------------------------------------------------------
# 4. VALIDATE NEW RULE LOGIC
# ------------------------------------------------------------

assert (
    audit_511["proposed_offdays"]
    + audit_511["proposed_active_days"]
    == audit_511["window_workdays"]
).all()

assert (
    audit_511["proposed_active_days"]
    == audit_511["available_channel_days"]
).all()

# Integer equivalence:
# O <= A/2  <=>  2O <= A  <=>  3O <= D
general_density_511 = (
    2 * audit_511["proposed_offdays"]
    <= audit_511["proposed_active_days"]
)

nonfive_511 = audit_511["window_workdays"].ne(5)

assert (
    audit_511.loc[nonfive_511, "density_rule_pass"]
    == general_density_511.loc[nonfive_511]
).all()

# Five-day windows must have zero Off-Days.
five_511 = audit_511["window_workdays"].eq(5)

assert (
    audit_511.loc[five_511, "density_rule_pass"]
    == audit_511.loc[five_511, "proposed_offdays"].eq(0)
).all()

assert (
    audit_511.loc[
        audit_511["new_rule_not_excluded"],
        "proposed_active_days"
    ] >= 2
).all()

# The proposed rule cannot retain a window that
# was excluded by the earlier flexible availability
# screen.
assert (
    ~audit_511["new_rule_not_excluded"]
    | audit_511["flexible_v2_reference_pass"]
).all()

# At this necessary-condition level, all windows
# passing a permitted V1 quota also satisfy the
# proposed maximum Off-Day limit.
assert (
    ~audit_511["v1_reference_pass"]
    | audit_511["new_rule_not_excluded"]
).all()

# ------------------------------------------------------------
# 5. COMPARE THREE APPROACHES BY DURATION
# ------------------------------------------------------------

summary_rows_511 = []

for duration in DURATIONS_511:

    group = audit_511.loc[
        audit_511["window_workdays"].eq(duration)
    ]

    v1_mask = group["v1_reference_pass"]
    flex_mask = group["flexible_v2_reference_pass"]
    new_mask = group["new_rule_not_excluded"]

    v1_windows = int(v1_mask.sum())
    flex_windows = int(flex_mask.sum())
    new_windows = int(new_mask.sum())

    v1_users = int(group.loc[
        v1_mask, "user"
    ].nunique())

    flex_users = int(group.loc[
        flex_mask, "user"
    ].nunique())

    new_users = int(group.loc[
        new_mask, "user"
    ].nunique())

    summary_rows_511.append({
        "Duration": duration,
        "Max_OffDays": (
            0 if duration == 5 else duration // 3
        ),
        "Min_ActiveDays": (
            duration if duration == 5
            else duration - duration // 3
        ),
        "Flexible_windows": flex_windows,
        "V1_windows": v1_windows,
        "New_windows": new_windows,
        "New_retention_pct": round(
            100 * new_windows / flex_windows, 2
        ),
        "Extra_windows_vs_V1": (
            new_windows - v1_windows
        ),
        "Windows_lost_vs_flexible": (
            flex_windows - new_windows
        ),
        "Flexible_users": flex_users,
        "V1_users": v1_users,
        "New_users": new_users,
        "Extra_users_vs_V1": new_users - v1_users,
        "Users_lost_vs_flexible": (
            flex_users - new_users
        )
    })

summary_511 = pd.DataFrame(summary_rows_511)

# ------------------------------------------------------------
# 6. CHECK COMPARISON CONSISTENCY
# ------------------------------------------------------------

assert summary_511.shape == (6, 14)

assert (
    summary_511["V1_windows"]
    <= summary_511["New_windows"]
).all()

assert (
    summary_511["New_windows"]
    <= summary_511["Flexible_windows"]
).all()

assert (
    summary_511["V1_users"]
    <= summary_511["New_users"]
).all()

assert (
    summary_511["New_users"]
    <= summary_511["Flexible_users"]
).all()

# The five-day proposed rule is identical to
# the V1 zero-Off-Day availability condition.
five_summary_511 = summary_511.loc[
    summary_511["Duration"].eq(5)
].iloc[0]

assert (
    five_summary_511["New_windows"]
    == five_summary_511["V1_windows"]
)

assert (
    five_summary_511["New_users"]
    == five_summary_511["V1_users"]
)

# ------------------------------------------------------------
# 7. SAVE NEW RESULTS — DIFFERENT FILENAMES
# ------------------------------------------------------------

# Keep a compact, reusable window-level output.
output_cols_511 = [
    "user",
    "window_workdays",
    "window_start",
    "window_end",
    "supports_2step",
    "available_channel_days",
    "forced_offdays",
    "proposed_offdays",
    "proposed_active_days",
    "max_allowed_offdays",
    "v1_reference_pass",
    "flexible_v2_reference_pass",
    "new_rule_not_excluded",
    "new_rule_status"
]

audit_511[output_cols_511].to_parquet(
    WINDOW_OUTPUT_511,
    index=False
)

summary_511.to_csv(
    SUMMARY_OUTPUT_511,
    index=False
)

# ------------------------------------------------------------
# 8. OUTPUT
# ------------------------------------------------------------

print("=" * 85)
print("CELL 11 — FORCED-ONLY OFF-DAYS & INJECTION DENSITY")
print("=" * 85)

print("\nRULES TESTED")
print("-" * 85)
print("Five-day duration        : Zero Off-Days")
print("Other durations          : Forced Off-Days only")
print("Maximum Off-Days         : floor(Duration / 3)")
print("Minimum density          : ActiveDays >= 2 * OffDays")
print("Optional Off-Days        : NOT ALLOWED")

print("\nA. PROPOSED RULE VS V1 AND FLEXIBLE V2")
print("-" * 85)

display(summary_511)

print("\nB. REASONS FOR EXCLUSION")
print("-" * 85)

display(
    audit_511.groupby(
        ["window_workdays", "new_rule_status"]
    ).size()
    .unstack(fill_value=0)
    .reindex(index=DURATIONS_511, fill_value=0)
)

print("\nC. EXAMPLE WINDOWS — 20 WORKDAYS")
print("-" * 85)

example_cols_511 = [
    "user",
    "window_start",
    "window_end",
    "forced_offdays",
    "proposed_active_days",
    "max_allowed_offdays",
    "new_rule_status"
]

example_511 = pd.concat([
    audit_511.loc[
        audit_511["window_workdays"].eq(20)
        & audit_511["new_rule_not_excluded"],
        example_cols_511
    ].head(3),
    audit_511.loc[
        audit_511["window_workdays"].eq(20)
        & rotation_511
        & ~audit_511["new_rule_not_excluded"],
        example_cols_511
    ].head(3)
], ignore_index=True)

display(example_511)

print("\nVALIDATION")
print("-" * 85)
print("995 candidate users             : PASS")
print("472,667 actual windows          : PASS")
print("Five-day zero-Off-Day condition : PASS")
print("Injection density arithmetic    : PASS")
print("V1 and flexible V2 consistency  : PASS")
print("No injection schedules created  : PASS")
print("Original data modified          : NO")
print("Previous Cell 10 outputs changed: NO")

print("\nSAVED OUTPUTS")
print("-" * 85)
print(WINDOW_OUTPUT_511.name)
print(SUMMARY_OUTPUT_511.name)

print("\nIMPORTANT")
print(
    "These are preliminary availability results. "
    "Switching, stealth and complete scheduling "
    "have not been tested together."
)


CELL 11 — FORCED-ONLY OFF-DAYS & INJECTION DENSITY

RULES TESTED
-------------------------------------------------------------------------------------
Five-day duration        : Zero Off-Days
Other durations          : Forced Off-Days only
Maximum Off-Days         : floor(Duration / 3)
Minimum density          : ActiveDays >= 2 * OffDays
Optional Off-Days        : NOT ALLOWED

A. PROPOSED RULE VS V1 AND FLEXIBLE V2
-------------------------------------------------------------------------------------


,Duration,Max_OffDays,Min_ActiveDays,Flexible_windows,V1_windows,New_windows,New_retention_pct,Extra_windows_vs_V1,Windows_lost_vs_flexible,Flexible_users,V1_users,New_users,Extra_users_vs_V1,Users_lost_vs_flexible
0,5,0,5,74244,30870,30870,41.58,0,43374,995,894,894,0,101
1,10,3,7,85888,51773,63717,74.19,11944,22171,993,881,942,61,51
2,20,6,14,82914,44281,58485,70.54,14204,24429,992,784,853,69,139
3,30,10,20,74133,37923,54672,73.75,16749,19461,988,723,828,105,160
4,40,13,27,64581,32263,46882,72.59,14619,17699,983,674,790,116,193
5,50,16,34,54870,27150,39518,72.02,12368,15352,982,633,774,141,208



B. REASONS FOR EXCLUSION
-------------------------------------------------------------------------------------


new_rule_status,No preliminary rotation support,Not excluded by proposed rule,Too many forced Off-Days
window_workdays,,,
5,25171,30870,43374
10,8556,63717,22171
20,1604,58485,24429
30,469,54672,19461
40,177,46882,17699
50,60,39518,15352



C. EXAMPLE WINDOWS — 20 WORKDAYS
-------------------------------------------------------------------------------------


,user,window_start,window_end,forced_offdays,proposed_active_days,max_allowed_offdays,new_rule_status
0,AAC0610,2011-01-12,2011-02-08,6,14,6,Not excluded by proposed rule
1,AAC0610,2011-01-13,2011-02-09,6,14,6,Not excluded by proposed rule
2,AAC0610,2011-01-17,2011-02-11,6,14,6,Not excluded by proposed rule
3,AAC0610,2011-01-03,2011-01-28,10,10,6,Too many forced Off-Days
4,AAC0610,2011-01-04,2011-01-31,9,11,6,Too many forced Off-Days
5,AAC0610,2011-01-05,2011-02-01,8,12,6,Too many forced Off-Days



VALIDATION
-------------------------------------------------------------------------------------
995 candidate users             : PASS
472,667 actual windows          : PASS
Five-day zero-Off-Day condition : PASS
Injection density arithmetic    : PASS
V1 and flexible V2 consistency  : PASS
No injection schedules created  : PASS
Original data modified          : NO
Previous Cell 10 outputs changed: NO

SAVED OUTPUTS
-------------------------------------------------------------------------------------
offday_forced_only_density_window_audit.parquet
offday_forced_only_density_comparison.csv

IMPORTANT
These are preliminary availability results. Switching, stealth and complete scheduling have not been tested together.



### 5.6 Integrated Structural Compatibility & Exact Switching Feasibility

#### Objective
Verify the joint consistency of the four structural dimensions — Duration, Switching, Same-Day Multichannel, and Off-Days — for every actual candidate window.

Unlike earlier independent feasibility screens, this evaluation checks whether exact Primary-Channel Switching targets can be achieved while obeying the revised forced-only Off-Day rule.

#### Approved Candidate Rules

1. Duration: 5, 10, 20, 30, 40, or 50 eligible workdays.
2. Off-Days: Only eligible workdays without natural activity in Email, USB, or HTTP Upload.
3. Injection density: Off-Days <= ActiveDays / 2.
4. Five-day exception: Zero Off-Days.
5. Switching: Range C (Low 10–25%, Medium 30–45%, High 50–65%).
6. Five-day Switching exception when ActiveDays = 5: Low = 1, Medium = 2, High = 3 switches.
7. Same-Day Multichannel: Optional; zero multichannel injection days are permitted.
8. Primary injection channel: Must be naturally active on its selected day.

#### Exact Switching Evaluation

For each user and actual window:

- Identify the naturally active channels on every eligible workday.
- Treat days without active target channels as forced Off-Days.
- Under the proposed forced-only rule, include every remaining day as an attack-active day.
- Calculate every attainable integer Primary-Channel switch count using dynamic programming.
- Compare attainable counts with the required Low, Medium, and High ranges.
- Apply the five-day exception separately.
- Check the availability of optional two-channel and three-channel same-day injection.

The algorithm explores possible primary-channel choices mathematically; it does not generate or save attack schedules.

#### Cross-Validation

- Reconcile all window keys and boundaries.
- Recalculate channel availability from original daily event counts.
- Match available days and forced Off-Days against Cell 11.
- Match two-channel and three-channel day counts against Cell 9.
- Validate Switching arithmetic using exact integer percentages.
- Independently verify selected short windows through exhaustive enumeration.
- Identify any contradiction between observed rotation support and exact switching calculations.

#### Expected Outputs

1. Window-level integrated structural feasibility audit.
2. Compatibility summary across all six durations.
3. Exact Switching-level coverage and exclusion reasons.
4. Five-day exception diagnostics.
5. Validation report confirming consistency between earlier notebooks and the integrated rules.

#### Scope and Limitations

This verifies structural feasibility under the currently proposed forced-only Off-Day rule.

It does not establish stealth or payload feasibility, select final attackers, impose cross-scenario spacing, or create attack schedules.

Those requirements remain for later notebooks.


In [10]:

# ============================================================
# CELL 12 — INTEGRATED STRUCTURAL COMPATIBILITY
# Exact primary switching via bitset dynamic programming
# ============================================================

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT_512 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_512 = (
    ROOT_512 / "reports" / "05_Simulation_Rules_Feasibility"
)

OFFDAY_PATH_512 = (
    REPORT_512
    / "04_Off_Days"
    / "offday_forced_only_density_window_audit.parquet"
)

MULTI_PATH_512 = (
    REPORT_512
    / "03_Same_Day_Multichannel"
    / "same_day_multichannel_window_audit.parquet"
)

DAILY_PATH_512 = (
    ROOT_512.parent / "Data_Engineering"
    / "Data_engineering" / "data" / "features"
    / "behavior_features_r62_v1.parquet"
)

DURATIONS_512 = [5, 10, 20, 30, 40, 50]
LEVELS_512 = ["Low", "Medium", "High"]

# Percentages as integers to avoid floating-point rounding.
RATES_512 = {
    "Low": (10, 25),
    "Medium": (30, 45),
    "High": (50, 65)
}

FIVE_DAY_EXCEPTION_512 = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

KEYS_512 = [
    "user", "window_workdays", "window_start"
]

for path in [
    OFFDAY_PATH_512,
    MULTI_PATH_512,
    DAILY_PATH_512
]:
    assert path.is_file(), f"Missing input: {path}"

# ------------------------------------------------------------
# 2. LOAD & RECONCILE PREVIOUS WINDOW RESULTS
# ------------------------------------------------------------

off_cols_512 = [
    "user", "window_workdays",
    "window_start", "window_end",
    "supports_2step",
    "available_channel_days",
    "forced_offdays",
    "proposed_active_days",
    "max_allowed_offdays",
    "new_rule_not_excluded"
]

multi_cols_512 = KEYS_512 + [
    "days_2plus",
    "days_exactly_2ch",
    "days_3ch",
    "days_EU",
    "days_EH",
    "days_UH",
    "days_EUH"
]

off_512 = pd.read_parquet(
    OFFDAY_PATH_512, columns=off_cols_512
)

multi_512 = pd.read_parquet(
    MULTI_PATH_512, columns=multi_cols_512
)

for frame in [off_512, multi_512]:
    frame["window_start"] = pd.to_datetime(
        frame["window_start"]
    )

off_512["window_end"] = pd.to_datetime(
    off_512["window_end"]
)

assert len(off_512) == len(multi_512) == 472667
assert off_512["user"].nunique() == 995

assert not off_512.duplicated(KEYS_512).any()
assert not multi_512.duplicated(KEYS_512).any()

windows_512 = off_512.merge(
    multi_512,
    on=KEYS_512,
    how="left",
    validate="one_to_one",
    indicator=True,
    sort=False
)

assert windows_512["_merge"].eq("both").all()

windows_512 = (
    windows_512
    .drop(columns="_merge")
    .sort_values(KEYS_512)
    .reset_index(drop=True)
)

assert windows_512.notna().all().all()
assert set(windows_512["window_workdays"]) == set(
    DURATIONS_512
)

# ------------------------------------------------------------
# 3. LOAD DAILY NATURAL CHANNEL STATES
# ------------------------------------------------------------

COUNT_COLS_512 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

daily_512 = pd.read_parquet(
    DAILY_PATH_512,
    columns=["user", "day"] + COUNT_COLS_512
)

candidate_users_512 = set(windows_512["user"])

daily_512 = daily_512.loc[
    daily_512["user"].isin(candidate_users_512)
].copy()

daily_512["day"] = pd.to_datetime(
    daily_512["day"]
)

daily_512 = daily_512.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert len(daily_512) == 103395
assert daily_512["user"].nunique() == 995
assert not daily_512.duplicated(["user", "day"]).any()
assert daily_512[COUNT_COLS_512].notna().all().all()
assert daily_512[COUNT_COLS_512].ge(0).all().all()

# Bit representation of naturally available channels:
# E = 1, U = 2, H = 4
# EU = 3, EH = 5, UH = 6, EUH = 7

counts_512 = daily_512[
    COUNT_COLS_512
].to_numpy()

daily_512["channel_state"] = (
    (counts_512[:, 0] > 0).astype(np.uint8)
    + 2 * (counts_512[:, 1] > 0).astype(np.uint8)
    + 4 * (counts_512[:, 2] > 0).astype(np.uint8)
)

daily_groups_512 = {
    user: group
    for user, group in daily_512.groupby(
        "user", sort=False
    )
}

# ------------------------------------------------------------
# 4. EXACT SWITCHING DYNAMIC PROGRAMMING
# ------------------------------------------------------------

# Bit s in the resulting uint64 value is 1 if
# EXACTLY s primary-channel switches are attainable.
#
# Example:
# bit 1 + bit 3 = 0b1010 = 10
# means exactly 1 or 3 switches can be achieved.
#
# All naturally available days are included.
# Only zero-channel days are skipped.
#
# The algorithm does not save actual channel sequences.

ONE_512 = np.uint64(1)
ZERO_512 = np.uint64(0)

possible_bits_512 = np.zeros(
    len(windows_512), dtype=np.uint64
)

# Daily metric order for independent reconciliation.
METRIC_COLS_512 = [
    "available_channel_days",
    "days_2plus",
    "days_exactly_2ch",
    "days_3ch",
    "days_EU",
    "days_EH",
    "days_UH",
    "days_EUH"
]

for user, user_windows in windows_512.groupby(
    "user", sort=False
):

    source = daily_groups_512[user]

    dates = source["day"].to_numpy(
        dtype="datetime64[ns]"
    )

    states = source["channel_state"].to_numpy(
        dtype=np.uint8
    )

    # Independently reconstruct natural activity counts.
    metrics = np.column_stack([
        states > 0,
        np.isin(states, [3, 5, 6, 7]),
        np.isin(states, [3, 5, 6]),
        states == 7,
        states == 3,
        states == 5,
        states == 6,
        states == 7
    ]).astype(np.int32)

    prefix = np.vstack([
        np.zeros((1, 8), dtype=np.int32),
        np.cumsum(metrics, axis=0, dtype=np.int32)
    ])

    for duration, group in user_windows.groupby(
        "window_workdays", sort=False
    ):

        duration = int(duration)
        indices = group.index.to_numpy()

        starts = group["window_start"].to_numpy(
            dtype="datetime64[ns]"
        )

        ends = group["window_end"].to_numpy(
            dtype="datetime64[ns]"
        )

        left = np.searchsorted(dates, starts)
        right = left + duration

        assert (left < len(dates)).all()
        assert (right <= len(dates)).all()

        assert np.array_equal(
            dates[left], starts
        )

        assert np.array_equal(
            dates[right - 1], ends
        )

        observed = prefix[right] - prefix[left]

        # Exact reconciliation with Cell 9 and Cell 11.
        for j, col in enumerate(METRIC_COLS_512):
            assert np.array_equal(
                observed[:, j],
                group[col].to_numpy(dtype=int)
            ), f"Mismatch: {user}, D={duration}, {col}"

        assert np.array_equal(
            duration - observed[:, 0],
            group["forced_offdays"].to_numpy(dtype=int)
        )

        n = len(group)

        # State: attainable switch counts when the
        # primary channel ends as E, U, or H.
        e = np.zeros(n, dtype=np.uint64)
        u = np.zeros(n, dtype=np.uint64)
        h = np.zeros(n, dtype=np.uint64)

        for offset in range(duration):

            mask = states[left + offset]
            active = mask != 0

            if not active.any():
                continue

            first = (e | u | h) == ZERO_512

            # Staying on the same channel adds 0 switches.
            # Changing primary channel adds 1 switch.
            #
            # First available day starts at 0 switches.

            new_e = np.where(
                (mask & 1) != 0,
                e
                | ((u | h) << ONE_512)
                | first.astype(np.uint64),
                ZERO_512
            )

            new_u = np.where(
                (mask & 2) != 0,
                u
                | ((e | h) << ONE_512)
                | first.astype(np.uint64),
                ZERO_512
            )

            new_h = np.where(
                (mask & 4) != 0,
                h
                | ((e | u) << ONE_512)
                | first.astype(np.uint64),
                ZERO_512
            )

            # Forced Off-Day: retain previous states.
            e = np.where(active, new_e, e)
            u = np.where(active, new_u, u)
            h = np.where(active, new_h, h)

        possible_bits_512[indices] = e | u | h

# ------------------------------------------------------------
# 5. INDEPENDENT EXHAUSTIVE VALIDATION
# ------------------------------------------------------------

# Verify the DP with an independent brute-force
# implementation on selected five-day windows.
# Short windows keep enumeration inexpensive.

def brute_force_bits_512(channel_states):
    choices = []

    for mask in channel_states:
        if mask == 0:
            continue

        channels = [
            c for c in range(3)
            if int(mask) & (1 << c)
        ]
        choices.append(channels)

    if not choices:
        return 0

    bits = 0

    for sequence in product(*choices):
        switches = sum(
            sequence[i] != sequence[i - 1]
            for i in range(1, len(sequence))
        )
        bits |= 1 << switches

    return bits


five_indices_512 = windows_512.index[
    windows_512["window_workdays"].eq(5)
].to_numpy()

sample_positions_512 = np.linspace(
    0,
    len(five_indices_512) - 1,
    num=24,
    dtype=int
)

sample_indices_512 = np.unique(
    five_indices_512[sample_positions_512]
)

for idx in sample_indices_512:

    row = windows_512.iloc[int(idx)]

    source = daily_groups_512[row["user"]]

    dates = source["day"].to_numpy(
        dtype="datetime64[ns]"
    )

    states = source["channel_state"].to_numpy(
        dtype=np.uint8
    )

    start = np.searchsorted(
        dates, np.datetime64(row["window_start"])
    )

    day_states = states[start:start + 5]

    assert len(day_states) == 5

    expected_bits = brute_force_bits_512(
        day_states
    )

    assert expected_bits == int(
        possible_bits_512[int(idx)]
    ), f"DP mismatch at window index {idx}"

# ------------------------------------------------------------
# 6. RECONCILE ROTATION SUPPORT
# ------------------------------------------------------------

# A 2-step natural rotation should correspond to
# the possibility of at least one primary switch.

HAS_SWITCH_MASK_512 = np.uint64(
    0xFFFFFFFFFFFFFFFE
)

dp_supports_switch_512 = (
    possible_bits_512 & HAS_SWITCH_MASK_512
) != ZERO_512

previous_support_512 = (
    windows_512["supports_2step"]
    .astype(bool)
    .to_numpy()
)

mismatches_512 = (
    dp_supports_switch_512 != previous_support_512
)

if mismatches_512.any():
    print("ROTATION SUPPORT MISMATCH — SAMPLE")
    display(
        windows_512.loc[
            mismatches_512,
            KEYS_512 + ["supports_2step"]
        ].head(10)
    )
    raise AssertionError(
        "Notebook 02/Cell 9 rotation support "
        "does not match exact daily-channel DP."
    )

# ------------------------------------------------------------
# 7. EXACT INTEGER SWITCHING REQUIREMENTS
# ------------------------------------------------------------

def range_bitmask_512(active_days, level):
    """Allowed exact switch counts for Range C."""

    transitions = active_days - 1

    if transitions < 1:
        return 0

    low_pct, high_pct = RATES_512[level]

    # Exact integer ceiling and floor:
    min_s = (transitions * low_pct + 99) // 100
    max_s = (transitions * high_pct) // 100

    if min_s > max_s:
        return 0

    return sum(
        1 << s
        for s in range(max(1, min_s), max_s + 1)
    )


# Precalculate masks so each window is compared
# without repeated mathematical enumeration.

allowed_512 = {}

for duration in DURATIONS_512:

    allowed_512[duration] = {}

    for level in LEVELS_512:

        table = np.zeros(
            duration + 1, dtype=np.uint64
        )

        for active_days in range(2, duration + 1):

            if duration == 5:
                # Explicit V1 exception.
                if active_days == 5:
                    switches = FIVE_DAY_EXCEPTION_512[level]
                    table[active_days] = np.uint64(
                        1 << switches
                    )
            else:
                table[active_days] = np.uint64(
                    range_bitmask_512(
                        active_days, level
                    )
                )

        allowed_512[duration][level] = table

durations_array_512 = (
    windows_512["window_workdays"].to_numpy(dtype=int)
)

active_array_512 = (
    windows_512["proposed_active_days"]
    .to_numpy(dtype=int)
)

assert (active_array_512 >= 0).all()
assert (active_array_512 <= durations_array_512).all()

for level in LEVELS_512:

    level_masks = np.zeros(
        len(windows_512), dtype=np.uint64
    )

    for duration in DURATIONS_512:

        mask = durations_array_512 == duration

        level_masks[mask] = (
            allowed_512[duration][level][
                active_array_512[mask]
            ]
        )

    windows_512[f"exact_{level.lower()}"] = (
        (possible_bits_512 & level_masks)
        != ZERO_512
    )

# ------------------------------------------------------------
# 8. INTEGRATE ALL FOUR STRUCTURAL DIMENSIONS
# ------------------------------------------------------------

# Off-Day compliance is the result of Cell 11,
# independently reconciled against daily data above.

offday_pass_512 = (
    windows_512["new_rule_not_excluded"]
    .astype(bool)
)

# Verify the stored Off-Day decision independently.
forced_512 = (
    windows_512["forced_offdays"].to_numpy(dtype=int)
)

max_off_512 = np.where(
    durations_array_512 == 5,
    0,
    durations_array_512 // 3
)

recomputed_offday_pass_512 = (
    previous_support_512
    & (forced_512 <= max_off_512)
    & (active_array_512 >= 2)
)

assert np.array_equal(
    offday_pass_512.to_numpy(),
    recomputed_offday_pass_512
)

windows_512["exact_any_level"] = (
    windows_512[
        ["exact_low", "exact_medium", "exact_high"]
    ].any(axis=1)
)

windows_512["structural_pass"] = (
    offday_pass_512
    & windows_512["exact_any_level"]
)

# Same-day multichannel injection is optional:
# it cannot invalidate an otherwise feasible
# single-primary-channel path.

windows_512["optional_2plus_supported"] = (
    windows_512["days_2plus"] > 0
)

windows_512["optional_3ch_supported"] = (
    windows_512["days_3ch"] > 0
)

windows_512["compatible_level_count"] = (
    windows_512[
        ["exact_low", "exact_medium", "exact_high"]
    ].sum(axis=1)
)

# Keep the complete attainable switching set
# in machine-readable bitmask form.

windows_512["exact_switch_count_bitmask"] = (
    possible_bits_512
)

# Diagnostics: how many exact switch counts are attainable?
bits_as_python_512 = [
    int(x) for x in possible_bits_512
]

windows_512["attainable_count_options"] = [
    b.bit_count() for b in bits_as_python_512
]

windows_512["min_attainable_switches"] = [
    (b & -b).bit_length() - 1 if b else -1
    for b in bits_as_python_512
]

windows_512["max_attainable_switches"] = [
    b.bit_length() - 1 if b else -1
    for b in bits_as_python_512
]

# Record one primary reason per window.
windows_512["integrated_status"] = np.select(
    [
        ~previous_support_512,
        previous_support_512
        & ~offday_pass_512.to_numpy(),
        offday_pass_512.to_numpy()
        & ~windows_512["exact_any_level"].to_numpy()
    ],
    [
        "No natural rotation support",
        "Fails forced-only Off-Day rule",
        "No exact Switching level attainable"
    ],
    default="Structurally compatible"
)

# ------------------------------------------------------------
# 9. STRONG LOGICAL INVARIANTS
# ------------------------------------------------------------

assert len(windows_512) == 472667
assert not windows_512.duplicated(KEYS_512).any()

assert (
    windows_512["forced_offdays"]
    + windows_512["proposed_active_days"]
    == windows_512["window_workdays"]
).all()

assert (
    windows_512["days_3ch"]
    <= windows_512["days_2plus"]
).all()

assert (
    windows_512["days_2plus"]
    <= windows_512["proposed_active_days"]
).all()

assert (
    windows_512["compatible_level_count"]
    .between(0, 3)
).all()

assert (
    ~windows_512["structural_pass"]
    | windows_512["exact_any_level"]
).all()

assert (
    ~windows_512["structural_pass"]
    | offday_pass_512
).all()

# Every attainable switch count must be no larger
# than the number of transitions between active days.
max_possible_512 = np.maximum(
    active_array_512 - 1, 0
)

has_state_512 = possible_bits_512 != ZERO_512

assert (
    windows_512.loc[
        has_state_512,
        "max_attainable_switches"
    ].to_numpy()
    <= max_possible_512[has_state_512]
).all()

assert (
    windows_512.loc[
        windows_512["structural_pass"],
        "min_attainable_switches"
    ] >= 0
).all()

# Five-day pass requires exactly five active days.
five_512 = windows_512["window_workdays"].eq(5)

assert (
    windows_512.loc[
        five_512 & windows_512["structural_pass"],
        "proposed_active_days"
    ] == 5
).all()

# ------------------------------------------------------------
# 10. SUMMARY BY DURATION
# ------------------------------------------------------------

summary_rows_512 = []
level_rows_512 = []
multi_rows_512 = []

for duration in DURATIONS_512:

    group = windows_512.loc[
        windows_512["window_workdays"].eq(duration)
    ]

    off_ok = group["new_rule_not_excluded"]
    compatible = group["structural_pass"]

    valid = group.loc[compatible]

    summary_rows_512.append({
        "Duration": duration,
        "All_windows": len(group),
        "Rotation_windows": int(
            group["supports_2step"].sum()
        ),
        "OffDay_pass_windows": int(off_ok.sum()),
        "Exact_compatible_windows": int(
            compatible.sum()
        ),
        "OffDay_pass_but_no_level": int(
            (
                off_ok
                & ~group["exact_any_level"]
            ).sum()
        ),
        "Exact_compatible_pct_of_rotation": round(
            100 * compatible.sum()
            / group["supports_2step"].sum(),
            2
        ),
        "Compatible_users": int(
            valid["user"].nunique()
        )
    })

    for level in LEVELS_512:

        passing_level = (
            compatible
            & group[f"exact_{level.lower()}"]
        )

        level_rows_512.append({
            "Duration": duration,
            "Level": level,
            "Compatible_windows": int(
                passing_level.sum()
            ),
            "Compatible_users": int(
                group.loc[
                    passing_level, "user"
                ].nunique()
            )
        })

    multi_rows_512.append({
        "Duration": duration,
        "Compatible_windows": len(valid),
        "With_2plus_channel_option": int(
            valid["optional_2plus_supported"].sum()
        ),
        "With_3channel_option": int(
            valid["optional_3ch_supported"].sum()
        ),
        "Users_with_2plus_option": int(
            valid.loc[
                valid["optional_2plus_supported"],
                "user"
            ].nunique()
        ),
        "Users_with_3channel_option": int(
            valid.loc[
                valid["optional_3ch_supported"],
                "user"
            ].nunique()
        )
    })

summary_512 = pd.DataFrame(summary_rows_512)
levels_512 = pd.DataFrame(level_rows_512)
multichannel_512 = pd.DataFrame(multi_rows_512)


# ------------------------------------------------------------
# 11. FIVE-DAY EXCEPTION DIAGNOSTIC — CORRECTED
# ------------------------------------------------------------

five_group_512 = windows_512.loc[
    five_512 & offday_pass_512
]

five_rows_512 = []

for level in LEVELS_512:

    general_mask = np.uint64(
        range_bitmask_512(5, level)
    )

    general_pass = (
        five_group_512["exact_switch_count_bitmask"]
        .to_numpy(dtype=np.uint64)
        & general_mask
    ) != ZERO_512

    exception_pass = five_group_512[
        f"exact_{level.lower()}"
    ]

    five_rows_512.append({
        "Level": level,
        "V1_exception_switches": (
            FIVE_DAY_EXCEPTION_512[level]
        ),
        "General_C_windows": int(
            general_pass.sum()
        ),
        "Exception_windows": int(
            exception_pass.sum()
        ),
        "Exception_users": int(
            five_group_512.loc[
                exception_pass, "user"
            ].nunique()
        )
    })

five_diagnostic_512 = pd.DataFrame(five_rows_512)


# ------------------------------------------------------------
# 12. GLOBAL VALIDATION AND USER COVERAGE
# ------------------------------------------------------------

assert summary_512.shape == (6, 8)
assert levels_512.shape == (18, 4)
assert multichannel_512.shape == (6, 6)
assert five_diagnostic_512.shape == (3, 5)

assert (
    summary_512["Exact_compatible_windows"]
    <= summary_512["OffDay_pass_windows"]
).all()

assert (
    summary_512["OffDay_pass_but_no_level"]
    == (
        summary_512["OffDay_pass_windows"]
        - summary_512["Exact_compatible_windows"]
    )
).all()

assert (
    multichannel_512["With_3channel_option"]
    <= multichannel_512["With_2plus_channel_option"]
).all()

assert (
    windows_512["integrated_status"]
    .value_counts()
    .sum()
    == 472667
)

global_users_512 = windows_512.loc[
    windows_512["structural_pass"],
    "user"
].nunique()

# ------------------------------------------------------------
# 13. SAVE ONLY NEW OUTPUTS
# ------------------------------------------------------------

WINDOW_OUTPUT_512 = (
    REPORT_512
    / "integrated_structural_window_audit.parquet"
)

INTEGRATED_DIR_512 = REPORT_512 / "05_Integrated_Compatibility"
INTEGRATED_DIR_512.mkdir(parents=True, exist_ok=True)

SUMMARY_OUTPUT_512 = (
    INTEGRATED_DIR_512
    / "integrated_structural_summary.csv"
)

LEVEL_OUTPUT_512 = (
    INTEGRATED_DIR_512
    / "integrated_switching_level_coverage.csv"
)

MULTI_OUTPUT_512 = (
    INTEGRATED_DIR_512
    / "integrated_multichannel_options.csv"
)

FIVE_OUTPUT_512 = (
    INTEGRATED_DIR_512
    / "integrated_five_day_exception.csv"
)

# All assertions have passed before files are written.
windows_512.to_parquet(
    WINDOW_OUTPUT_512, index=False
)

summary_512.to_csv(
    SUMMARY_OUTPUT_512, index=False
)

levels_512.to_csv(
    LEVEL_OUTPUT_512, index=False
)

multichannel_512.to_csv(
    MULTI_OUTPUT_512, index=False
)

five_diagnostic_512.to_csv(
    FIVE_OUTPUT_512, index=False
)

# ------------------------------------------------------------
# 14. OUTPUT
# ------------------------------------------------------------

print("=" * 90)
print("CELL 12 — INTEGRATED STRUCTURAL COMPATIBILITY")
print("=" * 90)

print("\nINPUT SCOPE")
print("-" * 90)
print("Candidate users        :", len(candidate_users_512))
print("Actual windows         :", f"{len(windows_512):,}")
print("Durations              :", DURATIONS_512)
print("Primary switching rule : Range C + five-day exception")
print("Off-Days               : Forced-only + minimum density")
print("Same-day multichannel  : Optional, zero allowed")

print("\nA. OVERALL STRUCTURAL COMPATIBILITY")
print("-" * 90)
display(summary_512)

print("\nB. EXACT SWITCHING LEVEL COVERAGE")
print("-" * 90)
display(levels_512)

print("\nC. SAME-DAY MULTICHANNEL OPTIONS")
print("-" * 90)
display(multichannel_512)

print("\nD. FIVE-DAY EXCEPTION")
print("-" * 90)
display(five_diagnostic_512)

print("\nE. REASONS FOR STRUCTURAL EXCLUSION")
print("-" * 90)
display(
    windows_512.groupby(
        ["window_workdays", "integrated_status"]
    ).size()
    .unstack(fill_value=0)
    .reindex(index=DURATIONS_512, fill_value=0)
)

print("\nF. GLOBAL USER COVERAGE")
print("-" * 90)
print("Candidate users:", len(candidate_users_512))
print("Users with >=1 structurally compatible window:",
      global_users_512)

print("\nVALIDATION")
print("-" * 90)
print("Window keys and boundaries        : PASS")
print("Daily natural channel counts      : PASS")
print("Cell 9 multichannel reconciliation: PASS")
print("Cell 11 Off-Day reconciliation    : PASS")
print("Notebook 02 rotation consistency  : PASS")
print("Exact integer switching arithmetic: PASS")
print("Independent brute-force DP tests  : PASS")
print("Five-day exception                : PASS")
print("All structural invariants         : PASS")
print("Original inputs modified          : NO")
print("Attack scenarios generated        : NO")
print("Attack payloads injected          : NO")

print("\nSAVED OUTPUTS")
print("-" * 90)

for path in [
    WINDOW_OUTPUT_512,
    SUMMARY_OUTPUT_512,
    LEVEL_OUTPUT_512,
    MULTI_OUTPUT_512,
    FIVE_OUTPUT_512
]:
    print(path.name)

print("\nIMPORTANT")
print(
    "Exact primary-channel switching has been tested "
    "under the forced-only Off-Day rule. "
    "Stealth, payload, multi-scenario assignment, and "
    "final injection feasibility remain untested."
)


CELL 12 — INTEGRATED STRUCTURAL COMPATIBILITY

INPUT SCOPE
------------------------------------------------------------------------------------------
Candidate users        : 995
Actual windows         : 472,667
Durations              : [5, 10, 20, 30, 40, 50]
Primary switching rule : Range C + five-day exception
Off-Days               : Forced-only + minimum density
Same-day multichannel  : Optional, zero allowed

A. OVERALL STRUCTURAL COMPATIBILITY
------------------------------------------------------------------------------------------


,Duration,All_windows,Rotation_windows,OffDay_pass_windows,Exact_compatible_windows,OffDay_pass_but_no_level,Exact_compatible_pct_of_rotation,Compatible_users
0,5,99415,74244,30870,30803,67,41.49,893
1,10,94444,85888,63717,60011,3706,69.87,931
2,20,84518,82914,58485,57812,673,69.73,853
3,30,74602,74133,54672,53966,706,72.80,827
4,40,64758,64581,46882,46580,302,72.13,789
5,50,54930,54870,39518,39314,204,71.65,773



B. EXACT SWITCHING LEVEL COVERAGE
------------------------------------------------------------------------------------------


,Duration,Level,Compatible_windows,Compatible_users
0,5,Low,21110,854
1,5,Medium,25865,879
2,5,High,18898,826
3,10,Low,40515,875
4,10,Medium,44038,904
5,10,High,41727,908
6,20,Low,44151,798
7,20,Medium,49724,829
8,20,High,39429,794
9,30,Low,41601,774



C. SAME-DAY MULTICHANNEL OPTIONS
------------------------------------------------------------------------------------------


,Duration,Compatible_windows,With_2plus_channel_option,With_3channel_option,Users_with_2plus_option,Users_with_3channel_option
0,5,30803,28756,3689,871,206
1,10,60011,57334,7885,920,226
2,20,57812,57087,10369,850,220
3,30,53966,53712,11202,826,217
4,40,46580,46486,10825,789,213
5,50,39314,39290,9755,773,211



D. FIVE-DAY EXCEPTION
------------------------------------------------------------------------------------------


,Level,V1_exception_switches,General_C_windows,Exception_windows,Exception_users
0,Low,1,21110,21110,854
1,Medium,2,0,25865,879
2,High,3,25865,18898,826



E. REASONS FOR STRUCTURAL EXCLUSION
------------------------------------------------------------------------------------------


integrated_status,Fails forced-only Off-Day rule,No exact Switching level attainable,No natural rotation support,Structurally compatible
window_workdays,,,,
5,43374,67,25171,30803
10,22171,3706,8556,60011
20,24429,673,1604,57812
30,19461,706,469,53966
40,17699,302,177,46580
50,15352,204,60,39314



F. GLOBAL USER COVERAGE
------------------------------------------------------------------------------------------
Candidate users: 995
Users with >=1 structurally compatible window: 936

VALIDATION
------------------------------------------------------------------------------------------
Window keys and boundaries        : PASS
Daily natural channel counts      : PASS
Cell 9 multichannel reconciliation: PASS
Cell 11 Off-Day reconciliation    : PASS
Notebook 02 rotation consistency  : PASS
Exact integer switching arithmetic: PASS
Independent brute-force DP tests  : PASS
Five-day exception                : PASS
All structural invariants         : PASS
Original inputs modified          : NO
Attack scenarios generated        : NO
Attack payloads injected          : NO

SAVED OUTPUTS
------------------------------------------------------------------------------------------
integrated_structural_window_audit.parquet
integrated_structural_summary.csv
integrated_switching_level_coverage.csv



### 5.7 Final Source-of-Truth & Cross-File Validation

#### Objective
Independently validate Notebook 05's structural simulation results against the original processed User-Day dataset before documentation and handoff.

#### Primary Sources
- user_day_r62_v1.parquet — processed User-Day source of truth.
- behavior_features_r62_v1.parquet — derived eligible Future features.
- integrated_structural_window_audit.parquet — final Notebook 05 structural output.

#### Validation Stages

1. Verify all 18 saved Notebook 05 outputs.
2. Validate primary User-Day keys, eligible workdays, and target-channel event counts.
3. Reconcile User-Day and Behavior Features, including the previously documented 275 missing feature rows.
4. Confirm that the discrepancy does not affect the 995 simulation candidates.
5. Reconstruct all 472,667 window boundaries using eligible User-Day records.
6. Independently recalculate channel availability, Off-Days, and same-day multichannel counts.
7. Recalculate exact primary-channel switch counts from User-Day using dynamic programming.
8. Verify selected five-day windows independently through exhaustive enumeration.
9. Reapply Range C, the five-day exception, and the forced-only Off-Day rules.
10. Reconcile window-level decisions with the saved summary CSV files.
11. Record SHA-256 fingerprints and generate a final validation report.

#### Mandatory Failure Policy
Any unexpected key mismatch, source-value mismatch, incorrect window boundary, invalid switching calculation, or inconsistent summary stops validation.

The previously known 275 missing feature rows are accepted only if their exact documented user-level distribution is reproduced and none affects the 995 candidate users.

#### Expected Output
final_validation_report.md

This report is written only after all implemented mandatory checks pass.

#### Limitations
User-Day is the processed daily source, not a reconstruction from raw event logs.

This validation does not cover Stealth, injected payloads, final attack scheduling, or model performance.

No existing datasets or Notebook 05 result files are modified.


In [11]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import hashlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# 1. Paths
ROOT = Path(r'C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation')
REPORT = ROOT / 'reports' / '05_Simulation_Rules_Feasibility'
DATA = ROOT.parent / 'Data_Engineering' / 'Data_engineering' / 'data'
RAW_PATH = DATA / 'processed' / 'user_day_r62_v1.parquet'
FEAT_PATH = DATA / 'features' / 'behavior_features_r62_v1.parquet'
MAIN_PATH = REPORT / 'integrated_structural_window_audit.parquet'
KEY = ['user','day']
WKEY = ['user','window_workdays','window_start']
COUNTS = ['external_email_count','usb_copy_count','www_upload_count']
DURATIONS = [5,10,20,30,40,50]
RANGES = {'Low':(10,25),'Medium':(30,45),'High':(50,65)}
EXCEPTION = {'Low':1,'Medium':2,'High':3}
EXPECTED = {
 'natural_window_switching_audit.parquet':472667,
 'natural_window_switching_enriched.parquet':472667,
 'switching_rule_integer_feasibility.parquet':1788,
 'switching_full_density_capacity_screen.parquet':72,
 'switching_range_comparison.csv':4,
 'same_day_multichannel_window_audit.parquet':472667,
 'same_day_multichannel_rule_comparison.csv':30,
 'offday_window_capacity_audit.parquet':472667,
 'offday_v1_15_quota_comparison.csv':15,
 'offday_v1_vs_v2_summary.csv':6,
 'offday_forced_only_density_window_audit.parquet':472667,
 'offday_forced_only_density_comparison.csv':6,
 'integrated_structural_window_audit.parquet':472667,
 'integrated_structural_summary.csv':6,
 'integrated_switching_level_coverage.csv':18,
 'integrated_multichannel_options.csv':6,
 'integrated_five_day_exception.csv':3,
}
# Final organized location of every Notebook 05 result.
# This mapping is used consistently by validation and inventory.
def output_path_513(name):
    if name == 'integrated_structural_window_audit.parquet':
        return REPORT / name
    if name.startswith(('natural_window_switching_', 'switching_')):
        folder = '02_Switching'
    elif name.startswith('same_day_multichannel_'):
        folder = '03_Same_Day_Multichannel'
    elif name.startswith('offday_'):
        folder = '04_Off_Days'
    elif name.startswith('integrated_'):
        folder = '05_Integrated_Compatibility'
    else:
        raise ValueError(f'Unknown Notebook 05 output: {name}')
    return REPORT / folder / name

checks=[]
def require(ok, msg, log=True):
    if not bool(ok): raise AssertionError('FAIL — '+msg)
    if log: checks.append(msg)
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
def read(name, cols=None):
    p=output_path_513(name)
    return pd.read_parquet(p,columns=cols) if p.suffix=='.parquet' else pd.read_csv(p)
def normalized_keys(frame, cols):
    k=frame[cols].copy()
    datefield='day' if cols==KEY else 'window_start'
    k[datefield]=pd.to_datetime(k[datefield],errors='raise')
    require(k.notna().all().all(),f'Nonmissing keys: {cols}')
    require(not k.duplicated(cols).any(),f'Unique keys: {cols}')
    return pd.MultiIndex.from_frame(k.sort_values(cols).reset_index(drop=True))

# 2. Inventory and file metadata
for p in [RAW_PATH, FEAT_PATH, MAIN_PATH]: require(p.is_file(),f'Source exists: {p.name}')
manifest=[]
for name,n in EXPECTED.items():
    p=output_path_513(name)
    require(p.is_file() and p.stat().st_size>0,f'Output available: {name}')
    if p.suffix=='.parquet':
        meta=pq.ParquetFile(p);rows=meta.metadata.num_rows;fields=meta.schema_arrow.names
    else:
        obj=pd.read_csv(p);rows=len(obj);fields=list(obj.columns)
    require(rows==n and len(fields)==len(set(fields)),f'Output schema/count: {name}')
    manifest.append({'file':name,'rows':rows,'columns':len(fields),'sha256':sha(p)})
extra=[p.name for p in REPORT.iterdir() if p.is_file() and p.suffix.lower() in {'.parquet','.csv'} and p.name not in {'integrated_structural_window_audit.parquet', 'outputs_manifest.csv'}]
require(not extra,f'No unclassified outputs: {extra}')

# 3. Source of truth — User-Day independently compared with Features
raw=pd.read_parquet(RAW_PATH,columns=KEY+['is_weekend','logon_count']+COUNTS)
feat=pd.read_parquet(FEAT_PATH,columns=KEY+['is_weekend','logon_count']+COUNTS)
for frame in [raw,feat]: frame['day']=pd.to_datetime(frame['day'],errors='raise')
require(len(raw)==1394010 and len(feat)==387162,'Expected source and feature populations')
raw_index=normalized_keys(raw,KEY)
feat_index=normalized_keys(feat,KEY)
require(raw['is_weekend'].isin([0,1]).all(), 'Valid primary weekend flags')
require(raw['logon_count'].notna().all(), 'Primary logon counts present')
require(((raw.day.dt.weekday>=5).to_numpy()==raw.is_weekend.to_numpy(dtype=bool)).all(),'Primary weekend flags match calendar')
start,end=pd.Timestamp('2011-01-01'),pd.Timestamp('2011-06-01')
raw_future=raw.loc[raw.day.between(start,end) & raw.is_weekend.eq(0) & raw.logon_count.gt(0)].copy()
require(len(raw_future)==387437,'Eligible Future User-Day rows = 387,437')
require(feat.day.between(start,end).all() & feat.is_weekend.eq(0).all() & feat.logon_count.gt(0).all(),'Feature days all eligible Future workdays')
future_idx=normalized_keys(raw_future,KEY)
missing=future_idx.difference(feat_index)
extra_feat=feat_index.difference(future_idx)
require(len(extra_feat)==0,'No feature keys outside primary eligible Future')
known_missing={'CDE1846':105,'MBG3183':105,'CMP2946':65}
missing_by_user=pd.Series(missing.get_level_values('user')).value_counts().to_dict()
print('SOURCE DISCREPANCY:',len(missing),missing_by_user)
require(len(missing)==275 and missing_by_user==known_missing,'Exactly 275 previously documented missing feature rows')
linked=feat.merge(raw_future,on=KEY,how='left',validate='one_to_one',indicator=True,suffixes=('_feature','_primary'))
require(linked['_merge'].eq('both').all(),'Every feature day found in User-Day')
for col in ['is_weekend','logon_count']+COUNTS:
    lhs=linked[f'{col}_feature'];rhs=linked[f'{col}_primary']
    require(lhs.notna().all() and rhs.notna().all() and np.array_equal(lhs.to_numpy(),rhs.to_numpy()),f'Exact primary-feature equality: {col}')

# 4. Candidate scope
main=read('integrated_structural_window_audit.parquet')
main['window_start']=pd.to_datetime(main['window_start'])
main['window_end']=pd.to_datetime(main['window_end'])
require(len(main)==472667 and main.user.nunique()==995,'Integrated scope: 995 users and 472,667 windows')
main_idx=normalized_keys(main,WKEY)
users=set(main.user)
cand_raw=raw_future.loc[raw_future.user.isin(users)].sort_values(KEY).reset_index(drop=True)
cand_feat=feat.loc[feat.user.isin(users)].sort_values(KEY).reset_index(drop=True)
require(len(cand_raw)==len(cand_feat)==103395,'Candidate primary/feature scope: 103,395 days')
require(normalized_keys(cand_raw,KEY).equals(normalized_keys(cand_feat,KEY)),'No source-to-feature key gaps for 995 candidates')

# 5. Window identity across outputs
other_window_files=[
 'natural_window_switching_audit.parquet','natural_window_switching_enriched.parquet',
 'offday_window_capacity_audit.parquet','offday_forced_only_density_window_audit.parquet'
]
for name in other_window_files:
    index=normalized_keys(read(name,WKEY),WKEY)
    require(index.equals(main_idx),f'Exact window population: {name}')

# 6. Independent daily reconstruction and exact switching DP
from collections import defaultdict
raw_by_user={u:g for u,g in cand_raw.groupby('user',sort=False)}
main=main.sort_values(WKEY).reset_index(drop=True)
computed=np.zeros(len(main),dtype=np.uint64)
metric_cols=['available_channel_days','days_2plus','days_exactly_2ch','days_3ch','days_EU','days_EH','days_UH','days_EUH']
validated_windows=0
for user,sub in main.groupby('user',sort=False):
    src=raw_by_user[user]
    dates=src.day.to_numpy(dtype='datetime64[ns]')
    c=src[COUNTS].to_numpy()
    states=((c[:,0]>0).astype(np.uint8)+2*(c[:,1]>0).astype(np.uint8)+4*(c[:,2]>0).astype(np.uint8))
    metrics=np.column_stack([states>0,np.isin(states,[3,5,6,7]),np.isin(states,[3,5,6]),states==7,states==3,states==5,states==6,states==7]).astype(np.int32)
    prefix=np.vstack([np.zeros((1,8),dtype=np.int32),np.cumsum(metrics,axis=0,dtype=np.int32)])
    for duration,group in sub.groupby('window_workdays',sort=False):
        duration=int(duration)
        ids=group.index.to_numpy()
        starts=group.window_start.to_numpy(dtype='datetime64[ns]')
        ends=group.window_end.to_numpy(dtype='datetime64[ns]')
        left=np.searchsorted(dates,starts);right=left+duration
        require((left<len(dates)).all() and (right<=len(dates)).all(),f'Window source indices {user}/{duration}', log=False)
        require(np.array_equal(dates[left],starts) and np.array_equal(dates[right-1],ends),f'Window boundaries from User-Day {user}/{duration}', log=False)
        actual=prefix[right]-prefix[left]
        for i,col in enumerate(metric_cols):
            require(np.array_equal(actual[:,i],group[col].to_numpy(dtype=int)),f'Primary channel counts {user}/{duration}/{col}', log=False)
        require(np.array_equal(duration-actual[:,0],group.forced_offdays.to_numpy(dtype=int)),'Primary forced Off-Day count '+str(user)+'/'+str(duration), log=False)
        n=len(group);e=np.zeros(n,dtype=np.uint64);u=np.zeros(n,dtype=np.uint64);h=np.zeros(n,dtype=np.uint64)
        for offset in range(duration):
            m=states[left+offset];active=m!=0
            first=(e|u|h)==0
            ne=np.where((m&1)!=0,e|((u|h)<<np.uint64(1))|first.astype(np.uint64),np.uint64(0))
            nu=np.where((m&2)!=0,u|((e|h)<<np.uint64(1))|first.astype(np.uint64),np.uint64(0))
            nh=np.where((m&4)!=0,h|((e|u)<<np.uint64(1))|first.astype(np.uint64),np.uint64(0))
            e=np.where(active,ne,e);u=np.where(active,nu,u);h=np.where(active,nh,h)
        computed[ids]=e|u|h
        validated_windows += len(group)

require(validated_windows==472667,'All 472,667 window boundaries and daily counts independently reconciled')
# Independent brute-force verification on deterministic five-day samples.
five_indices=main.index[main.window_workdays.eq(5)].to_numpy()
verify_indices=np.unique(five_indices[np.linspace(0,len(five_indices)-1,30,dtype=int)])
for idx in verify_indices:
    row=main.iloc[int(idx)]
    source=raw_by_user[row.user]
    dates=source.day.to_numpy(dtype='datetime64[ns]')
    c=source[COUNTS].to_numpy()
    states=((c[:,0]>0).astype(np.uint8)+2*(c[:,1]>0).astype(np.uint8)+4*(c[:,2]>0).astype(np.uint8))
    left=np.searchsorted(dates,np.datetime64(row.window_start))
    active_states=states[left:left+5]
    choices=[[j for j in range(3) if (int(v)&(1<<j))] for v in active_states if v]
    brute=0
    if choices:
        for seq in product(*choices):
            switches=sum(seq[k]!=seq[k-1] for k in range(1,len(seq)))
            brute|=1<<switches
    require(brute==int(computed[int(idx)]),f'Independent brute-force five-day window {idx}',log=False)
require(len(verify_indices)==30,'30 independent brute-force window checks')

stored=main.exact_switch_count_bitmask.to_numpy(dtype=np.uint64)
require(np.array_equal(computed,stored),'Exact DP bitmasks independently rebuilt from primary User-Day (ALL windows)')
require(np.array_equal(((computed & np.uint64(0xFFFFFFFFFFFFFFFE))!=0),main.supports_2step.to_numpy(dtype=bool)),'DP rotation matches saved support flags')
intbits=[int(v) for v in computed]
require(np.array_equal(np.array([v.bit_count() for v in intbits]),main.attainable_count_options.to_numpy(dtype=int)),'Bitmask option counts')
require(np.array_equal(np.array([(v & -v).bit_length()-1 if v else -1 for v in intbits]),main.min_attainable_switches.to_numpy(dtype=int)),'Bitmask minimum switch counts')
require(np.array_equal(np.array([v.bit_length()-1 for v in intbits]),main.max_attainable_switches.to_numpy(dtype=int)),'Bitmask maximum switch counts')

# 7. Recalculate rules and statuses
D=main.window_workdays.to_numpy(dtype=int);A=main.proposed_active_days.to_numpy(dtype=int);O=main.forced_offdays.to_numpy(dtype=int)
require(np.array_equal(D,A+O) and np.array_equal(A,main.available_channel_days.to_numpy(dtype=int)),'Window duration reconciles against forced Off-Days')
limit=np.where(D==5,0,D//3)
require(np.array_equal(limit,main.max_allowed_offdays.to_numpy(dtype=int)),'Approved Off-Day limits')
rotation=main.supports_2step.to_numpy(dtype=bool)
off_ok=rotation & (A>=2) & (O<=limit)
require(np.array_equal(off_ok,main.new_rule_not_excluded.to_numpy(dtype=bool)),'Off-Day pass flags')
require(np.all(2*O[off_ok]<=A[off_ok]),'Injection density holds for qualifying windows')
levels=[]
for level,(lo,hi) in RANGES.items():
    ok=np.zeros(len(main),dtype=bool)
    for duration in DURATIONS:
        for a in np.unique(A[D==duration]):
            ix=(D==duration)&(A==a)
            if duration==5:
                bit=(1<<EXCEPTION[level]) if a==5 else 0
            elif a>=2:
                t=int(a)-1;low=(t*lo+99)//100;high=(t*hi)//100
                bit=sum(1<<j for j in range(max(1,low),high+1)) if low<=high else 0
            else:bit=0
            ok[ix]=(computed[ix] & np.uint64(bit))!=0
    require(np.array_equal(ok,main[f'exact_{level.lower()}'].to_numpy(dtype=bool)),f'Exact {level} switching')
    levels.append(ok)
any_level=np.logical_or.reduce(levels)
passed=off_ok&any_level
require(np.array_equal(any_level,main.exact_any_level.to_numpy(dtype=bool)),'Any exact switching level')
require(np.array_equal(passed,main.structural_pass.to_numpy(dtype=bool)),'Joint structural pass')
require(np.array_equal(sum(levels),main.compatible_level_count.to_numpy(dtype=int)),'Compatible-level count')
require(np.array_equal(main.optional_2plus_supported.to_numpy(dtype=bool),main.days_2plus.to_numpy()>0),'Optional two-plus flag')
require(np.array_equal(main.optional_3ch_supported.to_numpy(dtype=bool),main.days_3ch.to_numpy()>0),'Optional three-channel flag')
status=np.select([~rotation,rotation & ~off_ok,off_ok & ~any_level],
    ['No natural rotation support','Fails forced-only Off-Day rule','No exact Switching level attainable'],
    default='Structurally compatible')
require(np.array_equal(status,main.integrated_status.to_numpy()),'Integrated exclusion reason per window')
require(len(main.loc[passed])==288486 and main.loc[passed,'user'].nunique()==936,'Expected 288,486 windows / 936 users')
summary=read('integrated_structural_summary.csv')
coverage=read('integrated_switching_level_coverage.csv')
multi=read('integrated_multichannel_options.csv')
five=read('integrated_five_day_exception.csv')
offsummary=read('offday_forced_only_density_comparison.csv')
for duration in DURATIONS:
    g=main.loc[main.window_workdays==duration]
    valid=g.loc[g.structural_pass];sr=summary.loc[summary.Duration==duration]
    require(len(sr)==1,f'Unique integrated summary D={duration}')
    vals={
        'All_windows':len(g),
        'Rotation_windows':int(g.supports_2step.sum()),
        'OffDay_pass_windows':int(g.new_rule_not_excluded.sum()),
        'Exact_compatible_windows':len(valid),
        'OffDay_pass_but_no_level':int((g.new_rule_not_excluded & ~g.exact_any_level).sum()),
        'Compatible_users':valid.user.nunique()
    }
    for field,value in vals.items():
        require(int(sr.iloc[0][field])==value,f'Integrated summary {duration}/{field}')
    expected_pct=round(100*len(valid)/int(g.supports_2step.sum()),2)
    require(abs(float(sr.iloc[0].Exact_compatible_pct_of_rotation)-expected_pct)<0.011,f'Integrated percentage D={duration}')
    for level in RANGES:
        x=coverage.loc[(coverage.Duration==duration)&(coverage.Level==level)]
        v=valid.loc[valid[f'exact_{level.lower()}']]
        require(len(x)==1 and int(x.iloc[0].Compatible_windows)==len(v) and int(x.iloc[0].Compatible_users)==v.user.nunique(),f'Level summary {duration}/{level}')
    m=multi.loc[multi.Duration==duration]
    require(len(m)==1,f'Unique Multichannel summary D={duration}')
    for field,value in {
        'Compatible_windows':len(valid),
        'With_2plus_channel_option':int(valid.optional_2plus_supported.sum()),
        'With_3channel_option':int(valid.optional_3ch_supported.sum()),
        'Users_with_2plus_option':valid.loc[valid.optional_2plus_supported,'user'].nunique(),
        'Users_with_3channel_option':valid.loc[valid.optional_3ch_supported,'user'].nunique()
    }.items():
        require(int(m.iloc[0][field])==value,f'Multichannel summary D={duration}/{field}')
    o=offsummary.loc[offsummary.Duration==duration]
    require(len(o)==1 and int(o.iloc[0].New_windows)==int(g.new_rule_not_excluded.sum())
            and int(o.iloc[0].New_users)==g.loc[g.new_rule_not_excluded,'user'].nunique(),
            'Off-Day summary D='+str(duration))
for level,require_switches in EXCEPTION.items():
    g=main.loc[(main.window_workdays==5)&main.new_rule_not_excluded]
    selected=g.loc[g[f'exact_{level.lower()}']]
    f=five.loc[five.Level==level]
    require(len(f)==1 and int(f.iloc[0].Exception_windows)==len(selected) and int(f.iloc[0].Exception_users)==selected.user.nunique() and int(f.iloc[0].V1_exception_switches)==require_switches,f'Five-day {level} exception summary')

# 8. Report only after all checks
path=REPORT/'final_validation_report.md'
lines=['# Notebook 05 — Source-of-Truth & Final Validation','',f'UTC: {datetime.now(timezone.utc).isoformat()}','','**PASS — all implemented checks completed.**','',f'- Source User-Day rows: {len(raw):,}',f'- Eligible Future User-Day rows: {len(raw_future):,}',f'- Future feature rows: {len(feat):,}',f'- Documented missing feature keys: {len(missing)}',f'- Candidate User-Day / feature days: {len(cand_raw):,}',f'- Candidate users: {len(users)}',f'- Window-level records: {len(main):,}',f'- Compatible windows: {int(passed.sum()):,}',f'- Compatible users: {main.loc[passed,"user"].nunique():,}',f'- Implemented validation assertions: {len(checks)}','','## Documented source discrepancy',str(missing_by_user),'','## Scope caveats','- Source/feature alignment verified; exact switching re-evaluated independently from User-Day.','- User-Day is a processed source, not an independent raw event reconstruction.','- These checks do not verify stealth/payload, final schedules, or model outcomes.','- SHA-256 checksums establish current fingerprints, not proof that historical files were unchanged.','','## Inventory','| File | Rows | Columns | SHA-256 |','|---|---:|---:|---|']
lines += [f'| {m["file"]} | {m["rows"]:,} | {m["columns"]} | `{m["sha256"]}` |' for m in manifest]
lines += ['','## Source SHA-256',f'- User-Day: `{sha(RAW_PATH)}`',f'- Features: `{sha(FEAT_PATH)}`','','## Validation checks'] + [f'- PASS: {check}' for check in checks]
tmp=REPORT/'final_validation_report.md.tmp';tmp.write_text('\n'.join(lines)+'\n',encoding='utf-8');tmp.replace(path)
print('CELL 13 — SOURCE-OF-TRUTH VALIDATION: PASS')
print('User-Day',len(raw),'Eligible Future',len(raw_future),'Features',len(feat))
print('Known missing feature days',len(missing),missing_by_user)
print('Candidate user-days validated',len(cand_raw),'users',len(users))
print('Windows',len(main),'Compatible',int(passed.sum()),'Users',main.loc[passed,'user'].nunique())
print('Checks passed',len(checks),'Files',len(manifest))
print('Report:',path)


SOURCE DISCREPANCY: 275 {'CDE1846': 105, 'MBG3183': 105, 'CMP2946': 65}
CELL 13 — SOURCE-OF-TRUTH VALIDATION: PASS
User-Day 1394010 Eligible Future 387437 Features 387162
Known missing feature days 275 {'CDE1846': 105, 'MBG3183': 105, 'CMP2946': 65}
Candidate user-days validated 103395 users 995
Windows 472667 Compatible 288486 Users 936
Checks passed 211 Files 17
Report: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\05_Simulation_Rules_Feasibility\final_validation_report.md



### 5.8 Data Dictionaries & Final Documentation

#### Objective
Document all 17 validated output datasets from Notebook 05, including their schemas, column meanings, calculations, units, and roles in the simulation feasibility analysis.

#### Tasks
1. Discover the actual columns and data types of every output file.
2. Create an individual Data Dictionary for each dataset.
3. Create one consolidated Master Data Dictionary.
4. Document the approved Duration, Switching, Same-Day Multichannel, and Off-Day rules.
5. Record the final structural feasibility results and validation status.
6. Flag any column whose meaning or calculation cannot be verified.

#### Output Directory
reports/05_Simulation_Rules_Feasibility/data_dictionaries/

#### Expected Outputs
- 17 individual Data Dictionaries
- notebook_05_master_data_dictionary.csv
- data_dictionary_review.csv
- notebook_summary.md (in the main report directory)

#### Quality Rules
Every actual output column must appear exactly once in its dataset dictionary.

Descriptions and calculation formulas must distinguish confirmed definitions from interpretations requiring further review.

The exploratory natural switching screen must not be treated as an approved Primary-Channel Switching rule.

This cell documents existing results only. It does not modify, regenerate, move, or overwrite the validated datasets.


In [12]:
# ============================================================
# CELL 14 — NOTEBOOK 05 VERIFIED DATA DICTIONARIES
# Uses actual schemas + the saved notebook source code.
# Does not change source datasets. Does not move files.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import re
import tempfile
import shutil
import pandas as pd
import pyarrow.parquet as pq

ROOT_514 = Path(
    r'C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized'
    r'\FYP_Data_Exfiltration\New_Attack_Simulation'
)
REPORT_514 = ROOT_514 / 'reports' / '05_Simulation_Rules_Feasibility'
DICT_DIR_514 = REPORT_514 / 'data_dictionaries'
# Keep the corrected notebook filename when saving locally. The original
# name is supported as a fallback if the corrected notebook replaces it.
NB_NAMES_514 = (
    '05_Simulation_Rules_Feasibility_Paths_Fixed.ipynb',
    '05_Simulation_Rules_Feasibility.ipynb',
)

# 17 approved/documented files after excluding the obsolete exploratory screen.
# notebook_cell is the producer's logical cell number, not the JSON index.
FILE_INFO_514 = {
 'natural_window_switching_audit.parquet': ('02_Switching', 6, 472667),
 'natural_window_switching_enriched.parquet': ('02_Switching', 7, 472667),
 'switching_rule_integer_feasibility.parquet': ('02_Switching', 8, 1788),
 'switching_full_density_capacity_screen.parquet': ('02_Switching', 8, 72),
 'switching_range_comparison.csv': ('02_Switching', 8, 4),
 'same_day_multichannel_window_audit.parquet': ('03_Same_Day_Multichannel', 9, 472667),
 'same_day_multichannel_rule_comparison.csv': ('03_Same_Day_Multichannel', 9, 30),
 'offday_window_capacity_audit.parquet': ('04_Off_Days', 10, 472667),
 'offday_v1_15_quota_comparison.csv': ('04_Off_Days', 10, 15),
 'offday_v1_vs_v2_summary.csv': ('04_Off_Days', 10, 6),
 'offday_forced_only_density_window_audit.parquet': ('04_Off_Days', 11, 472667),
 'offday_forced_only_density_comparison.csv': ('04_Off_Days', 11, 6),
 'integrated_structural_window_audit.parquet': ('05_Integrated_Compatibility', 12, 472667),
 'integrated_structural_summary.csv': ('05_Integrated_Compatibility', 12, 6),
 'integrated_switching_level_coverage.csv': ('05_Integrated_Compatibility', 12, 18),
 'integrated_multichannel_options.csv': ('05_Integrated_Compatibility', 12, 6),
 'integrated_five_day_exception.csv': ('05_Integrated_Compatibility', 12, 3),
}

# Every output file remains in the organized dimension folder.
# The primary integrated Parquet stays at the report root.
def output_path_514(filename):
    if filename == 'integrated_structural_window_audit.parquet':
        return REPORT_514 / filename
    return REPORT_514 / FILE_INFO_514[filename][0] / filename

assert REPORT_514.is_dir(), f'Missing report folder: {REPORT_514}'
notebook_paths_514 = [
    base / name
    for name in NB_NAMES_514
    for base in (Path.cwd(), ROOT_514)
    if (base / name).is_file()
]
if not notebook_paths_514:
    for name in NB_NAMES_514:
        notebook_paths_514.extend(ROOT_514.rglob(name))
assert notebook_paths_514, (
    'Cannot locate the current notebook source. Keep the downloaded corrected '
    'notebook filename, or replace the original notebook under its original name.'
)
NOTEBOOK_514 = notebook_paths_514[0]
nb_obj_514 = json.loads(NOTEBOOK_514.read_text(encoding='utf-8'))
source_cells_514 = {
    i: ''.join(cell.get('source', []))
    for i, cell in enumerate(nb_obj_514.get('cells', []))
    if cell.get('cell_type') == 'code'
}

# Match producer code by the exact output save operation, not by guessed cell numbering.
# An output filename merely appearing as a read or validation does not make it the producer.
def producing_cell_514(filename):
    matches = []
    for idx, source in source_cells_514.items():
        if filename not in source:
            continue
        if re.search(r'\.to_(?:parquet|csv)\s*\(', source):
            # Only expected producer blocks have a save and filename reference.
            # Limit the match to known logical producer cells below.
            first = source.splitlines()[1:6]
            headline = ' '.join(first).upper()
            if re.search(r'CELL\s+(6|7|8|9|10|11|12)\b', headline):
                matches.append((idx, source))
    return matches

# Files and producer role as verified by code inspection.
PRODUCER_HEADLINE_514 = {
    6: 'CELL 6 — NATURAL SWITCHING',
    7: 'CELL 7 — NATURAL DOMINANT-CHANNEL SWITCHING',
    8: 'CELL 8 — CORRECTED PRIMARY SWITCHING',
    9: 'CELL 9 — SAME-DAY',
    10: 'CELL 10 — OFF-DAYS',
    11: 'CELL 11 — FORCED-ONLY',
    12: 'CELL 12 — INTEGRATED',
}
producer_code_514 = {}
for number, prefix in PRODUCER_HEADLINE_514.items():
    matches = [(idx, s) for idx, s in source_cells_514.items() if prefix in s.upper()[:300]]
    assert len(matches) == 1, f'Expected one producer code cell for {prefix}; found {len(matches)}'
    producer_code_514[number] = matches[0]

for fname, (_, producer, expected_rows) in FILE_INFO_514.items():
    path = output_path_514(fname)
    assert path.is_file() and path.stat().st_size > 0, f'Missing or empty: {fname}'
    if producer is not None:
        source = producer_code_514[producer][1]
        assert fname in source, f'Notebook producer Cell {producer} does not reference {fname}'
        assert re.search(r'\.to_(?:parquet|csv)\s*\(', source), f'No save code in Cell {producer}'

# All descriptions below are based on the equations / data transformations in this notebook.
# An unverified column receives REVIEW_REQUIRED rather than a guessed meaning.
DOC_514 = {}
def define_514(name, description, calculation='Stored value / identifier', unit='—', null_meaning='Not defined as zero; inspect source and context'):
    DOC_514[name] = (description, calculation, unit, null_meaning)

# Identity and natural window metrics — Cell 6.
define_514('user', 'CERT user identifier', 'Source user identifier; candidate population restricted to 995 users', 'user ID')
define_514('window_workdays', 'Window duration measured in consecutive eligible user-workdays', 'D in {5, 10, 20, 30, 40, 50}; not calendar-day length', 'eligible workdays')
define_514('window_start', 'Date of the first eligible user-workday in the window', 'dates[start_index]', 'date')
define_514('window_end', 'Date of the last eligible user-workday in the window', 'dates[start_index + D - 1]', 'date')
define_514('start_eligible_position', 'Zero-based index of the first eligible user day in its future sequence', 'start_index', 'index')
define_514('end_eligible_position', 'Zero-based index of the last eligible user day in its future sequence', 'start_index + D - 1', 'index')
define_514('natural_active_days', 'Eligible days with at least one active target channel', 'Count of (external_email_count > 0 OR usb_copy_count > 0 OR www_upload_count > 0)', 'days')
define_514('email_active_days', 'Eligible window days with external email activity', 'Count of external_email_count > 0', 'days')
define_514('usb_active_days', 'Eligible window days with USB copy activity', 'Count of usb_copy_count > 0', 'days')
define_514('http_active_days', 'Eligible window days with HTTP upload activity', 'Count of www_upload_count > 0', 'days')
define_514('observed_channels', 'Number of distinct target channels observed at least once inside a window', 'I(email_active_days>0)+I(usb_active_days>0)+I(http_active_days>0)', 'channels')
define_514('natural_multichannel_days', 'Days with activity in at least two target channels', 'Count daily channel masks in {3,5,6,7}; E=1, U=2, H=4', 'days')
define_514('natural_adjacent_direct_switches', 'Natural direct single-channel switches on immediately adjacent eligible days', 'Count adjacent eligible-day pairs with distinct masks in {1,2,4}', 'switches')
define_514('natural_adjacent_set_changes', 'Changes of active channel sets across immediately adjacent eligible days', 'Count adjacent positive-mask pairs whose masks differ', 'changes')
define_514('natural_active_direct_switches', 'Direct changes between single-channel states across successive naturally active days', 'Count successive positive-mask pairs with distinct masks in {1,2,4}, skipping zero-mask days', 'switches')
define_514('natural_active_overlap_changes', 'Natural set changes between active days involving multi-channel overlap', 'natural_active_set_changes − natural_active_direct_switches', 'changes')
define_514('natural_active_set_changes', 'Changes in channel-set masks between successive naturally active days', 'Count differences of successive positive daily channel masks', 'changes')
define_514('natural_active_transition_pairs', 'Possible transition pairs between naturally active days', 'max(natural_active_days − 1, 0)', 'pairs')
define_514('natural_set_change_rate_pct', 'Percent of naturally active transitions with any channel-set change', 'round(100 × natural_active_set_changes / max(natural_active_transition_pairs,1),2)', '%')
define_514('supports_2step', 'Preliminary ability to have at least one primary-channel change in a window', '(natural_active_days >= 2) AND (observed_channels >= 2)', 'boolean')

# Descriptive, normalized natural dominant-channel metrics — Cell 7.
DOM_FORMULA_514 = 'Daily channel dominance = unique positive maximum of count/historical_p95; tie or zero activity gives no valid dominant label'
define_514('dominant_email_days', 'Days uniquely dominated by Email using normalized natural counts', DOM_FORMULA_514 + '; count dominant label=Email', 'days')
define_514('dominant_usb_days', 'Days uniquely dominated by USB using normalized natural counts', DOM_FORMULA_514 + '; count dominant label=USB', 'days')
define_514('dominant_http_days', 'Days uniquely dominated by HTTP upload using normalized natural counts', DOM_FORMULA_514 + '; count dominant label=HTTP', 'days')
define_514('dominant_valid_days', 'Days with a unique positive normalized dominant channel', 'dominant_email_days + dominant_usb_days + dominant_http_days', 'days')
define_514('dominant_tied_days', 'Days where at least two positive normalized channel counts tie for maximum', 'Count daily tied maximum (np.isclose with rtol=1e-9 and atol=1e-12)', 'days')
define_514('dominant_no_activity_days', 'Days without a positive normalized channel activity', 'Count days whose maximum normalized channel activity is not positive', 'days')
define_514('dominant_channels_observed', 'Distinct channels appearing as unique natural daily dominants in the window', 'I(dominant_email_days>0)+I(dominant_usb_days>0)+I(dominant_http_days>0)', 'channels')
define_514('natural_dominant_switches', 'Changes between consecutive valid natural dominant labels, skipping invalid/tied days', 'Number of changes between successive valid daily dominant labels', 'switches')
define_514('dominant_transition_pairs', 'Possible transitions between valid natural dominant days', 'max(dominant_valid_days-1,0)', 'pairs')
define_514('natural_dominant_switch_rate_pct', 'Percent of valid natural dominant-label transitions with a change', 'round(100×natural_dominant_switches/dominant_transition_pairs,2) where pairs>0', '%', 'NaN when dominant_transition_pairs = 0')
define_514('has_dominant_switch', 'Whether the window has one or more natural dominant-channel switches', 'natural_dominant_switches > 0', 'boolean')
define_514('has_tied_day', 'Whether any day has a tied positive dominant label', 'dominant_tied_days > 0', 'boolean')

# Mathematically possible primary channel switch rates — Cell 8.
define_514('range_id', 'Switching-range candidate identifier A, B, C or D', 'Rate-band scenario from RANGES_55', 'category')
define_514('Range', 'Switching-range candidate identifier A, B, C or D', 'Same as range_id, renamed for comparison tables', 'category')
define_514('duration', 'Candidate duration D in eligible workdays', 'One of the six approved window lengths', 'days')
define_514('Duration', 'Candidate duration D in eligible workdays', 'Grouping by window_workdays (or duration) in applicable source cell', 'days')
define_514('attack_active_days', 'Hypothetical count of injection-active days for integer feasibility', 'A in 2..Duration; mathematical screening, not chosen dates', 'days')
define_514('level', 'Switching-intensity level Low, Medium or High', 'One of three nominated levels in the candidate rate range', 'category')
define_514('Level', 'Switching-intensity level Low, Medium or High', 'One of Low / Medium / High', 'category')
define_514('min_rate', 'Lower fractional rate limit for this candidate range and level', 'RANGES_55[range_id][level][0], e.g. 0.10', 'fraction')
define_514('max_rate', 'Upper fractional rate limit for this candidate range and level', 'RANGES_55[range_id][level][1], e.g. 0.25', 'fraction')
define_514('valid_switch_counts', 'Comma-separated exact positive integer switch counts within allowed rate', 'All s in [1,A-1] with min_rate ≤ s/(A-1) ≤ max_rate', 'switch counts as text')
define_514('option_count', 'Number of feasible exact integer switch counts', 'len(valid_switch_counts)', 'integer options')
define_514('integer_feasible', 'Whether at least one exact positive integer switch count is allowed', 'option_count > 0', 'boolean')
define_514('min_switches', 'Smallest mathematically permitted positive switch count', 'min(valid_switch_counts), when nonempty', 'switches', 'NaN if no permitted switch count')
define_514('max_switches', 'Largest mathematically permitted switch count', 'max(valid_switch_counts), when nonempty', 'switches', 'NaN if no permitted switch count')
define_514('Level_cases', 'Number of evaluated (duration, active-days, level) cases for the range', 'GroupBy Range; count level rows', 'cases')
define_514('Feasible_level_cases', 'Cases in which at least one exact integer switch count fits the range', 'GroupBy Range; sum(integer_feasible)', 'cases')
define_514('Infeasible_level_cases', 'Cases with no exact integer switch-count solution', 'Level_cases − Feasible_level_cases', 'cases')
define_514('Total_integer_options', 'Total mathematical switch-count choices accumulated across cases', 'GroupBy Range; sum(option_count)', 'options')
define_514('Active_day_configs_all_3_levels', 'Number of duration/active-day combinations where all three levels have integer solutions', 'GroupBy Range,duration,attack_active_days; all(integer_feasible across levels); sum by Range', 'configs')
define_514('Active_days_tested', 'Active-day count in the full-density reference screen', 'A=Duration only; a diagnostic reference, not mandatory injection density', 'days')
define_514('Integer_feasible', 'Whether at least one exact integer rate solution exists in the full-density reference', 'Same as integer_feasible for A=Duration', 'boolean')
define_514('Min_required_switches', 'Minimum switch count required by the range in full-density reference', 'min_switches for A=Duration if feasible', 'switches', 'Null if integer-infeasible')
define_514('Max_allowed_switches', 'Maximum switch count allowed by the range in full-density reference', 'max_switches for A=Duration if feasible', 'switches', 'Null if integer-infeasible')
define_514('Users_supporting_min_steps', 'Users whose NB02 upper-bound capacity supports at least S+1 alternating steps', 'Count users with max_feasible_rotation_steps >= Min_required_switches+1', 'users')
define_514('Rotation_window_lower_bound', 'Conservative count of eligible windows supporting required rotation-step threshold', 'Sum NB02 windows_supporting_ksteps at next available threshold ≥ required steps; 0 when none', 'windows')
define_514('Rotation_window_upper_bound', 'Upper count bound from nearest lower available step threshold', 'Sum NB02 windows_supporting_ksteps at nearest threshold ≤ required steps; total windows when none', 'windows')

# Optional multichannel activity — Cell 9.
define_514('days_2plus', 'Days with at least two naturally active channels in one eligible day', 'Count daily E/U/H masks in {3,5,6,7}', 'days')
define_514('days_exactly_2ch', 'Days with exactly two naturally active channels', 'days_EU+days_EH+days_UH', 'days')
define_514('days_3ch', 'Days with all three naturally active channels', 'Count daily mask = 7', 'days')
define_514('days_EU', 'Days with Email and USB active, HTTP inactive', 'Count daily mask = 3 (E=1,U=2,H=4)', 'days')
define_514('days_EH', 'Days with Email and HTTP active, USB inactive', 'Count daily mask = 5', 'days')
define_514('days_UH', 'Days with USB and HTTP active, Email inactive', 'Count daily mask = 6', 'days')
define_514('days_EUH', 'Days with all Email, USB and HTTP active', 'Count daily mask = 7; equal to days_3ch', 'days')
define_514('Requirement', 'Optional or mandatory same-day multichannel rule tested', 'One of: optional/zero, ≥1/≥2/≥3 multichannel days, or ≥1 three-channel day', 'category')
define_514('Retained_windows', 'Rotation-capable actual windows satisfying this multichannel requirement', 'Count supports_2step windows meeting Requirement', 'windows')
define_514('Window_coverage_pct', 'Percent of rotation-capable windows retained under the requirement', 'round(100 × Retained_windows / rotation-capable windows for D, 2)', '%')
define_514('Users_with_a_window', 'Distinct users retaining at least one window under the requirement', 'nunique(user) among retained windows', 'users')
define_514('Users_lost_vs_optional', 'Users without a retained window compared with optional rule', 'Number of rotation-capable users − Users_with_a_window', 'users')

# Forced Off-Days and comparison screens — Cell 10.
define_514('available_channel_days', 'Eligible window days with ≥1 naturally active E/U/H channel', 'Count daily target-channel availability; equals proposed_active_days in Cell 11', 'days')
define_514('forced_offdays', 'Eligible days when no E/U/H channel is naturally active and hence no injection is permitted', 'window_workdays − available_channel_days', 'days')
define_514('v2_min_offdays', 'Minimum possible Off-Days in flexible V2 (unavoidable zeros)', 'forced_offdays', 'days')
define_514('v2_max_offdays', 'Maximum Off-Days in flexible V2 allowing ≥2 active days', 'window_workdays − 2', 'days')
define_514('v2_flexible_not_excluded', 'Whether flexible V2 preliminarily allows the window', 'supports_2step AND v2_min_offdays ≤ v2_max_offdays', 'boolean')
define_514('v1_possible_quota_count', 'Number of original V1 Off-Day quotas preliminarily feasible for the window', 'Count V1 quotas O with forced_offdays ≤ O and D−O ≥ 2', 'quotas')
define_514('v1_any_quota_not_excluded', 'Whether any original V1 Off-Day quota is preliminarily feasible', 'v1_possible_quota_count > 0', 'boolean')
define_514('V1_OffDays', 'Fixed Off-Day quota tested in legacy V1 rule', 'D5:[0], D10:[1,2], D20:[2,3,4], D30:[4,5,6], D40:[6,7,8], D50:[8,9,10]', 'days')
define_514('V1_ActiveDays', 'Injection-active days implied by legacy V1 quota', 'Duration − V1_OffDays', 'days')
define_514('Windows_not_excluded', 'Rotation-support windows able to satisfy the specified V1 quota', 'Count windows with forced_offdays ≤ V1_OffDays and V1_ActiveDays ≥ 2', 'windows')
define_514('Window_retention_pct', 'Percent of V2 rotation windows retained for this V1 quota', 'round(100 × Windows_not_excluded / V2_windows,2)', '%')
define_514('Users_with_window', 'Distinct users with ≥1 window satisfying this quota', 'nunique(user) among feasible windows', 'users')
define_514('Users_lost_vs_V2', 'Users lost relative to flexible V2 availability support', 'V2_users − Users_with_window', 'users')
define_514('V2_windows', 'Windows admitted by preliminary flexible V2 rule', 'Count v2_flexible_not_excluded windows for Duration', 'windows')
define_514('V2_users', 'Users having a window admitted by flexible V2', 'nunique(user) for V2_windows', 'users')
define_514('V1_windows', 'Windows passing at least one legacy V1 quota', 'Count v1_any_quota_not_excluded windows', 'windows')
define_514('V1_users', 'Users having ≥1 window passing any legacy V1 quota', 'nunique(user) for V1_windows', 'users')
define_514('Windows_lost_by_V1', 'Windows discarded by legacy V1 compared with V2', 'V2_windows − V1_windows', 'windows')
define_514('Users_lost_by_V1', 'Users discarded by legacy V1 compared with V2', 'V2_users − V1_users', 'users')
define_514('V1_window_retention_pct', 'Percent of V2 windows retained by any legacy V1 quota', 'round(100 × V1_windows / V2_windows,2)', '%')

# Approved forced-only Off-Day density — Cell 11.
define_514('proposed_offdays', 'Forced-only proposed Off-Days in a selected window', 'proposed_offdays = forced_offdays; no optional Off-Days', 'days')
define_514('proposed_active_days', 'All naturally channel-active eligible days assigned to potential injection under forced-only design', 'window_workdays − proposed_offdays = available_channel_days', 'days')
define_514('max_allowed_offdays', 'Maximum forced Off-Days allowed by density rule', '0 for D=5, otherwise floor(D/3) (equivalent to 2O ≤ A)', 'days')
define_514('v1_reference_pass', 'Previous V1 quota preliminary support, for comparison only', 'Inherited v1_any_quota_not_excluded', 'boolean')
define_514('flexible_v2_reference_pass', 'Flexible V2 preliminary support, for comparison only', 'Inherited v2_flexible_not_excluded', 'boolean')
define_514('new_rule_not_excluded', 'Whether proposed forced-only Off-Days rule preliminarily admits this window', 'supports_2step AND proposed_active_days≥2 AND proposed_offdays≤max_allowed_offdays AND D5 exception', 'boolean')
define_514('new_rule_status', 'Primary reason for passing or failing the forced-only Off-Day screening', 'np.select: no rotation / fewer than 2 active days / too many forced Off-Days / not excluded', 'category')
define_514('Max_OffDays', 'Allowed maximum forced Off-Days for each duration', '0 if D=5 else floor(D/3)', 'days')
define_514('Min_ActiveDays', 'Minimum channel-active days implied by approved density rule', 'Duration − Max_OffDays', 'days')
define_514('Flexible_windows', 'Windows admitted by flexible V2', 'Count flexible_v2_reference_pass within each duration', 'windows')
define_514('New_windows', 'Windows admitted by proposed forced-only density rule', 'Count new_rule_not_excluded within each duration', 'windows')
define_514('New_retention_pct', 'Percent of flexible V2 windows retained under new rule', 'round(100 × New_windows / Flexible_windows, 2)', '%')
define_514('Extra_windows_vs_V1', 'Additional windows admitted by new rule versus V1', 'New_windows − V1_windows', 'windows')
define_514('Windows_lost_vs_flexible', 'Windows lost by new rule compared with flexible V2', 'Flexible_windows − New_windows', 'windows')
define_514('Flexible_users', 'Users having ≥1 window admitted by flexible V2', 'Distinct users with flexible_v2_reference_pass', 'users')
define_514('New_users', 'Users having ≥1 window admitted by new rule', 'Distinct users with new_rule_not_excluded', 'users')
define_514('Extra_users_vs_V1', 'Users gained by new rule compared with V1', 'New_users − V1_users', 'users')
define_514('Users_lost_vs_flexible', 'Users lost by new rule compared with flexible V2', 'Flexible_users − New_users', 'users')

# Exact feasibility integration — Cell 12.
define_514('exact_low', 'Exact primary-channel switch count in Range C Low is achievable on all active days', 'DP attainable-bitmask intersects Low allowed switches; D5 uses exactly 1 switch', 'boolean')
define_514('exact_medium', 'Exact primary-channel switch count in Range C Medium is achievable on all active days', 'DP attainable-bitmask intersects Medium allowed switches; D5 uses exactly 2 switches', 'boolean')
define_514('exact_high', 'Exact primary-channel switch count in Range C High is achievable on all active days', 'DP attainable-bitmask intersects High allowed switches; D5 uses exactly 3 switches', 'boolean')
define_514('exact_any_level', 'Whether any of Low, Medium, High has an attainable exact switch count', 'exact_low OR exact_medium OR exact_high', 'boolean')
define_514('structural_pass', 'Final integrated structural feasibility decision', 'new_rule_not_excluded AND exact_any_level; optional multichannel never blocks eligibility', 'boolean')
define_514('optional_2plus_supported', 'Whether window contains an available day with ≥2 natural target channels', 'days_2plus > 0', 'boolean')
define_514('optional_3ch_supported', 'Whether window contains an available day with all three natural target channels', 'days_3ch > 0', 'boolean')
define_514('compatible_level_count', 'Number of switch-intensity levels exactly achievable in the window', 'I(exact_low)+I(exact_medium)+I(exact_high)', 'levels')
define_514('exact_switch_count_bitmask', 'uint64 bitset enumerating all exactly attainable primary-switch counts under natural channel availability', 'Bit s set iff some one-primary-channel-per-active-day sequence has exactly s switches; dynamic programming over E/U/H masks', 'bitset integer')
define_514('attainable_count_options', 'Number of distinct exact switch-count totals encoded in bitmask', 'int(exact_switch_count_bitmask).bit_count()', 'options')
define_514('min_attainable_switches', 'Smallest attainable exact primary switch count; −1 if no path', 'Index of least-significant set bit, or −1 if mask=0', 'switches')
define_514('max_attainable_switches', 'Largest attainable exact primary switch count; −1 if no path', 'bit_length(mask)−1; returns −1 for mask=0', 'switches')
define_514('integrated_status', 'Primary structural exclusion reason or successful structural compatibility', 'No natural rotation / fails forced-only Off-Days / no exact switching level / structurally compatible', 'category')
define_514('All_windows', 'All actual evaluated windows for this duration', 'Count windows grouped by window_workdays', 'windows')
define_514('Rotation_windows', 'Windows with preliminary 2-step natural rotation support', 'Sum(supports_2step) by Duration', 'windows')
define_514('OffDay_pass_windows', 'Windows admitted by forced-only Off-Day density rule', 'Sum(new_rule_not_excluded) by Duration', 'windows')
define_514('Exact_compatible_windows', 'Windows meeting integrated structural pass criteria', 'Sum(structural_pass) by Duration', 'windows')
define_514('OffDay_pass_but_no_level', 'Off-Day-admitted windows without an attainable exact switch-rate level', 'Count(new_rule_not_excluded AND NOT exact_any_level)', 'windows')
define_514('Exact_compatible_pct_of_rotation', 'Percent of rotation-capable windows fully compatible', 'round(100×Exact_compatible_windows/Rotation_windows,2)', '%')
define_514('Compatible_users', 'Distinct users with ≥1 structurally compatible window (or within level where grouped)', 'nunique(user) for structural_pass; additionally exact_level when Level supplied', 'users')
define_514('Compatible_windows', 'Structurally compatible windows for a given duration and optional switching level', 'Count(structural_pass); if Level given, also exact_<level>', 'windows')
define_514('With_2plus_channel_option', 'Integrated-compatible windows allowing optional same-day use of ≥2 target channels', 'Count(structural_pass AND optional_2plus_supported)', 'windows')
define_514('With_3channel_option', 'Integrated-compatible windows allowing optional same-day use of all three channels', 'Count(structural_pass AND optional_3ch_supported)', 'windows')
define_514('Users_with_2plus_option', 'Users with compatible windows allowing a same-day 2+-channel option', 'nunique(user) where structural_pass AND optional_2plus_supported', 'users')
define_514('Users_with_3channel_option', 'Users with compatible windows allowing a same-day 3-channel option', 'nunique(user) where structural_pass AND optional_3ch_supported', 'users')
define_514('V1_exception_switches', 'Required exact primary switches for D=5 exception', 'Low 1, Medium 2, High 3; four active transition pairs', 'switches')
define_514('General_C_windows', 'D5 Off-Day-valid windows that can meet the unmodified Range C integer rule', 'Count DP masks intersecting general Range C allowed switch-mask (without V1 exception)', 'windows')
define_514('Exception_windows', 'D5 Off-Day-valid windows able to meet the V1 five-day exception level', 'Count exact_<level> for D=5 after Off-Day rule', 'windows')
define_514('Exception_users', 'Distinct users with D5 windows meeting this V1 exception level', 'nunique(user) for passing exception windows', 'users')

# Difference between unmodified Range C rates and D5 exception is intentional.
# Source-proven: 10–25% Low, 30–45% Medium, 50–65% High; D5 exact 1/2/3.

# Trace upstream lineage for inherited columns. This is a documentation chain,
# not a dependency on those files being moved during Cell 15.
PARENTS_514 = {
 'natural_window_switching_enriched.parquet': ['natural_window_switching_audit.parquet'],
 'offday_window_capacity_audit.parquet': ['same_day_multichannel_window_audit.parquet'],
 'offday_forced_only_density_window_audit.parquet': ['offday_window_capacity_audit.parquet'],
 'integrated_structural_window_audit.parquet': [
     'offday_forced_only_density_window_audit.parquet',
     'same_day_multichannel_window_audit.parquet'
 ],
}

# Read a real physical schema (no inferred list of columns).
def schema_514(path):
    if path.suffix == '.parquet':
        pf = pq.ParquetFile(path)
        schema = pf.schema_arrow
        return pf.metadata.num_rows, [
            (field.name, str(field.type), str(field.nullable))
            for field in schema
        ]
    data = pd.read_csv(path)
    return len(data), [
        (column, str(data[column].dtype) + ' (CSV inference)', 'Not declared in CSV')
        for column in data.columns
    ]

# File-specific descriptions where identical column names have special meanings.
FILE_DOC_514 = {
 ('integrated_switching_level_coverage.csv', 'Compatible_windows'):
    ('Number of windows satisfying structural_pass and the specified exact switching Level',
     'Count(structural_pass AND exact_<level>)', 'windows'),
 ('integrated_structural_summary.csv', 'Compatible_users'):
    ('Number of distinct users with ≥1 structurally compatible window within Duration',
     'nunique(user where structural_pass)', 'users'),
 ('integrated_switching_level_coverage.csv', 'Compatible_users'):
    ('Distinct users with ≥1 structurally compatible window for this Duration and Level',
     'nunique(user where structural_pass AND exact_<level>)', 'users'),
}

# Source-level AST traces. An expression is included only when a producing cell
# actually names that column in a dataframe assignment or literal dictionary.
def source_trace_514(code, column):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return ('', '')
    matches = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if (isinstance(target, ast.Subscript)
                    and isinstance(target.slice, ast.Constant)
                    and target.slice.value == column):
                    matches.append((node.lineno, ast.unparse(node.value)))
        if isinstance(node, ast.Dict):
            for key, value in zip(node.keys, node.values):
                if isinstance(key, ast.Constant) and key.value == column:
                    matches.append((getattr(key, 'lineno', node.lineno), ast.unparse(value)))
    if not matches:
        return ('', '')
    line, expr = matches[0]
    # Preserve full expression as source expression; show column-specific line.
    return (str(line), expr)

# Check the populated source line when a known column is inherited.
def origin_514(filename, column):
    producer = FILE_INFO_514[filename][1]
    if producer is None:
        return ('', '', 'NOT_IN_UPLOADED_NOTEBOOK')
    _, code = producer_code_514[producer]
    line, expr = source_trace_514(code, column)
    if line:
        return (f'Cell {producer}, line {line}', expr, 'PRODUCER')
    for parent in PARENTS_514.get(filename, []):
        p_source, p_expr, p_kind = origin_514(parent, column)
        if p_source:
            return (p_source + f' (inherited in Cell {producer})', p_expr, 'INHERITED')
    # Summary .agg and .groupby outputs can be described by their verified named
    # aggregation expressions in DOC_514 even without a literal dictionary key.
    return (f'Cell {producer} (groupby/derived or upstream)', '', 'SOURCE_REVIEWED')

rows_all_514 = []
files_info_514 = []
review_rows_514 = []

for fname, (folder, producer, expected_rows) in FILE_INFO_514.items():
    path = output_path_514(fname)
    nrows, schema_fields = schema_514(path)
    assert nrows == expected_rows, f'{fname}: expected {expected_rows} rows, got {nrows}'
    names = [x[0] for x in schema_fields]
    assert names and len(set(names)) == len(names), f'Invalid schema: {fname}'
    files_info_514.append((fname, folder, nrows, len(names)))

    for ordinal, (col, dtype, nullable) in enumerate(schema_fields, 1):
        verified_code = producer is not None
        override = FILE_DOC_514.get((fname, col))
        doc = DOC_514.get(col)
        if override:
            desc, calc, unit = override
            null_text = 'Not defined as zero; inspect source and context'
        elif doc:
            desc, calc, unit, null_text = doc
        else:
            desc, calc, unit = '', '', ''
            null_text = 'Unknown — inspect actual producing expression'

        origin, source_expr, source_kind = origin_514(fname, col)
        if not verified_code:
            # No original generation code for this legacy screen in uploaded NB.
            status = 'REVIEW_REQUIRED_LEGACY_PRODUCER_MISSING'
            desc, calc = '', ''
            source_expr, origin = '', 'Legacy screen: generator absent from uploaded notebook'
        elif desc:
            status = 'VERIFIED_FROM_NOTEBOOK'
        elif source_expr:
            status = 'SOURCE_EXPRESSION_ONLY_REVIEW_DESCRIPTION'
            calc = f'See exact notebook source expression (Cell {producer})'
        else:
            status = 'REVIEW_REQUIRED_NO_SOURCE_DEFINITION'

        row = {
            'file_name': fname,
            'file_group': folder,
            'column_order': ordinal,
            'column_name': col,
            'data_type': dtype,
            'parquet_nullable_or_csv_inference': nullable,
            'description': desc,
            'calculation': calc,
            'unit': unit,
            'null_meaning': null_text,
            'source_location': origin,
            'source_expression': source_expr,
            'documentation_status': status,
            'output_role': ('EXPLORATORY_ONLY_NOT_APPROVED' if producer is None
                            else 'APPROVED_OR_COMPARATIVE_ANALYSIS'),
        }
        rows_all_514.append(row)
        if status != 'VERIFIED_FROM_NOTEBOOK':
            review_rows_514.append(row)

master_514 = pd.DataFrame(rows_all_514)
assert master_514['file_name'].nunique() == len(FILE_INFO_514)
assert len(master_514) == len(master_514.drop_duplicates(['file_name','column_name']))
assert (master_514.groupby('file_name').size().to_dict()
        == {name: count for name,_,_,count in files_info_514})

# Full report includes only validated existing computed results; no rewrite of data.
summary_csv_514 = pd.read_csv(output_path_514('integrated_structural_summary.csv'))
main_514 = pd.read_parquet(
    REPORT_514 / 'integrated_structural_window_audit.parquet',
    columns=['user','structural_pass']
)
valid_514 = main_514.loc[main_514['structural_pass'].astype(bool)]
assert len(main_514) == 472667
assert len(valid_514) == 288486
assert valid_514['user'].nunique() == 936
assert main_514['user'].nunique() == 995

summary_text_514 = f'''# Notebook 05 — Simulation Rules Feasibility & Selection

## Purpose
Evaluate structural feasibility of prospective low-and-slow, multi-channel exfiltration attacks on naturally eligible user-workdays. No injection, attack-user assignment, or stealth estimation is performed in this notebook.

## Approved structural rules (verified from this notebook)

- **Duration:** D ∈ {{5, 10, 20, 30, 40, 50}} eligible workdays; windows follow each user's eligible-day order, not calendar-day intervals.
- **Target channels:** Email = external_email_count > 0; USB = usb_copy_count > 0; HTTP = www_upload_count > 0. Natural channel activity is not replaced or deleted.
- **Primary-Channel Switching:** Range C selected. Low 10–25%, Medium 30–45%, High 50–65%. For A proposed active days, T=A−1 transitions; valid integer switches S satisfy ceil(T×min_rate) ≤ S ≤ floor(T×max_rate). Cell 12 uses bitset dynamic programming to test exact S under available daily channels.
- **Five-day exception:** D=5 requires all 5 eligible days active and allows exact switching counts Low=1, Medium=2, High=3, overriding general Range C integer feasibility.
- **Off-Days:** Forced-only: O = D − available_channel_days; no optional Off-Days. D=5 requires O=0; otherwise O≤floor(D/3), equivalently 2O≤A. Candidate windows also require supports_2step and at least two active days.
- **Same-day multichannel:** Optional, including zero days. Natural exact-two masks {{EU,EH,UH}} and all-three mask {{EUH}} are audited; multichannel opportunity does not block otherwise viable windows.
- **Structural pass:** new_rule_not_excluded AND (exact_low OR exact_medium OR exact_high). This is feasibility only, not an injected schedule.

## Final validated results

- Candidate users: **995**
- Actual windows checked: **{len(main_514):,}**
- Structurally feasible windows: **{len(valid_514):,}**
- Users with ≥1 structurally feasible window: **{valid_514['user'].nunique():,}**
- Source-of-truth Cell 13: **PASS** after revalidation of retained outputs (see `final_validation_report.md` for current check and file counts).
- Reconciled source discrepancy: 275 eligible Future user-days in processed User-Day but absent from Features, distributed across 3 user IDs (`CDE1846`:105, `MBG3183`:105, `CMP2946`:65). None of the 995 simulation candidates are affected. The discrepancy and candidate impact were tested; causal explanation and original CERT label confirmation belong to upstream Ground Truth/Data Engineering sources.

## Handoff

Primary output: `integrated_structural_window_audit.parquet`; key: `user + window_workdays + window_start`. **Keep this file in the report root** after Cell 15.

All other dimension outputs are stored in numbered dimension folders. Individual dictionaries follow the same folder structure under `data_dictionaries/`.

### Column documentation coverage

- Output datasets documented: **{len(files_info_514)}**
- Actual output columns documented: **{len(master_514)}**
- Columns with source-backed prose definitions: **{int((master_514['documentation_status']=='VERIFIED_FROM_NOTEBOOK').sum())}**
- Columns flagged for targeted review: **{len(review_rows_514)}**
- The deprecated exploratory `natural_switching_level_screen.parquet` was intentionally excluded from final deliverables and dictionaries; it is not a dependency of structural feasibility.

## Scope limits

Results verify structural eligibility against processed User-Day and derived Features only. They do not establish payload stealth, exact injection volumes, complete channel scheduling, or classification metrics.

Generated UTC: {datetime.now(timezone.utc).isoformat()}
'''

# Write to temporary staging and promote only after all dictionaries validate.
with tempfile.TemporaryDirectory(prefix='_tmp_nb05_dict_', dir=REPORT_514) as tmp_dir_str:
    tmp_dir_514 = Path(tmp_dir_str)
    staged_dict_514 = tmp_dir_514 / 'data_dictionaries'
    staged_dict_514.mkdir()
    for filename, (folder, _, _) in FILE_INFO_514.items():
        subset = master_514.loc[master_514['file_name'].eq(filename)]
        assert len(subset) > 0
        target = staged_dict_514 / folder / f'{Path(filename).stem}_dictionary.csv'
        target.parent.mkdir(parents=True, exist_ok=True)
        subset.to_csv(target, index=False, encoding='utf-8-sig')
        assert pd.read_csv(target).shape[0] == len(subset)
    master_path_514 = staged_dict_514 / 'notebook_05_master_data_dictionary.csv'
    master_514.to_csv(master_path_514, index=False, encoding='utf-8-sig')
    pd.DataFrame(review_rows_514, columns=master_514.columns).to_csv(
        staged_dict_514 / 'data_dictionary_review.csv', index=False, encoding='utf-8-sig'
    )
    (tmp_dir_514 / 'notebook_summary.md').write_text(summary_text_514, encoding='utf-8')
    assert len(list(staged_dict_514.rglob('*_dictionary.csv'))) == 18  # 17 individual + 1 master
    assert pd.read_csv(master_path_514).shape[0] == len(master_514)

    DICT_DIR_514.mkdir(parents=True, exist_ok=True)
    for p in staged_dict_514.rglob('*.csv'):
        destination = DICT_DIR_514 / p.relative_to(staged_dict_514)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, destination)
    # Never duplicate individual dictionaries in the dictionary root.
    expected_root_docs_514 = {'notebook_05_master_data_dictionary.csv', 'data_dictionary_review.csv'}
    unexpected_root_docs_514 = {p.name for p in DICT_DIR_514.glob('*.csv')} - expected_root_docs_514
    assert not unexpected_root_docs_514, (
        f'Unexpected dictionaries in root: {sorted(unexpected_root_docs_514)}'
    )
    assert len(list(DICT_DIR_514.rglob('*.csv'))) == 19  # 17 individual + master + review
    shutil.copy2(tmp_dir_514 / 'notebook_summary.md', REPORT_514 / 'notebook_summary.md')

print('=' * 75)
print('CELL 14 — SOURCE-GROUNDED DATA DICTIONARIES')
print('=' * 75)
print('Notebook source       :', NOTEBOOK_514)
print('Output files reviewed :', len(files_info_514))
print('Columns documented    :', len(master_514))
print('Fully described       :', int((master_514['documentation_status']=='VERIFIED_FROM_NOTEBOOK').sum()))
print('Flagged for review    :', len(review_rows_514))
print('Dictionaries saved to :', DICT_DIR_514)
print('Summary saved to      :', REPORT_514 / 'notebook_summary.md')
print('\nFILE COVERAGE')
display(pd.DataFrame(files_info_514, columns=['file','folder','rows','columns']))
if review_rows_514:
    print('\nCOLUMNS FLAGGED FOR REVIEW (showing first 30):')
    display(pd.DataFrame(review_rows_514)[['file_name','column_name','documentation_status']].head(30))
print('\nCELL 14: DOCUMENTATION GENERATED — inspect review flags before Cell 15.')


CELL 14 — SOURCE-GROUNDED DATA DICTIONARIES
Notebook source       : c:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\notebooks\05_Simulation_Rules_Feasibility_Paths_Fixed.ipynb
Output files reviewed : 17
Columns documented    : 210
Fully described       : 210
Flagged for review    : 0
Dictionaries saved to : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\05_Simulation_Rules_Feasibility\data_dictionaries
Summary saved to      : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\05_Simulation_Rules_Feasibility\notebook_summary.md

FILE COVERAGE


,file,folder,rows,columns
0,natural_window_switching_audit.parquet,02_Switching,472667,20
1,natural_window_switching_enriched.parquet,02_Switching,472667,30
2,switching_rule_integer_feasibility.parquet,02_Switching,1788,11
3,switching_full_density_capacity_screen.parquet,02_Switching,72,10
4,switching_range_comparison.csv,02_Switching,4,6
5,same_day_multichannel_window_audit.parquet,03_Same_Day_Multichannel,472667,12
6,same_day_multichannel_rule_comparison.csv,03_Same_Day_Multichannel,30,6
7,offday_window_capacity_audit.parquet,04_Off_Days,472667,19
8,offday_v1_15_quota_comparison.csv,04_Off_Days,15,7
9,offday_v1_vs_v2_summary.csv,04_Off_Days,6,8



CELL 14: DOCUMENTATION GENERATED — inspect review flags before Cell 15.



### 5.9 Final Report Organization

#### Objective
Organize all validated Notebook 05 outputs into dimension-specific folders while keeping the final integrated dataset easily accessible.

#### Organization Rules
- Move Switching outputs into 02_Switching.
- Move Same-Day Multichannel outputs into 03_Same_Day_Multichannel.
- Move Off-Day outputs into 04_Off_Days.
- Move integrated summaries into 05_Integrated_Compatibility.
- Keep integrated_structural_window_audit.parquet in the report root.
- Keep all Data Dictionaries together in data_dictionaries.
- Keep the final validation report and notebook summary in the root.

#### Safety Checks
- Verify all 17 datasets exist before moving.
- Reject missing files and destination conflicts.
- Compare SHA-256 checksums before and after moving.
- Generate outputs_manifest.csv with final file paths.
- Do not change dataset contents or calculations.

#### Expected Result
A clean, validated, dimension-based Notebook 05 report directory.


In [13]:

# ============================================================
# CELL 15 — FINAL REPORT ORGANIZATION
# ============================================================

from pathlib import Path
import hashlib
import csv

ROOT_515 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

REPORT_515 = (
    ROOT_515 / "reports" / "05_Simulation_Rules_Feasibility"
)

GROUPS_515 = {
    "02_Switching": [
        "natural_window_switching_audit.parquet",
        "natural_window_switching_enriched.parquet",
        "switching_rule_integer_feasibility.parquet",
        "switching_full_density_capacity_screen.parquet",
        "switching_range_comparison.csv",
    ],
    "03_Same_Day_Multichannel": [
        "same_day_multichannel_window_audit.parquet",
        "same_day_multichannel_rule_comparison.csv",
    ],
    "04_Off_Days": [
        "offday_window_capacity_audit.parquet",
        "offday_v1_15_quota_comparison.csv",
        "offday_v1_vs_v2_summary.csv",
        "offday_forced_only_density_window_audit.parquet",
        "offday_forced_only_density_comparison.csv",
    ],
    "05_Integrated_Compatibility": [
        "integrated_structural_summary.csv",
        "integrated_switching_level_coverage.csv",
        "integrated_multichannel_options.csv",
        "integrated_five_day_exception.csv",
    ],
}

FINAL_FILE_515 = "integrated_structural_window_audit.parquet"
DICT_DIR_515 = REPORT_515 / "data_dictionaries"

def sha256_515(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. PREFLIGHT — NO FILES MOVED YET
# ------------------------------------------------------------

assert REPORT_515.is_dir(), "Report directory missing"

assert (REPORT_515 / FINAL_FILE_515).is_file(), (
    "Main integrated file missing"
)

assert (REPORT_515 / "final_validation_report.md").is_file(), (
    "Cell 13 report missing"
)

assert (REPORT_515 / "notebook_summary.md").is_file(), (
    "Cell 14 summary missing"
)

assert (
    DICT_DIR_515 / "notebook_05_master_data_dictionary.csv"
).is_file(), "Master dictionary missing"

individual_dicts_515 = [
    p for p in DICT_DIR_515.rglob("*.csv")
    if p.name not in {
        "notebook_05_master_data_dictionary.csv",
        "data_dictionary_review.csv"
    }
]

assert len(individual_dicts_515) == 17, (
    f"Expected 17 individual dictionaries, "
    f"found {len(individual_dicts_515)}"
)

entries_515 = [
    (folder, name)
    for folder, names in GROUPS_515.items()
    for name in names
]

assert len(entries_515) == 16
assert len({name for _, name in entries_515}) == 16

moves_515 = []
records_515 = []

for folder, name in entries_515:
    src = REPORT_515 / name
    dst = REPORT_515 / folder / name

    if src.is_file() and dst.exists():
        raise RuntimeError(
            f"Source and destination both exist: {name}"
        )

    if src.is_file():
        current = src
        moves_515.append((src, dst))
    elif dst.is_file():
        current = dst
    else:
        raise FileNotFoundError(
            f"Missing output: {name}"
        )

    assert current.stat().st_size > 0, name

    records_515.append({
        "dimension": folder,
        "file": name,
        "destination": dst,
        "sha256_before": sha256_515(current),
    })

# ------------------------------------------------------------
# 2. CREATE FOLDERS
# ------------------------------------------------------------

for folder in GROUPS_515:
    (REPORT_515 / folder).mkdir(
        parents=True, exist_ok=True
    )

duration_dir_515 = REPORT_515 / "01_Duration"
duration_dir_515.mkdir(parents=True, exist_ok=True)

# No standalone Duration output was saved in Cell 3.
duration_readme_515 = duration_dir_515 / "README.md"

if not duration_readme_515.exists():
    duration_readme_515.write_text(
        "# Duration\n\n"
        "Approved durations: 5, 10, 20, 30, 40, 50 "
        "eligible workdays.\n\n"
        "No standalone Duration output was saved in "
        "Notebook 05. Duration-level results are available "
        "in integrated_structural_summary.csv and the "
        "final integrated window audit.\n",
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 3. MOVE FILES
# ------------------------------------------------------------

for src, dst in moves_515:
    src.replace(dst)

# ------------------------------------------------------------
# 4. VERIFY CONTENT INTEGRITY
# ------------------------------------------------------------

for item in records_515:
    dst = item["destination"]

    assert dst.is_file(), f"Missing after move: {dst}"

    assert sha256_515(dst) == item["sha256_before"], (
        f"Checksum mismatch: {dst.name}"
    )

    assert not (REPORT_515 / item["file"]).exists(), (
        f"Old root copy still exists: {item['file']}"
    )

# ------------------------------------------------------------
# 5. GENERATE FINAL OUTPUT MANIFEST
# ------------------------------------------------------------

manifest_515 = REPORT_515 / "outputs_manifest.csv"
tmp_manifest_515 = REPORT_515 / "outputs_manifest.csv.tmp"

with tmp_manifest_515.open(
    "w", newline="", encoding="utf-8-sig"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "dimension",
        "file_name",
        "relative_path",
        "size_bytes",
        "sha256"
    ])

    for item in records_515:
        path = item["destination"]

        writer.writerow([
            item["dimension"],
            item["file"],
            path.relative_to(REPORT_515).as_posix(),
            path.stat().st_size,
            sha256_515(path)
        ])

    final_path = REPORT_515 / FINAL_FILE_515

    writer.writerow([
        "Final_Integrated_Output",
        FINAL_FILE_515,
        FINAL_FILE_515,
        final_path.stat().st_size,
        sha256_515(final_path)
    ])

tmp_manifest_515.replace(manifest_515)

# ------------------------------------------------------------
# 6. FINAL VERIFICATION
# ------------------------------------------------------------

with manifest_515.open(
    encoding="utf-8-sig", newline=""
) as f:
    manifest_rows_515 = list(csv.DictReader(f))

assert len(manifest_rows_515) == 17

for row in manifest_rows_515:
    path = REPORT_515 / row["relative_path"]

    assert path.is_file()
    assert sha256_515(path) == row["sha256"]

assert len(list(DICT_DIR_515.rglob("*.csv"))) == 19

# ------------------------------------------------------------
# 7. DISPLAY FINAL STRUCTURE
# ------------------------------------------------------------

print("=" * 75)
print("CELL 15 — FINAL REPORT ORGANIZATION")
print("=" * 75)

print("\nOrganization: PASS")
print("Files moved:", len(moves_515))
print("Total output datasets:", len(manifest_rows_515))
print("Individual dictionaries:", len(individual_dicts_515))
print("SHA-256 integrity: PASS")

print("\nFINAL REPORT STRUCTURE")
print("-" * 75)

for folder in [
    "01_Duration",
    "02_Switching",
    "03_Same_Day_Multichannel",
    "04_Off_Days",
    "05_Integrated_Compatibility",
    "data_dictionaries"
]:
    path = REPORT_515 / folder
    print(f"{folder}/ — {len(list(path.iterdir()))} items")

print("\nROOT FILES")
for name in [
    FINAL_FILE_515,
    "final_validation_report.md",
    "notebook_summary.md",
    "outputs_manifest.csv"
]:
    print("  ", name)

print("\nALL CONTENT CHECKSUMS VERIFIED")
print("NOTEBOOK 05 REPORT ORGANIZATION COMPLETE")


CELL 15 — FINAL REPORT ORGANIZATION

Organization: PASS
Files moved: 0
Total output datasets: 17
Individual dictionaries: 17
SHA-256 integrity: PASS

FINAL REPORT STRUCTURE
---------------------------------------------------------------------------
01_Duration/ — 1 items
02_Switching/ — 5 items
03_Same_Day_Multichannel/ — 2 items
04_Off_Days/ — 5 items
05_Integrated_Compatibility/ — 4 items
data_dictionaries/ — 6 items

ROOT FILES
   integrated_structural_window_audit.parquet
   final_validation_report.md
   notebook_summary.md
   outputs_manifest.csv

ALL CONTENT CHECKSUMS VERIFIED
NOTEBOOK 05 REPORT ORGANIZATION COMPLETE


In [14]:

# ============================================================
# CELL 15.1 — ORGANIZE DATA DICTIONARIES
# ============================================================

from pathlib import Path
import hashlib
import shutil

ROOT_5151 = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

DICT_DIR_5151 = (
    ROOT_5151
    / "reports"
    / "05_Simulation_Rules_Feasibility"
    / "data_dictionaries"
)

GROUPS_5151 = {
    "02_Switching": [
        "natural_window_switching_audit_dictionary.csv",
        "natural_window_switching_enriched_dictionary.csv",
        "switching_rule_integer_feasibility_dictionary.csv",
        "switching_full_density_capacity_screen_dictionary.csv",
        "switching_range_comparison_dictionary.csv",
    ],
    "03_Same_Day_Multichannel": [
        "same_day_multichannel_window_audit_dictionary.csv",
        "same_day_multichannel_rule_comparison_dictionary.csv",
    ],
    "04_Off_Days": [
        "offday_window_capacity_audit_dictionary.csv",
        "offday_v1_15_quota_comparison_dictionary.csv",
        "offday_v1_vs_v2_summary_dictionary.csv",
        "offday_forced_only_density_window_audit_dictionary.csv",
        "offday_forced_only_density_comparison_dictionary.csv",
    ],
    "05_Integrated_Compatibility": [
        "integrated_structural_window_audit_dictionary.csv",
        "integrated_structural_summary_dictionary.csv",
        "integrated_switching_level_coverage_dictionary.csv",
        "integrated_multichannel_options_dictionary.csv",
        "integrated_five_day_exception_dictionary.csv",
    ],
}

ROOT_DOCS_5151 = [
    "notebook_05_master_data_dictionary.csv",
    "data_dictionary_review.csv",
]

def sha256_5151(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# 1. Preflight: ensure no missing files or conflicts.
assert DICT_DIR_5151.is_dir()

for name in ROOT_DOCS_5151:
    assert (DICT_DIR_5151 / name).is_file(), name

entries_5151 = [
    (folder, filename)
    for folder, filenames in GROUPS_5151.items()
    for filename in filenames
]

assert len(entries_5151) == 17
assert len({name for _, name in entries_5151}) == 17

moves_5151 = []
checksums_5151 = {}

for folder, filename in entries_5151:
    src = DICT_DIR_5151 / filename
    dst = DICT_DIR_5151 / folder / filename

    if src.exists() and dst.exists():
        raise RuntimeError(
            f"Duplicate source and destination: {filename}"
        )

    if src.is_file():
        current = src
        moves_5151.append((src, dst))
    elif dst.is_file():
        current = dst
    else:
        raise FileNotFoundError(filename)

    assert current.stat().st_size > 0, filename
    checksums_5151[filename] = sha256_5151(current)

# 2. Create folders and move dictionaries.
for folder in GROUPS_5151:
    (DICT_DIR_5151 / folder).mkdir(
        parents=True, exist_ok=True
    )

for src, dst in moves_5151:
    shutil.move(str(src), str(dst))

# 3. Verify every dictionary after moving.
for folder, filename in entries_5151:
    dst = DICT_DIR_5151 / folder / filename

    assert dst.is_file(), filename
    assert sha256_5151(dst) == checksums_5151[filename]
    assert not (DICT_DIR_5151 / filename).exists()

# 4. Verify that only two CSVs remain in root.
remaining_5151 = sorted(
    p.name for p in DICT_DIR_5151.glob("*.csv")
)

assert remaining_5151 == sorted(ROOT_DOCS_5151), (
    f"Unexpected root CSV files: {remaining_5151}"
)

# 5. Final output.
print("=" * 65)
print("CELL 15.1 — DATA DICTIONARY ORGANIZATION")
print("=" * 65)

print("Status: PASS")
print("Individual dictionaries verified: 17")
print("Moved this run:", len(moves_5151))
print("SHA-256 integrity: PASS")

print("\nFINAL STRUCTURE")
for folder, filenames in GROUPS_5151.items():
    print(f"{folder}/ — {len(filenames)} dictionaries")

print("\nRoot files:")
for name in ROOT_DOCS_5151:
    print(" ", name)

print("\nORGANIZATION COMPLETE")


CELL 15.1 — DATA DICTIONARY ORGANIZATION
Status: PASS
Individual dictionaries verified: 17
Moved this run: 0
SHA-256 integrity: PASS

FINAL STRUCTURE
02_Switching/ — 5 dictionaries
03_Same_Day_Multichannel/ — 2 dictionaries
04_Off_Days/ — 5 dictionaries
05_Integrated_Compatibility/ — 5 dictionaries

Root files:
  notebook_05_master_data_dictionary.csv
  data_dictionary_review.csv

ORGANIZATION COMPLETE
